In [ ]:
!pip install -q torch torchaudio librosa soundfile pesq pystoi pandas tabulate
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])").html

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 12.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.9 MB/s eta 0:00:00


In [ ]:
# -*- coding: utf-8 -*-


import os, sys, math, time, copy, json, argparse, warnings, logging
from typing import Dict, List, Optional, Tuple, Any
from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import scipy.signal as signal
import scipy.io.wavfile as wav_io
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Normal

import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as TAF

try:
    import librosa
    import librosa.display
    LIBROSA_AVAILABLE = True
except ImportError:
    LIBROSA_AVAILABLE = False
    print("[WARN] librosa not available, using torchaudio fallback")

try:
    import soundfile as sf
    SF_AVAILABLE = True
except ImportError:
    SF_AVAILABLE = False

try:
    from pesq import pesq
    PESQ_AVAILABLE = True
except ImportError:
    PESQ_AVAILABLE = False
    print("[WARN] pesq not available, PESQ metric will be skipped")

try:
    from pystoi import stoi
    STOI_AVAILABLE = True
except ImportError:
    STOI_AVAILABLE = False
    print("[WARN] pystoi not available, STOI metric will be skipped")

try:
    import torch_geometric.nn as gnn_layers
    from torch_geometric.data import Data as GraphData, Batch as GraphBatch
    TORCH_GEOMETRIC_AVAILABLE = True
except ImportError:
    TORCH_GEOMETRIC_AVAILABLE = False
    print("[WARN] torch_geometric not available, using custom GNN implementation")

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

try:
    from tabulate import tabulate
    TABULATE_AVAILABLE = True
except ImportError:
    TABULATE_AVAILABLE = False

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)


# =============================================================================
# SECTION 1: CONFIGURATION
# =============================================================================

@dataclass
class AudioConfig:
    sample_rate: int = 22050
    hop_length: int = 256
    win_length: int = 1024
    n_fft: int = 1024
    n_mels: int = 80
    fmin: float = 80.0
    fmax: float = 7600.0
    min_level_db: float = -100.0
    ref_level_db: float = 20.0
    preemphasis: float = 0.97
    max_wav_value: float = 32768.0
    trim_silence: bool = True
    trim_top_db: float = 40.0

@dataclass
class ModelConfig:
    latent_dim: int = 128
    vae_hidden_dim: int = 256
    vae_num_layers: int = 4
    gnn_hidden_dim: int = 256
    gnn_num_layers: int = 4
    gnn_num_heads: int = 4
    fastspeech_hidden: int = 256
    fastspeech_num_heads: int = 2
    fastspeech_num_layers: int = 4
    fastspeech_kernel_size: int = 9
    fastspeech_filter_size: int = 1024
    hifigan_upsample_rates: List[int] = field(default_factory=lambda: [8, 8, 2, 2])
    hifigan_upsample_kernel_sizes: List[int] = field(default_factory=lambda: [16, 16, 4, 4])
    hifigan_upsample_initial_channel: int = 512
    hifigan_resblock_kernel_sizes: List[int] = field(default_factory=lambda: [3, 7, 11])
    hifigan_resblock_dilation_sizes: List[List[int]] = field(default_factory=lambda: [[1, 3, 5], [1, 3, 5], [1, 3, 5]])
    dropout: float = 0.1
    num_emotions: int = 7
    pitch_bins: int = 256
    energy_bins: int = 256

@dataclass
class TrainConfig:
    batch_size: int = 8
    learning_rate: float = 1e-4
    num_epochs: int = 100
    warmup_steps: int = 4000
    beta_kl: float = 0.0001
    beta_kl_max: float = 0.1
    beta_kl_anneal_steps: int = 20000
    grad_clip: float = 1.0
    checkpoint_dir: str = "checkpoints"
    log_interval: int = 50
    save_interval: int = 10
    lambda_pitch: float = 1.0
    lambda_energy: float = 1.0
    lambda_duration: float = 1.0
    lambda_mel: float = 1.0
    lambda_emotion: float = 0.5

@dataclass
class EmotionProfile:
    name: str
    pitch_shift: float
    speed_factor: float
    energy_scale: float
    pitch_variance: float
    energy_variance: float
    duration_factor: float
    voice_quality: str
    description: str
    intensity_range: Tuple[float, float]


EMOTION_PROFILES: Dict[str, EmotionProfile] = {
    "neutral": EmotionProfile(
        name="neutral",
        pitch_shift=0.0,
        speed_factor=1.0,
        energy_scale=1.0,
        pitch_variance=0.05,
        energy_variance=0.05,
        duration_factor=1.0,
        voice_quality="modal",
        description="Calm, flat affect, balanced pitch and energy",
        intensity_range=(0.3, 0.7)
    ),
    "happy": EmotionProfile(
        name="happy",
        pitch_shift=2.5,
        speed_factor=1.12,
        energy_scale=1.20,
        pitch_variance=0.18,
        energy_variance=0.15,
        duration_factor=0.92,
        voice_quality="bright",
        description="Higher pitch, faster rate, more energy, wider range",
        intensity_range=(0.5, 1.0)
    ),
    "sad": EmotionProfile(
        name="sad",
        pitch_shift=-2.0,
        speed_factor=0.84,
        energy_scale=0.72,
        pitch_variance=0.06,
        energy_variance=0.05,
        duration_factor=1.22,
        voice_quality="breathy",
        description="Lower pitch, slower rate, reduced energy, narrow range",
        intensity_range=(0.3, 0.8)
    ),
    "angry": EmotionProfile(
        name="angry",
        pitch_shift=1.5,
        speed_factor=1.08,
        energy_scale=1.55,
        pitch_variance=0.22,
        energy_variance=0.28,
        duration_factor=0.88,
        voice_quality="tense",
        description="High energy, tense voice, aggressive pitch jumps",
        intensity_range=(0.6, 1.0)
    ),
    "fearful": EmotionProfile(
        name="fearful",
        pitch_shift=3.5,
        speed_factor=1.18,
        energy_scale=0.85,
        pitch_variance=0.25,
        energy_variance=0.20,
        duration_factor=0.90,
        voice_quality="tremulous",
        description="High, unstable pitch, fast rate, irregular energy",
        intensity_range=(0.5, 1.0)
    ),
    "surprised": EmotionProfile(
        name="surprised",
        pitch_shift=4.0,
        speed_factor=1.05,
        energy_scale=1.30,
        pitch_variance=0.30,
        energy_variance=0.22,
        duration_factor=0.85,
        voice_quality="bright",
        description="Very high pitch onset, wide pitch range, strong onset energy",
        intensity_range=(0.5, 1.0)
    ),
    "disgusted": EmotionProfile(
        name="disgusted",
        pitch_shift=-1.0,
        speed_factor=0.92,
        energy_scale=1.10,
        pitch_variance=0.12,
        energy_variance=0.18,
        duration_factor=1.08,
        voice_quality="creaky",
        description="Low, creaky voice, moderate energy, deliberate pacing",
        intensity_range=(0.3, 0.9)
    ),
}

EMOTION_IDX = {name: i for i, name in enumerate(EMOTION_PROFILES.keys())}
IDX_EMOTION = {i: name for name, i in EMOTION_IDX.items()}


# =============================================================================
# SECTION 2: AUDIO PROCESSING UTILITIES
# =============================================================================

class AudioProcessor:
    """
    Handles all audio I/O, feature extraction, and signal processing.
    Features extracted: Mel spectrogram, F0 (pitch), Energy, Duration (phone-level).
    """

    def __init__(self, config: AudioConfig):
        self.config = config
        self.sr = config.sample_rate
        self.hop = config.hop_length
        self.win = config.win_length
        self.n_fft = config.n_fft
        self.n_mels = config.n_mels

        self.mel_transform = T.MelSpectrogram(
            sample_rate=self.sr,
            n_fft=self.n_fft,
            win_length=self.win,
            hop_length=self.hop,
            f_min=config.fmin,
            f_max=config.fmax,
            n_mels=self.n_mels,
            power=1.0,
            normalized=False,
            center=True,
        )
        self.amplitude_to_db = T.AmplitudeToDB(stype='amplitude', top_db=80.0)

    def load_audio(self, path: str) -> Tuple[torch.Tensor, int]:
        """Load audio file and resample if needed."""
        try:
            sample_rate_file, data = wav_io.read(path)
            if data.dtype == np.int16:
                data = data.astype(np.float32) / 32768.0
            elif data.dtype == np.int32:
                data = data.astype(np.float32) / 2147483648.0
            elif data.dtype != np.float32:
                data = data.astype(np.float32)
            if data.ndim == 2:
                data = data.mean(axis=1)
            waveform = torch.from_numpy(data).unsqueeze(0)
            orig_sr = sample_rate_file
        except Exception:
            waveform, orig_sr = torchaudio.load(path)
        if orig_sr != self.sr:
            resampler = T.Resample(orig_sr, self.sr)
            waveform = resampler(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if self.config.trim_silence:
            waveform = self._trim_silence(waveform)
        return waveform, self.sr

    def _trim_silence(self, waveform: torch.Tensor, top_db: float = 40.0) -> torch.Tensor:
        """Trim leading/trailing silence from waveform."""
        rms = torch.sqrt(torch.mean(waveform ** 2, dim=-1))
        thresh = 10 ** (-top_db / 20.0) * waveform.abs().max()
        mask = waveform.abs() > thresh
        if mask.any():
            start = mask.float().argmax().item()
            end = len(mask[0]) - mask[0].flip(0).float().argmax().item()
            waveform = waveform[:, int(start):int(end)]
        return waveform

    def preemphasis(self, waveform: torch.Tensor) -> torch.Tensor:
        """Apply preemphasis filter."""
        coef = self.config.preemphasis
        filtered = torch.cat([waveform[:, :1], waveform[:, 1:] - coef * waveform[:, :-1]], dim=1)
        return filtered

    def get_mel_spectrogram(self, waveform: torch.Tensor) -> torch.Tensor:
        """Compute log-mel spectrogram. Returns [n_mels, T]."""
        waveform = self.preemphasis(waveform)
        mel = self.mel_transform(waveform)
        mel_db = self.amplitude_to_db(mel)
        mel_db = mel_db.squeeze(0)
        return mel_db

    def get_pitch(self, waveform: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Extract F0 (fundamental frequency) using autocorrelation.
        Returns f0_hz [T_frames] and voiced_flag [T_frames].
        """
        wav_np = waveform.squeeze().numpy()
        f0, voiced = self._yin_f0(wav_np, self.sr, self.hop)
        f0_tensor = torch.from_numpy(f0).float()
        voiced_tensor = torch.from_numpy(voiced).float()
        return f0_tensor, voiced_tensor

    def _yin_f0(
        self,
        wav: np.ndarray,
        sr: int,
        hop: int,
        f_min: float = 80.0,
        f_max: float = 400.0,
        threshold: float = 0.12,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        YIN algorithm for fundamental frequency estimation.
        Based on: de Cheveigne & Kawahara (2002).
        """
        frame_size = 1024
        min_lag = int(sr / f_max)
        max_lag = int(sr / f_min)

        num_frames = (len(wav) - frame_size) // hop + 1
        f0 = np.zeros(num_frames)
        voiced = np.zeros(num_frames, dtype=bool)

        for i in range(num_frames):
            start = i * hop
            frame = wav[start: start + frame_size]
            if len(frame) < frame_size:
                break
            df = self._difference_function(frame, frame_size, max_lag)
            cmdf = self._cumulative_mean_normalized(df, max_lag)
            tau = self._absolute_threshold(cmdf, min_lag, max_lag, threshold)
            if tau is not None:
                f0[i] = sr / self._parabolic_interpolation(cmdf, tau)
                voiced[i] = True

        f0 = self._smooth_f0(f0, voiced)
        f0_log = np.where(voiced, np.log(np.maximum(f0, 1e-5)), 0.0)
        return f0_log.astype(np.float32), voiced.astype(np.float32)

    def _difference_function(self, frame: np.ndarray, N: int, max_lag: int) -> np.ndarray:
        df = np.zeros(max_lag)
        for tau in range(1, max_lag):
            diff = frame[:N - tau] - frame[tau:N]
            df[tau] = np.sum(diff ** 2)
        return df

    def _cumulative_mean_normalized(self, df: np.ndarray, max_lag: int) -> np.ndarray:
        cmdf = np.zeros(max_lag)
        cmdf[0] = 1.0
        cumsum = 0.0
        for tau in range(1, max_lag):
            cumsum += df[tau]
            if cumsum > 0:
                cmdf[tau] = df[tau] * tau / cumsum
            else:
                cmdf[tau] = 1.0
        return cmdf

    def _absolute_threshold(
        self,
        cmdf: np.ndarray,
        min_lag: int,
        max_lag: int,
        threshold: float
    ) -> Optional[int]:
        tau = min_lag
        while tau < max_lag:
            if cmdf[tau] < threshold:
                while tau + 1 < max_lag and cmdf[tau + 1] < cmdf[tau]:
                    tau += 1
                return tau
            tau += 1
        return int(np.argmin(cmdf[min_lag:max_lag])) + min_lag if max_lag > min_lag else None

    def _parabolic_interpolation(self, array: np.ndarray, x: int) -> float:
        if x <= 0 or x >= len(array) - 1:
            return float(x)
        denom = 2.0 * (2.0 * array[x] - array[x - 1] - array[x + 1])
        if abs(denom) < 1e-9:
            return float(x)
        delta = (array[x - 1] - array[x + 1]) / denom
        return float(x) + delta

    def _smooth_f0(self, f0: np.ndarray, voiced: np.ndarray, kernel_size: int = 3) -> np.ndarray:
        smoothed = f0.copy()
        for i in range(kernel_size // 2, len(f0) - kernel_size // 2):
            if voiced[i]:
                window = f0[i - kernel_size // 2: i + kernel_size // 2 + 1]
                smoothed[i] = np.mean(window[voiced[i - kernel_size // 2: i + kernel_size // 2 + 1]])
        return smoothed

    def get_energy(self, mel: torch.Tensor) -> torch.Tensor:
        """
        Compute frame-level energy from mel spectrogram.
        Returns [T_frames] energy values.
        """
        energy = torch.norm(mel, dim=0, p=2)
        return energy

    def get_duration(self, waveform: torch.Tensor) -> torch.Tensor:
        """
        Estimate pseudo-phoneme durations using energy-based segmentation.
        Returns list of frame durations per segment.
        """
        mel = self.get_mel_spectrogram(waveform)
        energy = self.get_energy(mel)
        T = energy.shape[0]
        segment_len = max(5, T // 20)
        durations = []
        for i in range(0, T, segment_len):
            end = min(i + segment_len, T)
            durations.append(end - i)
        return torch.tensor(durations, dtype=torch.float32)

    def extract_all_features(self, waveform: torch.Tensor) -> Dict[str, torch.Tensor]:
        """Extract all features needed by the model."""
        mel = self.get_mel_spectrogram(waveform)
        f0, voiced = self.get_pitch(waveform)
        energy = self.get_energy(mel)
        duration = self.get_duration(waveform)

        T_mel = mel.shape[1]
        if f0.shape[0] != T_mel:
            f0 = F.interpolate(f0.unsqueeze(0).unsqueeze(0), size=T_mel, mode='linear', align_corners=False).squeeze()
            voiced = F.interpolate(voiced.unsqueeze(0).unsqueeze(0), size=T_mel, mode='nearest').squeeze()
        if energy.shape[0] != T_mel:
            energy = F.interpolate(energy.unsqueeze(0).unsqueeze(0), size=T_mel, mode='linear', align_corners=False).squeeze()

        return {
            "mel": mel,
            "f0": f0,
            "voiced": voiced,
            "energy": energy,
            "duration": duration,
            "T": T_mel,
        }

    def save_audio(self, waveform: torch.Tensor, path: str, sample_rate: Optional[int] = None):
        """Save waveform to WAV file."""
        sr = sample_rate or self.sr
        wav_np = waveform.squeeze().cpu().numpy()
        wav_int16 = np.clip(wav_np * 32767.0, -32768, 32767).astype(np.int16)
        wav_io.write(path, sr, wav_int16)
        logger.info(f"Saved audio -> {path}")

    def load_or_generate_audio(self, path: Optional[str] = None, duration_sec: float = 3.0) -> torch.Tensor:
        """Load audio or generate demo signal."""
        if path and os.path.exists(path):
            waveform, _ = self.load_audio(path)
            return waveform
        logger.info("No input file found, generating demo audio signal...")
        return self._generate_demo_signal(duration_sec)

    def _generate_demo_signal(self, duration_sec: float = 3.0) -> torch.Tensor:
        """
        Generate a realistic-sounding synthetic voice signal using
        glottal pulse model + formant filtering.
        """
        t = np.linspace(0, duration_sec, int(self.sr * duration_sec), endpoint=False)

        f0_hz = 120.0
        t_pitch = np.linspace(0, duration_sec, len(t))
        f0_contour = f0_hz + 8 * np.sin(2 * np.pi * 0.8 * t_pitch) + \
                     3 * np.sin(2 * np.pi * 3.0 * t_pitch)

        phase = np.cumsum(2 * np.pi * f0_contour / self.sr)
        glottal = np.zeros_like(t)
        for k in range(1, 12):
            amp = 1.0 / (k ** 1.2)
            glottal += amp * np.sin(k * phase)

        formant_freqs = [700, 1200, 2600, 3500]
        formant_bws = [130, 200, 350, 400]
        voiced_signal = np.zeros_like(glottal)
        for ff, bw in zip(formant_freqs, formant_bws):
            b, a = signal.butter(2, [(ff - bw / 2) / (self.sr / 2), (ff + bw / 2) / (self.sr / 2)], btype='band')
            voiced_signal += signal.lfilter(b, a, glottal)

        voiced_signal = voiced_signal / (np.max(np.abs(voiced_signal)) + 1e-8)

        silence_samples = int(0.1 * self.sr)
        speech_on_off = np.ones(len(t))
        speech_on_off[:silence_samples] = 0
        speech_on_off[-silence_samples:] = 0
        mid = len(t) // 2
        speech_on_off[mid: mid + silence_samples // 2] = 0

        amplitude_envelope = np.random.uniform(0.7, 1.0, len(t) // (self.sr // 10) + 1)
        amplitude_envelope = np.repeat(amplitude_envelope, self.sr // 10)[:len(t)]
        amplitude_envelope = np.convolve(amplitude_envelope, np.ones(self.sr // 20) / (self.sr // 20), mode='same')

        signal_out = voiced_signal * speech_on_off * amplitude_envelope
        noise = np.random.randn(len(t)) * 0.005
        signal_out = signal_out + noise
        signal_out = signal_out / (np.max(np.abs(signal_out)) + 1e-8) * 0.9

        return torch.from_numpy(signal_out.astype(np.float32)).unsqueeze(0)


# =============================================================================
# SECTION 3: VARIATIONAL AUTOENCODER (VAE)
# =============================================================================

class ResidualBlock1D(nn.Module):
    """1D Residual block with layer normalization."""

    def __init__(self, channels: int, kernel_size: int = 3, dilation: int = 1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, dilation=dilation, padding=padding)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, dilation=dilation, padding=padding)
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = x.transpose(1, 2)
        x = self.norm1(x)
        x = x.transpose(1, 2)
        x = self.act(self.conv1(x))
        x = x.transpose(1, 2)
        x = self.norm2(x)
        x = x.transpose(1, 2)
        x = self.conv2(x)
        return x + residual


class VAEEncoder(nn.Module):
    """
    Variational Autoencoder Encoder.
    Input: Mel spectrogram [B, n_mels, T]
    Output: mu, log_var [B, latent_dim]
    """

    def __init__(self, n_mels: int, latent_dim: int, hidden_dim: int, num_layers: int):
        super().__init__()
        self.n_mels = n_mels
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim

        self.input_proj = nn.Conv1d(n_mels, hidden_dim, kernel_size=1)

        self.res_blocks = nn.ModuleList([
            ResidualBlock1D(hidden_dim, kernel_size=3, dilation=2 ** i)
            for i in range(num_layers)
        ])

        self.conv_downsample = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.Conv1d(hidden_dim * 2, hidden_dim * 2, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.Conv1d(hidden_dim * 2, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

        self.dropout = nn.Dropout(0.1)

    def forward(self, mel: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.input_proj(mel)
        for block in self.res_blocks:
            x = block(x)
        x = self.dropout(x)
        x = self.conv_downsample(x)
        x = self.pool(x).squeeze(-1)
        mu = self.fc_mu(x)
        log_var = self.fc_log_var(x)
        log_var = torch.clamp(log_var, -10.0, 4.0)
        return mu, log_var


class VAEDecoder(nn.Module):
    """
    Variational Autoencoder Decoder.
    Input: latent z [B, latent_dim], target_len T
    Output: mel [B, n_mels, T]
    """

    def __init__(self, n_mels: int, latent_dim: int, hidden_dim: int, num_layers: int):
        super().__init__()
        self.n_mels = n_mels
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim

        self.fc = nn.Linear(latent_dim, hidden_dim * 8)

        self.upsample = nn.Sequential(
            nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
        )

        self.res_blocks = nn.ModuleList([
            ResidualBlock1D(hidden_dim, kernel_size=3, dilation=2 ** i)
            for i in range(num_layers)
        ])

        self.output_proj = nn.Conv1d(hidden_dim, n_mels, kernel_size=1)

    def forward(self, z: torch.Tensor, target_len: int) -> torch.Tensor:
        x = self.fc(z)
        B = z.shape[0]
        x = x.view(B, self.hidden_dim, 8)
        x = self.upsample(x)
        for block in self.res_blocks:
            x = block(x)
        x = self.output_proj(x)
        if x.shape[-1] != target_len:
            x = F.interpolate(x, size=target_len, mode='linear', align_corners=False)
        return x


class VoiceVAE(nn.Module):
    """
    Full Variational Autoencoder for voice feature encoding.
    Also encodes emotion-conditioning and disentangles speaker identity.
    """

    def __init__(self, config: ModelConfig, n_mels: int = 80):
        super().__init__()
        self.config = config
        self.n_mels = n_mels
        self.latent_dim = config.latent_dim

        self.encoder = VAEEncoder(
            n_mels=n_mels,
            latent_dim=config.latent_dim,
            hidden_dim=config.vae_hidden_dim,
            num_layers=config.vae_num_layers
        )

        self.decoder = VAEDecoder(
            n_mels=n_mels,
            latent_dim=config.latent_dim,
            hidden_dim=config.vae_hidden_dim,
            num_layers=config.vae_num_layers
        )

        self.emotion_embedding = nn.Embedding(config.num_emotions, config.latent_dim)
        nn.init.normal_(self.emotion_embedding.weight, 0.0, 0.1)

        self.emotion_mapper = nn.Sequential(
            nn.Linear(config.latent_dim * 2, config.latent_dim * 2),
            nn.GELU(),
            nn.Linear(config.latent_dim * 2, config.latent_dim),
            nn.Tanh()
        )

        self.pitch_predictor = nn.Sequential(
            nn.Linear(config.latent_dim, config.vae_hidden_dim),
            nn.GELU(),
            nn.Linear(config.vae_hidden_dim, 1)
        )
        self.energy_predictor = nn.Sequential(
            nn.Linear(config.latent_dim, config.vae_hidden_dim),
            nn.GELU(),
            nn.Linear(config.vae_hidden_dim, 1)
        )

    def encode(self, mel: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.encoder(mel)

    def reparameterize(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def decode(self, z: torch.Tensor, target_len: int) -> torch.Tensor:
        return self.decoder(z, target_len)

    def condition_on_emotion(self, z: torch.Tensor, emotion_id: torch.Tensor) -> torch.Tensor:
        """Shift latent code towards target emotion."""
        emotion_emb = self.emotion_embedding(emotion_id)
        combined = torch.cat([z, emotion_emb], dim=-1)
        z_emotion = self.emotion_mapper(combined)
        return z_emotion

    def predict_pitch(self, z: torch.Tensor) -> torch.Tensor:
        return self.pitch_predictor(z).squeeze(-1)

    def predict_energy(self, z: torch.Tensor) -> torch.Tensor:
        return self.energy_predictor(z).squeeze(-1)

    def forward(
        self,
        mel: torch.Tensor,
        emotion_id: Optional[torch.Tensor] = None
    ) -> Dict[str, torch.Tensor]:
        B, n_mels, T = mel.shape
        mu, log_var = self.encode(mel)
        z = self.reparameterize(mu, log_var)

        if emotion_id is not None:
            z_conditioned = self.condition_on_emotion(z, emotion_id)
        else:
            z_conditioned = z

        mel_recon = self.decode(z_conditioned, T)
        pitch_pred = self.predict_pitch(z_conditioned)
        energy_pred = self.predict_energy(z_conditioned)

        return {
            "mel_recon": mel_recon,
            "mu": mu,
            "log_var": log_var,
            "z": z,
            "z_conditioned": z_conditioned,
            "pitch_pred": pitch_pred,
            "energy_pred": energy_pred,
        }

    def get_kl_loss(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        kl = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
        return kl


# =============================================================================
# SECTION 4: GRAPH NEURAL NETWORK (GNN) FOR EMOTION MAPPING
# =============================================================================

class GraphAttentionLayer(nn.Module):
    """
    Multi-head Graph Attention Layer (GAT).
    Computes attention over graph edges to propagate emotion features.
    """

    def __init__(self, in_features: int, out_features: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.num_heads = num_heads
        self.head_dim = out_features // num_heads

        self.W_q = nn.Linear(in_features, out_features)
        self.W_k = nn.Linear(in_features, out_features)
        self.W_v = nn.Linear(in_features, out_features)
        self.W_o = nn.Linear(out_features, out_features)

        self.attn_weight = nn.Parameter(torch.randn(num_heads, self.head_dim * 2))
        nn.init.xavier_uniform_(self.attn_weight.unsqueeze(0))

        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(out_features)
        self.scale = math.sqrt(self.head_dim)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_weight: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        x: [N, in_features]
        edge_index: [2, E]  (source, target)
        edge_weight: [E] optional
        """
        N = x.shape[0]
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(N, self.num_heads, self.head_dim)
        K = K.view(N, self.num_heads, self.head_dim)
        V = V.view(N, self.num_heads, self.head_dim)

        src_idx = edge_index[0]
        tgt_idx = edge_index[1]

        q_src = Q[src_idx]
        k_tgt = K[tgt_idx]
        v_tgt = V[tgt_idx]

        attn_input = torch.cat([q_src, k_tgt], dim=-1)
        e = (attn_input * self.attn_weight.unsqueeze(0)).sum(dim=-1)
        e = self.leaky_relu(e)

        if edge_weight is not None:
            e = e * edge_weight.unsqueeze(-1)

        attn = torch.zeros(N, self.num_heads, device=x.device)
        exp_e = torch.exp(e - e.max())
        attn.index_add_(0, tgt_idx, exp_e)
        attn_norm = attn[tgt_idx] + 1e-8
        alpha = exp_e / attn_norm
        alpha = self.dropout(alpha)

        weighted_v = v_tgt * alpha.unsqueeze(-1)
        out = torch.zeros(N, self.num_heads, self.head_dim, device=x.device)
        out.index_add_(0, tgt_idx, weighted_v)
        out = out.view(N, -1)
        out = self.W_o(out)
        out = self.norm(out + x if out.shape == x.shape else out)
        return out


class EmotionGraphNode:
    """
    Represents a node in the emotion graph.
    Each emotion is a node; voice features are node attributes.
    """

    def __init__(
        self,
        emotion_name: str,
        emotion_id: int,
        feature_vector: np.ndarray
    ):
        self.emotion_name = emotion_name
        self.emotion_id = emotion_id
        self.feature_vector = feature_vector


class EmotionGNN(nn.Module):
    """
    Graph Neural Network for emotion-to-feature mapping.

    Architecture:
    - Nodes: Each emotion is a node with feature attributes
    - Edges: Connections between related emotions (valence/arousal proximity)
    - Message passing learns how to transform voice features across emotions
    - Output: Transformed feature vector for target emotion

    The graph captures emotional relationships:
    - happy <-> surprised (both high arousal, positive valence)
    - sad <-> fearful (both low valence)
    - angry <-> disgusted (both negative valence, different arousal)
    - neutral <-> all (neutral is the anchor)
    """

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.num_emotions = config.num_emotions
        self.hidden_dim = config.gnn_hidden_dim
        self.latent_dim = config.latent_dim

        # Emotion node feature dimension: latent_dim + prosody_features (6)
        self.node_feat_dim = config.latent_dim + 6

        self.node_encoder = nn.Sequential(
            nn.Linear(self.node_feat_dim, self.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(self.hidden_dim, self.hidden_dim),
        )

        self.gat_layers = nn.ModuleList([
            GraphAttentionLayer(
                in_features=self.hidden_dim,
                out_features=self.hidden_dim,
                num_heads=config.gnn_num_heads,
                dropout=config.dropout
            )
            for _ in range(config.gnn_num_layers)
        ])

        self.ffn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.hidden_dim, self.hidden_dim * 2),
                nn.GELU(),
                nn.Dropout(config.dropout),
                nn.Linear(self.hidden_dim * 2, self.hidden_dim),
            )
            for _ in range(config.gnn_num_layers)
        ])

        self.norm_layers = nn.ModuleList([
            nn.LayerNorm(self.hidden_dim)
            for _ in range(config.gnn_num_layers)
        ])

        self.output_proj = nn.Sequential(
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.GELU(),
            nn.Linear(self.hidden_dim, config.latent_dim),
            nn.Tanh()
        )

        self.prosody_decoder = nn.Sequential(
            nn.Linear(config.latent_dim, self.hidden_dim),
            nn.GELU(),
            nn.Linear(self.hidden_dim, 6),
        )

        self.edge_index, self.edge_weight = self._build_emotion_graph()

        self.emotion_base_nodes = nn.Parameter(
            torch.randn(config.num_emotions, self.node_feat_dim) * 0.1
        )

    def _build_emotion_graph(self) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Build emotion graph based on valence-arousal space.
        Emotions closer in VA space have stronger edges.

        Valence-Arousal positions (approximate):
          neutral:   V=0.5, A=0.5
          happy:     V=0.9, A=0.7
          sad:       V=0.1, A=0.2
          angry:     V=0.1, A=0.9
          fearful:   V=0.2, A=0.8
          surprised: V=0.6, A=0.9
          disgusted: V=0.1, A=0.6
        """
        va_positions = {
            "neutral":   (0.5, 0.5),
            "happy":     (0.9, 0.7),
            "sad":       (0.1, 0.2),
            "angry":     (0.1, 0.9),
            "fearful":   (0.2, 0.8),
            "surprised": (0.6, 0.9),
            "disgusted": (0.1, 0.6),
        }

        emotions = list(EMOTION_PROFILES.keys())
        n = len(emotions)
        edges_src = []
        edges_tgt = []
        weights = []

        for i, e1 in enumerate(emotions):
            for j, e2 in enumerate(emotions):
                if i != j:
                    v1, a1 = va_positions[e1]
                    v2, a2 = va_positions[e2]
                    dist = math.sqrt((v1 - v2) ** 2 + (a1 - a2) ** 2)
                    max_dist = math.sqrt(2.0)
                    similarity = 1.0 - dist / max_dist
                    if similarity > 0.3 or e1 == "neutral" or e2 == "neutral":
                        edges_src.append(i)
                        edges_tgt.append(j)
                        weights.append(similarity)

        edge_index = torch.tensor([edges_src, edges_tgt], dtype=torch.long)
        edge_weight = torch.tensor(weights, dtype=torch.float32)
        return edge_index, edge_weight

    def forward(
        self,
        z_source: torch.Tensor,
        source_emotion_id: torch.Tensor,
        target_emotion_id: torch.Tensor,
        source_prosody: Optional[torch.Tensor] = None
    ) -> Dict[str, torch.Tensor]:
        """
        z_source: [B, latent_dim] — source voice latent
        source_emotion_id: [B] — source emotion index
        target_emotion_id: [B] — target emotion index
        source_prosody: [B, 6] — [pitch_mean, pitch_std, energy_mean, energy_std, speed, duration]
        """
        B = z_source.shape[0]
        device = z_source.device

        edge_index = self.edge_index.to(device)
        edge_weight = self.edge_weight.to(device)

        if source_prosody is None:
            source_prosody = torch.zeros(B, 6, device=device)

        results = []
        for b in range(B):
            z_b = z_source[b].unsqueeze(0)
            src_id = source_emotion_id[b].item()
            tgt_id = target_emotion_id[b].item()
            prosody_b = source_prosody[b].unsqueeze(0)

            node_feats = self.emotion_base_nodes.clone()
            source_feat = torch.cat([z_b, prosody_b], dim=-1)
            node_feats[src_id] = node_feats[src_id] + source_feat.squeeze(0)

            x = self.node_encoder(node_feats)

            for layer_idx in range(len(self.gat_layers)):
                x_attn = self.gat_layers[layer_idx](x, edge_index, edge_weight)
                x_ffn = self.ffn_layers[layer_idx](x_attn)
                x = self.norm_layers[layer_idx](x_ffn + x)

            target_node = x[tgt_id].unsqueeze(0)
            z_target = self.output_proj(target_node)
            results.append(z_target)

        z_out = torch.cat(results, dim=0)
        prosody_out = self.prosody_decoder(z_out)

        return {
            "z_target": z_out,
            "prosody_target": prosody_out,
        }

    def get_emotion_embeddings(self) -> Dict[str, torch.Tensor]:
        """Return current learned emotion node representations."""
        embeddings = {}
        for i, name in enumerate(EMOTION_PROFILES.keys()):
            embeddings[name] = self.emotion_base_nodes[i].detach()
        return embeddings


# =============================================================================
# SECTION 5: FASTSPEECH 2 VARIANCE ADAPTOR
# =============================================================================

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)


class FFTBlock(nn.Module):
    """
    Feed-Forward Transformer Block used in FastSpeech2.
    Multi-head self-attention + position-wise FFN with Conv1d.
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        kernel_size: int = 9,
        dropout: float = 0.1
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        padding = (kernel_size - 1) // 2
        self.ffn = nn.Sequential(
            nn.Conv1d(d_model, d_ff, kernel_size=kernel_size, padding=padding),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(d_ff, d_model, kernel_size=kernel_size, padding=padding),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        attn_out, _ = self.self_attn(x, x, x, key_padding_mask=mask)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_in = x.transpose(1, 2)
        ffn_out = self.ffn(ffn_in).transpose(1, 2)
        x = self.norm2(x + self.dropout(ffn_out))
        return x


class VariancePredictor(nn.Module):
    """
    FastSpeech 2 Variance Predictor.
    Predicts frame-level pitch / energy / duration from hidden features.
    """

    def __init__(self, in_channels: int, out_channels: int = 1, kernel_size: int = 3):
        super().__init__()
        padding = (kernel_size - 1) // 2
        self.layers = nn.Sequential(
            nn.Conv1d(in_channels, in_channels, kernel_size=kernel_size, padding=padding),
            nn.GELU(),
            nn.LayerNorm(in_channels),
            nn.Conv1d(in_channels, in_channels, kernel_size=kernel_size, padding=padding),
            nn.GELU(),
            nn.LayerNorm(in_channels),
            nn.Linear(in_channels, out_channels),
        )
        self._init_layers()

    def _init_layers(self):
        for layer in self.layers:
            if isinstance(layer, (nn.Conv1d, nn.Linear)):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, T, C] -> out: [B, T, out_channels]"""
        out = x
        for layer in self.layers:
            if isinstance(layer, nn.Conv1d):
                out = layer(out.transpose(1, 2)).transpose(1, 2)
            elif isinstance(layer, nn.LayerNorm):
                out = layer(out)
            else:
                out = layer(out)
        return out


class LengthRegulator(nn.Module):
    """
    FastSpeech 2 Length Regulator.
    Expands/contracts sequence according to predicted durations.
    """

    def __init__(self):
        super().__init__()

    def forward(
        self,
        x: torch.Tensor,
        duration: torch.Tensor,
        target_len: Optional[int] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        x: [B, T_phone, C]
        duration: [B, T_phone] integer durations
        Returns regulated x: [B, T_mel, C]
        """
        output = []
        out_lens = []
        for b in range(x.shape[0]):
            dur = duration[b].clamp(min=0).long()
            if dur.sum() == 0:
                dur = dur + 1
            regulated = torch.repeat_interleave(x[b], dur, dim=0)
            output.append(regulated)
            out_lens.append(regulated.shape[0])

        max_len = max(out_lens) if target_len is None else target_len
        padded = torch.zeros(x.shape[0], max_len, x.shape[-1], device=x.device)
        for b, (out, length) in enumerate(zip(output, out_lens)):
            actual_len = min(length, max_len)
            padded[b, :actual_len] = out[:actual_len]

        out_lens_tensor = torch.tensor(out_lens, dtype=torch.long, device=x.device)
        return padded, out_lens_tensor


class PitchEmbedding(nn.Module):
    """Quantized pitch embedding with continuous interpolation."""

    def __init__(self, num_bins: int, d_model: int, f0_min: float = -3.0, f0_max: float = 3.0):
        super().__init__()
        self.num_bins = num_bins
        self.f0_min = f0_min
        self.f0_max = f0_max
        self.embedding = nn.Embedding(num_bins + 1, d_model)
        self.register_buffer('bins', torch.linspace(f0_min, f0_max, num_bins))

    def forward(self, f0: torch.Tensor) -> torch.Tensor:
        bins = self.bins.to(f0.device)
        f0_clamp = f0.clamp(self.f0_min, self.f0_max)
        indices = torch.bucketize(f0_clamp, bins).clamp(0, self.num_bins)
        return self.embedding(indices)


class EnergyEmbedding(nn.Module):
    """Quantized energy embedding."""

    def __init__(self, num_bins: int, d_model: int, e_min: float = 0.0, e_max: float = 150.0):
        super().__init__()
        self.num_bins = num_bins
        self.e_min = e_min
        self.e_max = e_max
        self.embedding = nn.Embedding(num_bins + 1, d_model)
        self.register_buffer('bins', torch.linspace(e_min, e_max, num_bins))

    def forward(self, energy: torch.Tensor) -> torch.Tensor:
        bins = self.bins.to(energy.device)
        e_clamp = energy.clamp(self.e_min, self.e_max)
        indices = torch.bucketize(e_clamp, bins).clamp(0, self.num_bins)
        return self.embedding(indices)


class FastSpeech2VarianceAdaptor(nn.Module):
    """
    FastSpeech 2 Variance Adaptor:
    - Duration Predictor -> Length Regulator (controls speech rate)
    - Pitch Predictor + Embedding (controls intonation)
    - Energy Predictor + Embedding (controls loudness/intensity)
    - Emotion conditioning via GNN output
    """

    def __init__(self, config: ModelConfig, n_mels: int = 80):
        super().__init__()
        self.config = config
        d_model = config.fastspeech_hidden
        self.n_mels = n_mels

        self.input_proj = nn.Linear(config.latent_dim, d_model)

        self.phoneme_encoder = nn.ModuleList([
            FFTBlock(
                d_model=d_model,
                num_heads=config.fastspeech_num_heads,
                d_ff=config.fastspeech_filter_size,
                kernel_size=config.fastspeech_kernel_size,
                dropout=config.dropout
            )
            for _ in range(config.fastspeech_num_layers)
        ])

        self.duration_predictor = VariancePredictor(d_model, out_channels=1)
        self.length_regulator = LengthRegulator()
        self.pitch_predictor = VariancePredictor(d_model, out_channels=1)
        self.energy_predictor = VariancePredictor(d_model, out_channels=1)

        self.pitch_embedding = PitchEmbedding(config.pitch_bins, d_model)
        self.energy_embedding = EnergyEmbedding(config.energy_bins, d_model)

        self.mel_encoder = nn.ModuleList([
            FFTBlock(
                d_model=d_model,
                num_heads=config.fastspeech_num_heads,
                d_ff=config.fastspeech_filter_size,
                kernel_size=config.fastspeech_kernel_size,
                dropout=config.dropout
            )
            for _ in range(config.fastspeech_num_layers)
        ])

        self.mel_proj = nn.Linear(d_model, n_mels)

        self.emotion_proj = nn.Linear(config.latent_dim, d_model)
        self.emotion_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )

        self.pos_enc = PositionalEncoding(d_model, dropout=config.dropout)

    def forward(
        self,
        z: torch.Tensor,
        emotion_z: torch.Tensor,
        target_mel_len: int,
        target_f0: Optional[torch.Tensor] = None,
        target_energy: Optional[torch.Tensor] = None,
        target_duration: Optional[torch.Tensor] = None,
        emotion_profile: Optional[EmotionProfile] = None,
    ) -> Dict[str, torch.Tensor]:
        """
        z: [B, latent_dim] — source content latent
        emotion_z: [B, latent_dim] — emotion-conditioned latent from GNN
        target_mel_len: int — desired output mel length
        """
        B = z.shape[0]
        device = z.device

        h = self.input_proj(z).unsqueeze(1)
        h = h.expand(-1, max(target_mel_len // 8, 4), -1)

        h = self.pos_enc(h.transpose(0, 1)).transpose(0, 1)

        for block in self.phoneme_encoder:
            h = block(h)

        emotion_h = self.emotion_proj(emotion_z).unsqueeze(1).expand_as(h)
        gate = self.emotion_gate(torch.cat([h, emotion_h], dim=-1))
        h = h * gate + emotion_h * (1 - gate)

        dur_pred = self.duration_predictor(h).squeeze(-1)
        dur_pred = F.softplus(dur_pred) + 1.0

        if emotion_profile is not None:
            dur_pred = dur_pred * emotion_profile.duration_factor

        if target_duration is not None:
            dur_for_reg = target_duration.unsqueeze(0).expand(B, -1) if target_duration.dim() == 1 else target_duration
            dur_int = dur_for_reg.clamp(min=1)
        else:
            total_dur = dur_pred.sum(dim=-1, keepdim=True)
            scale = target_mel_len / (total_dur + 1e-8)
            dur_int = (dur_pred * scale).round().clamp(min=1).long()

        h_expanded, _ = self.length_regulator(h, dur_int, target_mel_len)

        pitch_pred = self.pitch_predictor(h_expanded).squeeze(-1)
        energy_pred = self.energy_predictor(h_expanded).squeeze(-1)

        if emotion_profile is not None:
            pitch_pred = pitch_pred + emotion_profile.pitch_shift * 0.1
            pitch_std = pitch_pred.std(dim=-1, keepdim=True)
            pitch_mean = pitch_pred.mean(dim=-1, keepdim=True)
            pitch_pred = pitch_mean + (pitch_pred - pitch_mean) * (
                1.0 + emotion_profile.pitch_variance * 2.0
            )
            energy_pred = energy_pred * emotion_profile.energy_scale
            energy_std = energy_pred.std(dim=-1, keepdim=True)
            energy_pred = energy_pred + energy_std * emotion_profile.energy_variance

        if target_f0 is not None:
            f0_for_emb = target_f0
        else:
            f0_for_emb = pitch_pred

        if target_energy is not None:
            e_for_emb = target_energy
        else:
            e_for_emb = energy_pred

        pitch_emb = self.pitch_embedding(f0_for_emb)
        energy_emb = self.energy_embedding(e_for_emb.clamp(0, 150))

        h_expanded = h_expanded + pitch_emb + energy_emb

        h_expanded = self.pos_enc(h_expanded.transpose(0, 1)).transpose(0, 1)
        for block in self.mel_encoder:
            h_expanded = block(h_expanded)

        mel_out = self.mel_proj(h_expanded)
        mel_out = mel_out.transpose(1, 2)

        if mel_out.shape[-1] != target_mel_len:
            mel_out = F.interpolate(mel_out, size=target_mel_len, mode='linear', align_corners=False)

        return {
            "mel": mel_out,
            "pitch_pred": pitch_pred,
            "energy_pred": energy_pred,
            "duration_pred": dur_pred,
        }


# =============================================================================
# SECTION 6: HIFI-GAN VOCODER
# =============================================================================

class ResBlock(nn.Module):
    """HiFi-GAN residual block with dilated convolutions for natural-sounding waveform."""

    def __init__(self, channels: int, kernel_size: int = 3, dilations: List[int] = (1, 3, 5)):
        super().__init__()
        self.convs1 = nn.ModuleList([
            nn.Sequential(
                nn.LeakyReLU(0.1),
                nn.utils.weight_norm(
                    nn.Conv1d(channels, channels, kernel_size, dilation=d,
                              padding=self._get_padding(kernel_size, d))
                )
            )
            for d in dilations
        ])
        self.convs2 = nn.ModuleList([
            nn.Sequential(
                nn.LeakyReLU(0.1),
                nn.utils.weight_norm(
                    nn.Conv1d(channels, channels, kernel_size, dilation=1,
                              padding=self._get_padding(kernel_size, 1))
                )
            )
            for _ in dilations
        ])

    @staticmethod
    def _get_padding(k: int, d: int) -> int:
        return (k * d - d) // 2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for c1, c2 in zip(self.convs1, self.convs2):
            xt = c1(x)
            xt = c2(xt)
            x = xt + x
        return x

    def remove_weight_norm(self):
        for layer in self.convs1:
            for sublayer in layer:
                if hasattr(sublayer, 'weight'):
                    try:
                        nn.utils.remove_weight_norm(sublayer)
                    except ValueError:
                        pass
        for layer in self.convs2:
            for sublayer in layer:
                if hasattr(sublayer, 'weight'):
                    try:
                        nn.utils.remove_weight_norm(sublayer)
                    except ValueError:
                        pass


class HiFiGANGenerator(nn.Module):
    """
    HiFi-GAN Generator (V1).
    Converts mel spectrogram to natural-sounding waveform via
    transposed convolutions and multi-receptive field fusion (MRF).

    Key property: multi-scale dilated residual blocks ensure
    the output captures both fine-grained and coarse speech features,
    resulting in natural-sounding audio (not robotic/AI-sounding).
    """

    def __init__(self, config: ModelConfig, n_mels: int = 80):
        super().__init__()
        self.config = config
        self.n_mels = n_mels

        self.pre_conv = nn.utils.weight_norm(
            nn.Conv1d(n_mels, config.hifigan_upsample_initial_channel, 7, 1, padding=3)
        )

        self.ups = nn.ModuleList()
        self.resblocks = nn.ModuleList()

        in_channels = config.hifigan_upsample_initial_channel
        for i, (u, k) in enumerate(zip(config.hifigan_upsample_rates, config.hifigan_upsample_kernel_sizes)):
            out_channels = in_channels // 2
            self.ups.append(
                nn.utils.weight_norm(
                    nn.ConvTranspose1d(in_channels, out_channels, k, u, padding=(k - u) // 2)
                )
            )
            for j in range(len(config.hifigan_resblock_kernel_sizes)):
                self.resblocks.append(
                    ResBlock(
                        out_channels,
                        config.hifigan_resblock_kernel_sizes[j],
                        config.hifigan_resblock_dilation_sizes[j]
                    )
                )
            in_channels = out_channels

        self.post_conv = nn.utils.weight_norm(
            nn.Conv1d(in_channels, 1, 7, 1, padding=3)
        )

        self.num_kernels = len(config.hifigan_resblock_kernel_sizes)
        self.num_upsamples = len(config.hifigan_upsample_rates)

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        """mel: [B, n_mels, T] -> waveform: [B, 1, T*hop_length]"""
        x = self.pre_conv(mel)
        for i in range(self.num_upsamples):
            x = F.leaky_relu(x, 0.1)
            x = self.ups[i](x)
            xs = None
            for j in range(self.num_kernels):
                if xs is None:
                    xs = self.resblocks[i * self.num_kernels + j](x)
                else:
                    xs += self.resblocks[i * self.num_kernels + j](x)
            x = xs / self.num_kernels
        x = F.leaky_relu(x)
        x = self.post_conv(x)
        x = torch.tanh(x)
        return x

    def remove_weight_norm(self):
        try:
            nn.utils.remove_weight_norm(self.pre_conv)
        except ValueError:
            pass
        for up in self.ups:
            try:
                nn.utils.remove_weight_norm(up)
            except ValueError:
                pass
        for block in self.resblocks:
            block.remove_weight_norm()
        try:
            nn.utils.remove_weight_norm(self.post_conv)
        except ValueError:
            pass


class HiFiGANMultiScaleDiscriminator(nn.Module):
    """Multi-Scale Discriminator for HiFi-GAN training."""

    def __init__(self):
        super().__init__()
        self.discriminators = nn.ModuleList([
            self._make_discriminator(),
            self._make_discriminator(),
            self._make_discriminator(),
        ])
        self.pooling = nn.ModuleList([
            nn.Identity(),
            nn.AvgPool1d(4, 2, padding=2),
            nn.AvgPool1d(4, 2, padding=2),
        ])

    def _make_discriminator(self):
        return nn.Sequential(
            nn.utils.weight_norm(nn.Conv1d(1, 128, 15, 1, padding=7)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(128, 128, 41, 4, groups=4, padding=20)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(128, 256, 41, 4, groups=16, padding=20)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(256, 512, 41, 4, groups=16, padding=20)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(512, 1024, 41, 4, groups=16, padding=20)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(1024, 1024, 5, 1, padding=2)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(1024, 1, 3, 1, padding=1)),
        )

    def forward(self, audio: torch.Tensor) -> List[torch.Tensor]:
        outputs = []
        x = audio
        for disc, pool in zip(self.discriminators, self.pooling):
            x = pool(x)
            y = disc(x)
            outputs.append(y)
        return outputs


class HiFiGANMultiPeriodDiscriminator(nn.Module):
    """Multi-Period Discriminator for HiFi-GAN training."""

    def __init__(self):
        super().__init__()
        self.periods = [2, 3, 5, 7, 11]
        self.discriminators = nn.ModuleList([
            self._make_period_discriminator(p) for p in self.periods
        ])

    def _make_period_discriminator(self, period: int):
        channels = [1, 32, 128, 512, 1024, 1024]
        layers = []
        for i in range(len(channels) - 1):
            k = (5, 1) if i == 0 else (5, 1)
            s = (3, 1) if i < 3 else (1, 1)
            p = (2, 0)
            layers.append(nn.utils.weight_norm(nn.Conv2d(channels[i], channels[i + 1], k, s, padding=p)))
            layers.append(nn.LeakyReLU(0.1))
        layers.append(nn.utils.weight_norm(nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0))))
        return nn.Sequential(*layers)

    def forward(self, audio: torch.Tensor) -> List[torch.Tensor]:
        outputs = []
        for period, disc in zip(self.periods, self.discriminators):
            B, C, T = audio.shape
            pad_size = (period - (T % period)) % period
            x = F.pad(audio, (0, pad_size))
            x = x.view(B, C, -1, period)
            y = disc(x)
            outputs.append(y)
        return outputs


# =============================================================================
# SECTION 7: COMPLETE SPEECH CONVERSION MODEL
# =============================================================================

class EmotionalSpeechConverter(nn.Module):
    """
    Full end-to-end Emotional Speech Conversion System.

    Pipeline:
    1. VAE: audio -> latent z (encodes speaker identity + content)
    2. GNN: (z, source_emotion, target_emotion) -> z_target (maps to emotion)
    3. FastSpeech2: z_target -> mel spectrogram (with pitch/energy/duration control)
    4. HiFi-GAN: mel -> waveform (natural audio)
    """

    def __init__(self, config: ModelConfig, audio_config: AudioConfig):
        super().__init__()
        self.model_config = config
        self.audio_config = audio_config
        n_mels = audio_config.n_mels

        self.vae = VoiceVAE(config, n_mels=n_mels)
        self.gnn = EmotionGNN(config)
        self.variance_adaptor = FastSpeech2VarianceAdaptor(config, n_mels=n_mels)
        self.vocoder = HiFiGANGenerator(config, n_mels=n_mels)

    def forward(
        self,
        mel: torch.Tensor,
        source_emotion_id: torch.Tensor,
        target_emotion_id: torch.Tensor,
        source_prosody: Optional[torch.Tensor] = None,
        target_mel_len: Optional[int] = None,
        emotion_profile: Optional[EmotionProfile] = None,
    ) -> Dict[str, torch.Tensor]:

        B, n_mels, T = mel.shape
        if target_mel_len is None:
            target_mel_len = T

        vae_out = self.vae(mel, emotion_id=source_emotion_id)
        z_source = vae_out["z"]
        mu = vae_out["mu"]
        log_var = vae_out["log_var"]
        mel_recon = vae_out["mel_recon"]

        gnn_out = self.gnn(
            z_source=z_source,
            source_emotion_id=source_emotion_id,
            target_emotion_id=target_emotion_id,
            source_prosody=source_prosody
        )
        z_target = gnn_out["z_target"]
        prosody_target = gnn_out["prosody_target"]

        va_out = self.variance_adaptor(
            z=z_source,
            emotion_z=z_target,
            target_mel_len=target_mel_len,
            emotion_profile=emotion_profile,
        )
        mel_converted = va_out["mel"]

        waveform = self.vocoder(mel_converted)

        return {
            "waveform": waveform,
            "mel_converted": mel_converted,
            "mel_recon": mel_recon,
            "z_source": z_source,
            "z_target": z_target,
            "mu": mu,
            "log_var": log_var,
            "pitch_pred": va_out["pitch_pred"],
            "energy_pred": va_out["energy_pred"],
            "duration_pred": va_out["duration_pred"],
            "prosody_target": prosody_target,
        }

    def convert_emotion(
        self,
        mel: torch.Tensor,
        source_emotion: str,
        target_emotion: str,
        target_mel_len: Optional[int] = None,
    ) -> Dict[str, torch.Tensor]:
        """Convert mel spectrogram from source to target emotion."""
        device = next(self.parameters()).device
        B = mel.shape[0] if mel.dim() == 3 else 1
        mel = mel.unsqueeze(0) if mel.dim() == 2 else mel
        mel = mel.to(device)

        src_id = torch.tensor([EMOTION_IDX[source_emotion]] * B, dtype=torch.long, device=device)
        tgt_id = torch.tensor([EMOTION_IDX[target_emotion]] * B, dtype=torch.long, device=device)
        profile = EMOTION_PROFILES[target_emotion]

        with torch.no_grad():
            out = self.forward(
                mel=mel,
                source_emotion_id=src_id,
                target_emotion_id=tgt_id,
                emotion_profile=profile,
                target_mel_len=target_mel_len or mel.shape[-1]
            )
        return out

    def get_latent(self, mel: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Extract VAE latent code from mel spectrogram."""
        mel = mel.unsqueeze(0) if mel.dim() == 2 else mel
        with torch.no_grad():
            mu, log_var = self.vae.encode(mel)
        return mu, log_var

    def count_parameters(self) -> Dict[str, int]:
        """Count trainable parameters per module."""
        counts = {}
        for name, module in [
            ("vae", self.vae),
            ("gnn", self.gnn),
            ("variance_adaptor", self.variance_adaptor),
            ("vocoder", self.vocoder),
        ]:
            counts[name] = sum(p.numel() for p in module.parameters() if p.requires_grad)
        counts["total"] = sum(counts.values())
        return counts


# =============================================================================
# SECTION 8: LOSS FUNCTIONS
# =============================================================================

class EmotionalSpeechLoss(nn.Module):
    """
    Combined loss function for the emotional speech conversion system.
    - Mel reconstruction loss (L1)
    - KL divergence for VAE
    - Pitch prediction loss
    - Energy prediction loss
    - Duration prediction loss
    - Feature matching loss (for GAN stability)
    - Emotion consistency loss
    """

    def __init__(self, train_config: TrainConfig):
        super().__init__()
        self.config = train_config
        self.l1 = nn.L1Loss()
        self.mse = nn.MSELoss()
        self.bce = nn.BCEWithLogitsLoss()
        self._step = 0

    def compute_kl_weight(self) -> float:
        """Anneal KL weight from 0 to beta_kl_max over warmup steps."""
        if self._step >= self.config.beta_kl_anneal_steps:
            return self.config.beta_kl_max
        frac = self._step / self.config.beta_kl_anneal_steps
        weight = self.config.beta_kl_max * (1 - math.cos(math.pi * frac)) / 2.0
        return max(self.config.beta_kl, weight)

    def generator_loss(
        self,
        disc_outputs: List[torch.Tensor]
    ) -> torch.Tensor:
        loss = torch.tensor(0.0)
        for d_out in disc_outputs:
            target = torch.ones_like(d_out)
            loss = loss + F.mse_loss(d_out, target)
        return loss / len(disc_outputs)

    def discriminator_loss(
        self,
        disc_real: List[torch.Tensor],
        disc_fake: List[torch.Tensor]
    ) -> torch.Tensor:
        loss = torch.tensor(0.0)
        for d_real, d_fake in zip(disc_real, disc_fake):
            real_target = torch.ones_like(d_real)
            fake_target = torch.zeros_like(d_fake)
            loss = loss + F.mse_loss(d_real, real_target) + F.mse_loss(d_fake, fake_target)
        return loss / len(disc_real)

    def forward(
        self,
        outputs: Dict[str, torch.Tensor],
        targets: Dict[str, torch.Tensor],
        beta_kl: Optional[float] = None,
    ) -> Dict[str, torch.Tensor]:
        self._step += 1
        losses = {}

        if "mel_recon" in outputs and "mel" in targets:
            losses["mel_recon"] = self.l1(outputs["mel_recon"], targets["mel"]) * self.config.lambda_mel

        if "mel_converted" in outputs and "mel" in targets:
            tgt_mel = targets["mel"]
            pred_mel = outputs["mel_converted"]
            if pred_mel.shape != tgt_mel.shape:
                pred_mel = F.interpolate(pred_mel, size=tgt_mel.shape[-1], mode='linear', align_corners=False)
            losses["mel_converted"] = self.l1(pred_mel, tgt_mel) * self.config.lambda_mel * 0.5

        if "mu" in outputs and "log_var" in outputs:
            kl_w = beta_kl if beta_kl is not None else self.compute_kl_weight()
            kl = -0.5 * torch.mean(1 + outputs["log_var"] - outputs["mu"].pow(2) - outputs["log_var"].exp())
            losses["kl"] = kl * kl_w

        if "pitch_pred" in outputs and "f0" in targets:
            f0_tgt = targets["f0"]
            f0_pred = outputs["pitch_pred"]
            if f0_pred.shape != f0_tgt.shape:
                f0_pred = F.interpolate(f0_pred.unsqueeze(1), size=f0_tgt.shape[-1], mode='linear', align_corners=False).squeeze(1)
            voiced = targets.get("voiced", torch.ones_like(f0_tgt))
            voiced_mask = voiced > 0.5
            if voiced_mask.sum() > 0:
                losses["pitch"] = self.mse(f0_pred[voiced_mask], f0_tgt[voiced_mask]) * self.config.lambda_pitch

        if "energy_pred" in outputs and "energy" in targets:
            e_tgt = targets["energy"]
            e_pred = outputs["energy_pred"]
            if e_pred.shape != e_tgt.shape:
                e_pred = F.interpolate(e_pred.unsqueeze(1), size=e_tgt.shape[-1], mode='linear', align_corners=False).squeeze(1)
            losses["energy"] = self.mse(e_pred, e_tgt) * self.config.lambda_energy

        if "duration_pred" in outputs and "duration" in targets:
            dur_tgt = targets["duration"].float()
            dur_pred = outputs["duration_pred"]
            min_len = min(dur_pred.shape[-1], dur_tgt.shape[-1])
            losses["duration"] = self.mse(dur_pred[..., :min_len], dur_tgt[..., :min_len]) * self.config.lambda_duration

        losses["total"] = sum(losses.values())
        return losses


# =============================================================================
# SECTION 9: DATASET
# =============================================================================

class SpeechDataset(Dataset):
    """
    Dataset for speech emotion conversion training.
    Supports loading from directory structure:
      data/
        neutral/  *.wav
        happy/    *.wav
        sad/      *.wav
        ...

    Also supports generating synthetic demo data for testing.
    """

    def __init__(
        self,
        data_dir: Optional[str],
        audio_processor: AudioProcessor,
        max_wav_len: int = 16384,
        max_mel_len: int = 512,
        use_synthetic: bool = False,
        synthetic_samples: int = 64,
    ):
        self.processor = audio_processor
        self.max_wav_len = max_wav_len
        self.max_mel_len = max_mel_len
        self.samples = []

        if use_synthetic or data_dir is None or not os.path.exists(data_dir or ""):
            logger.info("Using synthetic dataset for training demo")
            self._create_synthetic_dataset(synthetic_samples)
        else:
            self._load_from_directory(data_dir)

    def _load_from_directory(self, data_dir: str):
        for emotion_name in EMOTION_PROFILES.keys():
            emotion_dir = os.path.join(data_dir, emotion_name)
            if not os.path.exists(emotion_dir):
                continue
            for fname in os.listdir(emotion_dir):
                if fname.lower().endswith(('.wav', '.mp3', '.flac', '.ogg')):
                    self.samples.append({
                        "path": os.path.join(emotion_dir, fname),
                        "emotion": emotion_name,
                        "emotion_id": EMOTION_IDX[emotion_name],
                    })
        logger.info(f"Loaded {len(self.samples)} audio samples from {data_dir}")

    def _create_synthetic_dataset(self, n_samples: int):
        emotions = list(EMOTION_PROFILES.keys())
        for i in range(n_samples):
            emotion = emotions[i % len(emotions)]
            self.samples.append({
                "path": None,
                "emotion": emotion,
                "emotion_id": EMOTION_IDX[emotion],
                "synthetic": True,
                "seed": i,
            })

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        sample = self.samples[idx]
        emotion_id = sample["emotion_id"]
        emotion_name = sample["emotion"]
        profile = EMOTION_PROFILES[emotion_name]

        if sample.get("synthetic"):
            torch.manual_seed(sample["seed"])
            waveform = self.processor._generate_demo_signal(duration_sec=2.0)
            waveform = self._apply_emotion_augmentation(waveform, profile)
        else:
            waveform, _ = self.processor.load_audio(sample["path"])

        waveform = self._pad_or_trim_waveform(waveform, self.max_wav_len)
        features = self.processor.extract_all_features(waveform)

        mel = features["mel"]
        f0 = features["f0"]
        voiced = features["voiced"]
        energy = features["energy"]
        duration = features["duration"]

        mel = self._pad_or_trim_mel(mel, self.max_mel_len)
        f0 = self._pad_or_trim_1d(f0, self.max_mel_len)
        voiced = self._pad_or_trim_1d(voiced, self.max_mel_len)
        energy = self._pad_or_trim_1d(energy, self.max_mel_len)

        pitch_mean = f0[voiced > 0.5].mean() if (voiced > 0.5).any() else torch.tensor(0.0)
        pitch_std = f0[voiced > 0.5].std() if (voiced > 0.5).sum() > 1 else torch.tensor(0.0)
        energy_mean = energy.mean()
        energy_std = energy.std()
        speed = torch.tensor(profile.speed_factor, dtype=torch.float32)
        dur_factor = torch.tensor(profile.duration_factor, dtype=torch.float32)
        prosody = torch.stack([pitch_mean, pitch_std, energy_mean, energy_std, speed, dur_factor])

        return {
            "mel": mel,
            "f0": f0,
            "voiced": voiced,
            "energy": energy,
            "duration": duration[:min(len(duration), 32)],
            "emotion_id": torch.tensor(emotion_id, dtype=torch.long),
            "prosody": prosody,
            "emotion_name": emotion_name,
        }

    def _apply_emotion_augmentation(self, waveform: torch.Tensor, profile: EmotionProfile) -> torch.Tensor:
        """Apply basic emotion-specific augmentation to synthetic waveform."""
        wav_np = waveform.squeeze().numpy()
        if profile.speed_factor != 1.0:
            indices = np.linspace(0, len(wav_np) - 1, int(len(wav_np) / profile.speed_factor))
            indices = np.clip(indices.astype(int), 0, len(wav_np) - 1)
            wav_np = wav_np[indices]
        amp = profile.energy_scale * np.random.uniform(0.8, 1.2)
        wav_np = wav_np * amp
        wav_np = wav_np / (np.max(np.abs(wav_np)) + 1e-8) * 0.9
        return torch.from_numpy(wav_np.astype(np.float32)).unsqueeze(0)

    def _pad_or_trim_waveform(self, waveform: torch.Tensor, target: int) -> torch.Tensor:
        waveform = waveform.squeeze(0)
        if waveform.shape[0] > target:
            return waveform[:target].unsqueeze(0)
        elif waveform.shape[0] < target:
            pad = target - waveform.shape[0]
            return F.pad(waveform, (0, pad)).unsqueeze(0)
        return waveform.unsqueeze(0)

    def _pad_or_trim_mel(self, mel: torch.Tensor, target_len: int) -> torch.Tensor:
        T = mel.shape[1]
        if T > target_len:
            return mel[:, :target_len]
        elif T < target_len:
            return F.pad(mel, (0, target_len - T))
        return mel

    def _pad_or_trim_1d(self, x: torch.Tensor, target: int) -> torch.Tensor:
        if x.shape[0] > target:
            return x[:target]
        elif x.shape[0] < target:
            return F.pad(x, (0, target - x.shape[0]))
        return x


def collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    """Custom collate function for variable-length duration sequences."""
    max_dur_len = max(s["duration"].shape[0] for s in batch)
    padded_dur = []
    for s in batch:
        d = s["duration"]
        if d.shape[0] < max_dur_len:
            d = F.pad(d, (0, max_dur_len - d.shape[0]))
        padded_dur.append(d)

    return {
        "mel": torch.stack([s["mel"] for s in batch]),
        "f0": torch.stack([s["f0"] for s in batch]),
        "voiced": torch.stack([s["voiced"] for s in batch]),
        "energy": torch.stack([s["energy"] for s in batch]),
        "duration": torch.stack(padded_dur),
        "emotion_id": torch.stack([s["emotion_id"] for s in batch]),
        "prosody": torch.stack([s["prosody"] for s in batch]),
    }


# =============================================================================
# SECTION 10: TRAINER
# =============================================================================

class EmotionalSpeechTrainer:
    """
    Full training loop for the Emotional Speech Conversion System.
    Includes:
    - VAE training with KL annealing
    - GNN training with emotion graph
    - FastSpeech2 variance adaptor training
    - HiFi-GAN vocoder training (generator + discriminator)
    """

    def __init__(
        self,
        model: EmotionalSpeechConverter,
        train_config: TrainConfig,
        audio_config: AudioConfig,
        device: torch.device,
        data_dir: Optional[str] = None,
    ):
        self.model = model.to(device)
        self.train_config = train_config
        self.audio_config = audio_config
        self.device = device

        self.msd = HiFiGANMultiScaleDiscriminator().to(device)
        self.mpd = HiFiGANMultiPeriodDiscriminator().to(device)

        self.gen_optimizer = optim.AdamW(
            list(model.vae.parameters()) +
            list(model.gnn.parameters()) +
            list(model.variance_adaptor.parameters()) +
            list(model.vocoder.parameters()),
            lr=train_config.learning_rate,
            betas=(0.9, 0.999),
            weight_decay=1e-4
        )
        self.disc_optimizer = optim.AdamW(
            list(self.msd.parameters()) + list(self.mpd.parameters()),
            lr=train_config.learning_rate,
            betas=(0.9, 0.999),
        )

        self.gen_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.gen_optimizer, T_max=train_config.num_epochs, eta_min=1e-6
        )
        self.disc_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.disc_optimizer, T_max=train_config.num_epochs, eta_min=1e-6
        )

        self.loss_fn = EmotionalSpeechLoss(train_config)

        audio_processor = AudioProcessor(audio_config)
        dataset = SpeechDataset(
            data_dir=data_dir,
            audio_processor=audio_processor,
            use_synthetic=(data_dir is None),
        )
        self.dataloader = DataLoader(
            dataset,
            batch_size=train_config.batch_size,
            shuffle=True,
            num_workers=0,
            collate_fn=collate_fn,
            drop_last=True,
        )

        os.makedirs(train_config.checkpoint_dir, exist_ok=True)
        self.global_step = 0
        self.best_loss = float('inf')

        self.history = {
            "gen_loss": [], "disc_loss": [], "mel_loss": [], "kl_loss": [],
            "pitch_loss": [], "energy_loss": [], "epoch": [],
        }

    def _to_device(self, batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        return {k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

    def _warmup_lr(self) -> float:
        """Noam learning rate warmup."""
        step = max(1, self.global_step)
        d_model = self.model.model_config.fastspeech_hidden
        warmup = self.train_config.warmup_steps
        lr = (d_model ** -0.5) * min(step ** -0.5, step * warmup ** -1.5)
        lr = lr * self.train_config.learning_rate * 100
        return lr

    def train_epoch(self, epoch: int) -> Dict[str, float]:
        """Train one epoch."""
        self.model.train()
        self.msd.train()
        self.mpd.train()

        epoch_losses = {
            "gen_total": 0.0, "disc_total": 0.0,
            "mel_recon": 0.0, "kl": 0.0, "pitch": 0.0, "energy": 0.0
        }
        n_batches = 0

        for batch in self.dataloader:
            batch = self._to_device(batch)
            mel = batch["mel"]
            emotion_id = batch["emotion_id"]
            prosody = batch["prosody"]

            B = mel.shape[0]
            target_ids = torch.randint(0, len(EMOTION_PROFILES), (B,), device=self.device)

            lr = self._warmup_lr()
            for pg in self.gen_optimizer.param_groups:
                pg['lr'] = lr

            outputs = self.model(
                mel=mel,
                source_emotion_id=emotion_id,
                target_emotion_id=target_ids,
                source_prosody=prosody,
                target_mel_len=mel.shape[-1],
            )

            waveform_fake = outputs["waveform"]
            mel_conv = outputs["mel_converted"]

            targets = {
                "mel": mel,
                "f0": batch["f0"],
                "voiced": batch["voiced"],
                "energy": batch["energy"],
                "duration": batch["duration"],
            }

            rec_losses = self.loss_fn(outputs, targets)
            mel_loss = rec_losses.get("mel_recon", torch.tensor(0.0, device=self.device))
            kl_loss = rec_losses.get("kl", torch.tensor(0.0, device=self.device))
            pitch_loss = rec_losses.get("pitch", torch.tensor(0.0, device=self.device))
            energy_loss = rec_losses.get("energy", torch.tensor(0.0, device=self.device))

            waveform_len = waveform_fake.shape[-1]
            dummy_real = torch.randn(B, 1, waveform_len, device=self.device) * 0.1

            msd_fake = self.msd(waveform_fake.detach())
            mpd_fake = self.mpd(waveform_fake.detach())
            msd_real = self.msd(dummy_real)
            mpd_real = self.mpd(dummy_real)

            disc_loss = self.loss_fn.discriminator_loss(
                msd_real + mpd_real, msd_fake + mpd_fake
            )

            self.disc_optimizer.zero_grad()
            disc_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(self.msd.parameters()) + list(self.mpd.parameters()),
                self.train_config.grad_clip
            )
            self.disc_optimizer.step()

            msd_fake_gen = self.msd(waveform_fake)
            mpd_fake_gen = self.mpd(waveform_fake)
            gen_adv_loss = self.loss_fn.generator_loss(msd_fake_gen + mpd_fake_gen)

            gen_loss = mel_loss + kl_loss + pitch_loss + energy_loss + gen_adv_loss * 0.1

            self.gen_optimizer.zero_grad()
            gen_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(self.model.parameters()),
                self.train_config.grad_clip
            )
            self.gen_optimizer.step()

            epoch_losses["gen_total"] += gen_loss.item()
            epoch_losses["disc_total"] += disc_loss.item()
            epoch_losses["mel_recon"] += mel_loss.item()
            epoch_losses["kl"] += kl_loss.item()
            epoch_losses["pitch"] += pitch_loss.item() if isinstance(pitch_loss, torch.Tensor) else pitch_loss
            epoch_losses["energy"] += energy_loss.item() if isinstance(energy_loss, torch.Tensor) else energy_loss

            n_batches += 1
            self.global_step += 1

            if self.global_step % self.train_config.log_interval == 0:
                logger.debug(
                    f"  Step {self.global_step}: "
                    f"G={gen_loss.item():.4f} D={disc_loss.item():.4f} "
                    f"Mel={mel_loss.item():.4f} KL={kl_loss.item():.5f}"
                )

        for k in epoch_losses:
            epoch_losses[k] /= max(n_batches, 1)

        return epoch_losses

    def save_checkpoint(self, epoch: int, loss: float):
        """Save model checkpoint."""
        ckpt_path = os.path.join(
            self.train_config.checkpoint_dir,
            f"checkpoint_epoch_{epoch:04d}.pt"
        )
        torch.save({
            "epoch": epoch,
            "global_step": self.global_step,
            "model_state_dict": self.model.state_dict(),
            "gen_optimizer_state_dict": self.gen_optimizer.state_dict(),
            "loss": loss,
        }, ckpt_path)
        logger.info(f"Saved checkpoint: {ckpt_path}")

        best_path = os.path.join(self.train_config.checkpoint_dir, "best_model.pt")
        if loss < self.best_loss:
            self.best_loss = loss
            torch.save(self.model.state_dict(), best_path)
            logger.info(f"New best model saved (loss={loss:.4f})")

    def load_checkpoint(self, path: str):
        """Load model from checkpoint."""
        ckpt = torch.load(path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.gen_optimizer.load_state_dict(ckpt["gen_optimizer_state_dict"])
        self.global_step = ckpt.get("global_step", 0)
        logger.info(f"Loaded checkpoint from {path} (epoch {ckpt['epoch']})")
        return ckpt["epoch"]

    def train(self, num_epochs: Optional[int] = None) -> Dict[str, List[float]]:
        """Full training loop."""
        n_epochs = num_epochs or self.train_config.num_epochs
        logger.info(f"Starting training for {n_epochs} epochs on {self.device}")
        param_counts = self.model.count_parameters()
        logger.info(f"Model parameters: {param_counts}")

        for epoch in range(1, n_epochs + 1):
            t0 = time.time()
            epoch_losses = self.train_epoch(epoch)
            elapsed = time.time() - t0

            self.gen_scheduler.step()
            self.disc_scheduler.step()

            self.history["gen_loss"].append(epoch_losses["gen_total"])
            self.history["disc_loss"].append(epoch_losses["disc_total"])
            self.history["mel_loss"].append(epoch_losses["mel_recon"])
            self.history["kl_loss"].append(epoch_losses["kl"])
            self.history["pitch_loss"].append(epoch_losses["pitch"])
            self.history["energy_loss"].append(epoch_losses["energy"])
            self.history["epoch"].append(epoch)

            logger.info(
                f"Epoch {epoch:3d}/{n_epochs} | "
                f"G={epoch_losses['gen_total']:.4f} | "
                f"D={epoch_losses['disc_total']:.4f} | "
                f"Mel={epoch_losses['mel_recon']:.4f} | "
                f"KL={epoch_losses['kl']:.5f} | "
                f"t={elapsed:.1f}s"
            )

            if epoch % self.train_config.save_interval == 0:
                self.save_checkpoint(epoch, epoch_losses["gen_total"])

        if self.history["gen_loss"]:
            self.save_checkpoint(n_epochs, self.history["gen_loss"][-1])
        return self.history


# =============================================================================
# SECTION 11: EVALUATION METRICS
# =============================================================================

class SpeechEvaluator:
    """
    Comprehensive evaluation suite for emotional speech conversion.

    Metrics:
    - MCD (Mel Cepstral Distortion): spectral distance, lower is better
    - F0-RMSE: pitch accuracy, lower is better
    - F0 Correlation: pitch pattern correlation, higher is better
    - Energy-RMSE: loudness accuracy, lower is better
    - PESQ: perceptual evaluation of speech quality [1-4.5], higher is better
    - STOI: short-time objective intelligibility [0-1], higher is better
    - Cosine Similarity: overall feature similarity [0-1], higher is better
    - Emotion Feature Changes: quantified changes vs neutral reference
    """

    def __init__(self, audio_config: AudioConfig):
        self.audio_config = audio_config
        self.processor = AudioProcessor(audio_config)

    def mel_cepstral_distortion(
        self,
        mel_ref: torch.Tensor,
        mel_hyp: torch.Tensor,
        n_mfcc: int = 13
    ) -> float:
        """
        Compute Mel Cepstral Distortion (MCD) between two mel spectrograms.
        MCD = (10/ln(10)) * sqrt(2 * sum((mc1-mc2)^2))
        """
        m1 = mel_ref.detach().cpu().numpy()
        m2 = mel_hyp.detach().cpu().numpy()

        if m1.shape != m2.shape:
            min_T = min(m1.shape[-1], m2.shape[-1])
            m1 = m1[..., :min_T]
            m2 = m2[..., :min_T]

        if m1.ndim == 3:
            m1 = m1[0]
            m2 = m2[0]

        m1 = m1[:n_mfcc, :]
        m2 = m2[:n_mfcc, :]

        diff = m1 - m2
        mcd = (10.0 / np.log(10.0)) * np.sqrt(2.0 * np.mean(np.sum(diff ** 2, axis=0)))
        return float(mcd)

    def f0_rmse(
        self,
        f0_ref: torch.Tensor,
        f0_hyp: torch.Tensor,
        voiced_ref: Optional[torch.Tensor] = None
    ) -> float:
        """Compute F0 RMSE in semitones on voiced frames."""
        r = f0_ref.detach().cpu().numpy().flatten()
        h = f0_hyp.detach().cpu().numpy().flatten()
        min_len = min(len(r), len(h))
        r = r[:min_len]
        h = h[:min_len]

        if voiced_ref is not None:
            v = voiced_ref.detach().cpu().numpy().flatten()[:min_len]
            mask = v > 0.5
        else:
            mask = (r > 0) & (h > 0)

        if mask.sum() == 0:
            return float('inf')

        r_voiced = r[mask]
        h_voiced = h[mask]
        rmse = np.sqrt(np.mean((r_voiced - h_voiced) ** 2))
        return float(rmse)

    def f0_correlation(
        self,
        f0_ref: torch.Tensor,
        f0_hyp: torch.Tensor,
        voiced_ref: Optional[torch.Tensor] = None
    ) -> float:
        """Compute Pearson correlation coefficient of F0 trajectories."""
        r = f0_ref.detach().cpu().numpy().flatten()
        h = f0_hyp.detach().cpu().numpy().flatten()
        min_len = min(len(r), len(h))
        r = r[:min_len]
        h = h[:min_len]

        if voiced_ref is not None:
            v = voiced_ref.detach().cpu().numpy().flatten()[:min_len]
            mask = v > 0.5
        else:
            mask = (r > 0) & (h > 0)

        if mask.sum() < 2:
            return 0.0

        corr = np.corrcoef(r[mask], h[mask])[0, 1]
        return float(corr) if not np.isnan(corr) else 0.0

    def energy_rmse(self, energy_ref: torch.Tensor, energy_hyp: torch.Tensor) -> float:
        """Compute Energy RMSE."""
        r = energy_ref.detach().cpu().numpy().flatten()
        h = energy_hyp.detach().cpu().numpy().flatten()
        min_len = min(len(r), len(h))
        return float(np.sqrt(np.mean((r[:min_len] - h[:min_len]) ** 2)))

    def cosine_similarity(self, mel_ref: torch.Tensor, mel_hyp: torch.Tensor) -> float:
        """Compute cosine similarity between average mel frame vectors."""
        r = mel_ref.detach().cpu()
        h = mel_hyp.detach().cpu()
        if r.dim() == 3:
            r = r[0]
        if h.dim() == 3:
            h = h[0]
        min_T = min(r.shape[1], h.shape[1])
        r_mean = r[:, :min_T].mean(dim=1)
        h_mean = h[:, :min_T].mean(dim=1)
        cos = F.cosine_similarity(r_mean.unsqueeze(0), h_mean.unsqueeze(0)).item()
        return float(cos)

    def compute_pesq(self, wav_ref: np.ndarray, wav_hyp: np.ndarray, sr: int = 16000) -> float:
        """Compute PESQ score (requires pesq package)."""
        if not PESQ_AVAILABLE:
            return float('nan')
        try:
            min_len = min(len(wav_ref), len(wav_hyp))
            score = pesq(sr, wav_ref[:min_len], wav_hyp[:min_len], 'wb')
            return float(score)
        except Exception:
            return float('nan')

    def compute_stoi(self, wav_ref: np.ndarray, wav_hyp: np.ndarray, sr: int = 16000) -> float:
        """Compute STOI (Short-Time Objective Intelligibility) score."""
        if not STOI_AVAILABLE:
            return float('nan')
        try:
            min_len = min(len(wav_ref), len(wav_hyp))
            score = stoi(wav_ref[:min_len], wav_hyp[:min_len], sr, extended=False)
            return float(score)
        except Exception:
            return float('nan')

    def compute_emotion_feature_changes(
        self,
        mel_ref: torch.Tensor,
        mel_hyp: torch.Tensor,
        f0_ref: Optional[torch.Tensor] = None,
        f0_hyp: Optional[torch.Tensor] = None,
        energy_ref: Optional[torch.Tensor] = None,
        energy_hyp: Optional[torch.Tensor] = None,
        emotion_profile: Optional[EmotionProfile] = None,
    ) -> Dict[str, float]:
        """
        Compute quantified feature changes between reference and converted speech.
        Returns percentage changes and expected changes from emotion profile.
        """
        changes = {}

        if f0_ref is not None and f0_hyp is not None:
            r = f0_ref.detach().cpu().numpy().flatten()
            h = f0_hyp.detach().cpu().numpy().flatten()
            min_len = min(len(r), len(h))
            voiced_r = r[:min_len] != 0
            if voiced_r.sum() > 0:
                mean_ref = np.mean(r[:min_len][voiced_r])
                mean_hyp = np.mean(h[:min_len][voiced_r])
                changes["pitch_mean_change_pct"] = (mean_hyp - mean_ref) / (abs(mean_ref) + 1e-8) * 100
                std_ref = np.std(r[:min_len][voiced_r])
                std_hyp = np.std(h[:min_len][voiced_r])
                changes["pitch_variance_change_pct"] = (std_hyp - std_ref) / (abs(std_ref) + 1e-8) * 100
            else:
                changes["pitch_mean_change_pct"] = 0.0
                changes["pitch_variance_change_pct"] = 0.0

        if energy_ref is not None and energy_hyp is not None:
            r = energy_ref.detach().cpu().numpy().flatten()
            h = energy_hyp.detach().cpu().numpy().flatten()
            min_len = min(len(r), len(h))
            mean_ref = np.mean(r[:min_len])
            mean_hyp = np.mean(h[:min_len])
            changes["energy_mean_change_pct"] = (mean_hyp - mean_ref) / (abs(mean_ref) + 1e-8) * 100
            std_ref = np.std(r[:min_len])
            std_hyp = np.std(h[:min_len])
            changes["energy_variance_change_pct"] = (std_hyp - std_ref) / (abs(std_ref) + 1e-8) * 100

        if mel_ref is not None and mel_hyp is not None:
            ref_dur = mel_ref.shape[-1]
            hyp_dur = mel_hyp.shape[-1]
            changes["duration_change_pct"] = (hyp_dur - ref_dur) / (ref_dur + 1e-8) * 100

        if emotion_profile is not None:
            changes["expected_pitch_shift_semitones"] = emotion_profile.pitch_shift
            changes["expected_speed_factor"] = emotion_profile.speed_factor
            changes["expected_energy_scale"] = emotion_profile.energy_scale
            changes["expected_pitch_variance"] = emotion_profile.pitch_variance
            changes["expected_energy_variance"] = emotion_profile.energy_variance
            changes["expected_duration_factor"] = emotion_profile.duration_factor
            changes["voice_quality"] = emotion_profile.voice_quality

        return changes

    def evaluate_conversion(
        self,
        mel_original: torch.Tensor,
        mel_converted: torch.Tensor,
        waveform_original: Optional[torch.Tensor] = None,
        waveform_converted: Optional[torch.Tensor] = None,
        f0_original: Optional[torch.Tensor] = None,
        f0_converted: Optional[torch.Tensor] = None,
        energy_original: Optional[torch.Tensor] = None,
        energy_converted: Optional[torch.Tensor] = None,
        voiced: Optional[torch.Tensor] = None,
        emotion_name: str = "unknown",
        emotion_profile: Optional[EmotionProfile] = None,
        sample_rate: int = 22050,
    ) -> Dict[str, Any]:
        """Run full evaluation suite and return metrics dict."""
        metrics = {"emotion": emotion_name}

        metrics["mcd"] = self.mel_cepstral_distortion(mel_original, mel_converted)
        metrics["cosine_similarity"] = self.cosine_similarity(mel_original, mel_converted)

        if f0_original is not None and f0_converted is not None:
            metrics["f0_rmse"] = self.f0_rmse(f0_original, f0_converted, voiced)
            metrics["f0_correlation"] = self.f0_correlation(f0_original, f0_converted, voiced)
        else:
            metrics["f0_rmse"] = float('nan')
            metrics["f0_correlation"] = float('nan')

        if energy_original is not None and energy_converted is not None:
            metrics["energy_rmse"] = self.energy_rmse(energy_original, energy_converted)
        else:
            metrics["energy_rmse"] = float('nan')

        if waveform_original is not None and waveform_converted is not None:
            wav_ref = waveform_original.squeeze().cpu().numpy()
            wav_hyp = waveform_converted.squeeze().cpu().numpy()

            resample_sr = 16000
            if sample_rate != resample_sr:
                resampler = T.Resample(sample_rate, resample_sr)
                wav_ref_t = resampler(torch.from_numpy(wav_ref).unsqueeze(0)).squeeze().numpy()
                wav_hyp_t = resampler(torch.from_numpy(wav_hyp).unsqueeze(0)).squeeze().numpy()
            else:
                wav_ref_t = wav_ref
                wav_hyp_t = wav_hyp

            metrics["pesq"] = self.compute_pesq(wav_ref_t, wav_hyp_t, resample_sr)
            metrics["stoi"] = self.compute_stoi(wav_ref_t, wav_hyp_t, resample_sr)
        else:
            metrics["pesq"] = float('nan')
            metrics["stoi"] = float('nan')

        feature_changes = self.compute_emotion_feature_changes(
            mel_original, mel_converted,
            f0_original, f0_converted,
            energy_original, energy_converted,
            emotion_profile
        )
        metrics.update(feature_changes)

        return metrics

    def build_evaluation_table(
        self,
        all_metrics: List[Dict[str, Any]],
        reference_emotion: str = "neutral"
    ) -> str:
        """
        Build a comprehensive evaluation table as a formatted string.

        Table columns:
          Emotion | MCD↓ | F0-RMSE↓ | F0-Corr↑ | E-RMSE↓ | PESQ↑ | STOI↑ | CosSim↑ |
          Pitch%Δ | Energy%Δ | Dur%Δ | Expected Shift | Voice Quality
        """
        headers = [
            "Emotion", "MCD↓", "F0-RMSE↓", "F0-Corr↑", "E-RMSE↓",
            "PESQ↑", "STOI↑", "CosSim↑",
            "Pitch%Δ", "Energy%Δ", "Dur%Δ",
            "Exp.Pitch(st)", "Exp.Energy×", "Voice Quality"
        ]

        rows = []
        for m in all_metrics:
            emotion = m.get("emotion", "unknown")
            profile = EMOTION_PROFILES.get(emotion)

            def fmt(v, decimals=3):
                if v is None or (isinstance(v, float) and math.isnan(v)):
                    return "N/A"
                return f"{v:.{decimals}f}"

            row = [
                emotion.upper(),
                fmt(m.get("mcd"), 2),
                fmt(m.get("f0_rmse"), 3),
                fmt(m.get("f0_correlation"), 3),
                fmt(m.get("energy_rmse"), 3),
                fmt(m.get("pesq"), 2),
                fmt(m.get("stoi"), 3),
                fmt(m.get("cosine_similarity"), 3),
                f"{m.get('pitch_mean_change_pct', 0.0):+.1f}%",
                f"{m.get('energy_mean_change_pct', 0.0):+.1f}%",
                f"{m.get('duration_change_pct', 0.0):+.1f}%",
                f"{m.get('expected_pitch_shift_semitones', 0.0):+.1f}st" if profile else "N/A",
                f"{m.get('expected_energy_scale', 1.0):.2f}×" if profile else "N/A",
                m.get("voice_quality", profile.voice_quality if profile else "N/A"),
            ]
            rows.append(row)

        if TABULATE_AVAILABLE:
            table = tabulate(rows, headers=headers, tablefmt="grid", floatfmt=".3f")
        else:
            col_widths = [max(len(h), max((len(str(r[i])) for r in rows), default=0))
                         for i, h in enumerate(headers)]
            sep = "+" + "+".join("-" * (w + 2) for w in col_widths) + "+"
            header_row = "|" + "|".join(f" {h:<{w}} " for h, w in zip(headers, col_widths)) + "|"
            lines = [sep, header_row, sep]
            for row in rows:
                data_row = "|" + "|".join(f" {str(c):<{w}} " for c, w in zip(row, col_widths)) + "|"
                lines.append(data_row)
            lines.append(sep)
            table = "\n".join(lines)

        return table

    def print_feature_analysis(self, all_metrics: List[Dict[str, Any]]):
        """Print detailed feature change analysis per emotion."""
        print("\n" + "=" * 80)
        print("EMOTION-WISE FEATURE CHANGE ANALYSIS")
        print("=" * 80)

        for m in all_metrics:
            emotion = m.get("emotion", "unknown")
            profile = EMOTION_PROFILES.get(emotion)
            if profile is None:
                continue

            print(f"\n[ {emotion.upper()} ]")
            print(f"  Description  : {profile.description}")
            print(f"  Voice Quality: {profile.voice_quality}")
            print(f"  Intensity    : {profile.intensity_range[0]:.1f} - {profile.intensity_range[1]:.1f}")
            print(f"\n  ACOUSTIC CHANGES vs NEUTRAL:")
            print(f"    Pitch Shift   : {profile.pitch_shift:+.1f} semitones (expected)")
            print(f"    Speed Factor  : {profile.speed_factor:.2f}× (expected)")
            print(f"    Energy Scale  : {profile.energy_scale:.2f}× (expected)")
            print(f"    Pitch Variance: {profile.pitch_variance:.3f} (expected)")
            print(f"    Energy Variance: {profile.energy_variance:.3f} (expected)")
            print(f"    Duration Factor: {profile.duration_factor:.2f}× (expected)")

            print(f"\n  MEASURED CHANGES:")
            pitch_chg = m.get("pitch_mean_change_pct", None)
            energy_chg = m.get("energy_mean_change_pct", None)
            dur_chg = m.get("duration_change_pct", None)
            if pitch_chg is not None:
                print(f"    Pitch Mean   : {pitch_chg:+.1f}%")
            if energy_chg is not None:
                print(f"    Energy Mean  : {energy_chg:+.1f}%")
            if dur_chg is not None:
                print(f"    Duration     : {dur_chg:+.1f}%")

            print(f"\n  QUALITY METRICS:")
            print(f"    MCD          : {m.get('mcd', float('nan')):.3f} dB (lower=better)")
            print(f"    F0 Corr.     : {m.get('f0_correlation', float('nan')):.3f} (higher=better)")
            print(f"    Cosine Sim.  : {m.get('cosine_similarity', float('nan')):.3f} (higher=better)")
            if not math.isnan(m.get("pesq", float('nan'))):
                print(f"    PESQ         : {m.get('pesq'):.2f} (higher=better)")
            if not math.isnan(m.get("stoi", float('nan'))):
                print(f"    STOI         : {m.get('stoi'):.3f} (higher=better)")

        print("\n" + "=" * 80)


# =============================================================================
# SECTION 12: POST-PROCESSING FOR NATURAL AUDIO
# =============================================================================

class NaturalAudioPostProcessor:
    """
    Post-processing pipeline to make generated audio sound more natural (human-like).

    Techniques:
    1. Spectral smoothing (reduces harshness)
    2. Micro-pitch jitter (human voice naturalness)
    3. Amplitude modulation (breath-like variations)
    4. Phase randomization in high frequencies (de-robotize)
    5. Formant enhancement
    6. Low-level noise addition (presence/air)
    7. Dynamic range compression (consistent loudness)
    """

    def __init__(self, sample_rate: int = 22050):
        self.sr = sample_rate

    def apply(self, waveform: torch.Tensor, emotion_profile: Optional[EmotionProfile] = None) -> torch.Tensor:
        """Apply full post-processing pipeline."""
        wav = waveform.squeeze().numpy()
        wav = self._normalize(wav)
        wav = self._add_jitter(wav, emotion_profile)
        wav = self._apply_spectral_shaping(wav, emotion_profile)
        wav = self._add_breath_modulation(wav, emotion_profile)
        wav = self._apply_natural_noise(wav, emotion_profile)
        wav = self._dynamic_range_compression(wav)
        wav = self._normalize(wav)
        return torch.from_numpy(wav.astype(np.float32)).unsqueeze(0)

    def _normalize(self, wav: np.ndarray, target_rms: float = 0.1) -> np.ndarray:
        rms = np.sqrt(np.mean(wav ** 2)) + 1e-8
        return wav * (target_rms / rms)

    def _add_jitter(self, wav: np.ndarray, profile: Optional[EmotionProfile] = None) -> np.ndarray:
        """
        Add micro-pitch jitter and shimmer to humanize the voice.
        The amount scales with emotion intensity and voice quality.
        """
        jitter_amount = 0.002
        shimmer_amount = 0.03

        if profile is not None:
            if profile.voice_quality in ("tremulous",):
                jitter_amount *= 3.0
                shimmer_amount *= 2.5
            elif profile.voice_quality in ("creaky",):
                jitter_amount *= 0.5
                shimmer_amount *= 0.8
            elif profile.voice_quality in ("tense",):
                jitter_amount *= 1.5

        np.random.seed(42)
        jitter_signal = 1.0 + np.random.normal(0, jitter_amount, len(wav))
        shimmer_signal = 1.0 + np.random.normal(0, shimmer_amount, len(wav))

        jitter_signal = np.convolve(jitter_signal, np.ones(50) / 50, mode='same')
        shimmer_signal = np.convolve(shimmer_signal, np.ones(100) / 100, mode='same')

        return wav * jitter_signal * shimmer_signal

    def _apply_spectral_shaping(self, wav: np.ndarray, profile: Optional[EmotionProfile] = None) -> np.ndarray:
        """Shape frequency content based on emotion's voice quality."""
        voice_quality = profile.voice_quality if profile else "modal"

        if voice_quality == "bright":
            b, a = signal.butter(2, 3000 / (self.sr / 2), btype='high')
            high_freq = signal.lfilter(b, a, wav)
            wav = wav + high_freq * 0.15
        elif voice_quality == "breathy":
            b, a = signal.butter(2, 500 / (self.sr / 2), btype='low')
            low_freq = signal.lfilter(b, a, wav)
            noise = np.random.randn(len(wav)) * 0.01
            b_noise, a_noise = signal.butter(2, [2000 / (self.sr / 2), 6000 / (self.sr / 2)], btype='band')
            colored_noise = signal.lfilter(b_noise, a_noise, noise)
            wav = wav + colored_noise
        elif voice_quality == "tense":
            b, a = signal.butter(2, [1500 / (self.sr / 2), 4000 / (self.sr / 2)], btype='band')
            mid = signal.lfilter(b, a, wav)
            wav = wav + mid * 0.2
        elif voice_quality == "creaky":
            period = int(self.sr / 100)
            creak = np.zeros(len(wav))
            for i in range(0, len(wav) - period, period):
                creak[i] = np.random.uniform(0.02, 0.08)
            b, a = signal.butter(2, 300 / (self.sr / 2), btype='low')
            creak_filtered = signal.lfilter(b, a, creak)
            wav = wav + creak_filtered
        elif voice_quality == "tremulous":
            freq = np.random.uniform(4.5, 6.5)
            t = np.linspace(0, len(wav) / self.sr, len(wav))
            tremolo = 1.0 + 0.04 * np.sin(2 * np.pi * freq * t)
            wav = wav * tremolo

        return wav

    def _add_breath_modulation(self, wav: np.ndarray, profile: Optional[EmotionProfile] = None) -> np.ndarray:
        """Simulate natural amplitude envelope variations (breathing rhythm)."""
        breath_rate = 0.3 if profile and profile.speed_factor < 0.9 else 0.45
        t = np.linspace(0, len(wav) / self.sr, len(wav))
        breath_env = 1.0 + 0.02 * np.sin(2 * np.pi * breath_rate * t)
        phrase_env = 1.0 + 0.04 * np.sin(2 * np.pi * 0.08 * t + np.pi / 4)
        return wav * breath_env * phrase_env

    def _apply_natural_noise(self, wav: np.ndarray, profile: Optional[EmotionProfile] = None) -> np.ndarray:
        """Add subtle colored noise for air/presence (makes audio feel 'live')."""
        noise_level = 0.003
        if profile and profile.voice_quality == "breathy":
            noise_level = 0.008
        elif profile and profile.voice_quality == "tense":
            noise_level = 0.001

        noise = np.random.randn(len(wav)) * noise_level
        b, a = signal.butter(1, 4000 / (self.sr / 2), btype='low')
        colored_noise = signal.lfilter(b, a, noise)
        return wav + colored_noise

    def _dynamic_range_compression(self, wav: np.ndarray, threshold: float = 0.3, ratio: float = 4.0) -> np.ndarray:
        """Apply dynamic range compression for consistent loudness (like a human vocal mic)."""
        compressed = wav.copy()
        above_threshold = np.abs(wav) > threshold
        compressed[above_threshold] = np.sign(wav[above_threshold]) * (
            threshold + (np.abs(wav[above_threshold]) - threshold) / ratio
        )
        return compressed


# =============================================================================
# SECTION 13: MAIN PIPELINE
# =============================================================================

class EmotionalSpeechPipeline:
    """
    High-level pipeline that ties everything together:
    1. Load audio
    2. Extract features
    3. Train model (or load pretrained)
    4. Convert to all emotions
    5. Post-process for naturalness
    6. Save outputs
    7. Run evaluation
    8. Print evaluation table
    """

    def __init__(
        self,
        audio_config: Optional[AudioConfig] = None,
        model_config: Optional[ModelConfig] = None,
        train_config: Optional[TrainConfig] = None,
        device: Optional[str] = None,
        output_dir: str = "emotional_speech_system/outputs",
        checkpoint_dir: str = "emotional_speech_system/checkpoints",
    ):
        self.audio_config = audio_config or AudioConfig()
        self.model_config = model_config or ModelConfig()
        self.train_config = train_config or TrainConfig(checkpoint_dir=checkpoint_dir)
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(checkpoint_dir, exist_ok=True)

        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(device)
        logger.info(f"Using device: {self.device}")

        self.audio_processor = AudioProcessor(self.audio_config)
        self.post_processor = NaturalAudioPostProcessor(self.audio_config.sample_rate)
        self.evaluator = SpeechEvaluator(self.audio_config)

        self.model = EmotionalSpeechConverter(self.model_config, self.audio_config)
        self.model = self.model.to(self.device)

        param_counts = self.model.count_parameters()
        logger.info(f"Model initialized. Parameters: {param_counts}")

    def train(
        self,
        data_dir: Optional[str] = None,
        num_epochs: int = 5,
        resume_from: Optional[str] = None,
    ):
        """Train the model."""
        trainer = EmotionalSpeechTrainer(
            model=self.model,
            train_config=self.train_config,
            audio_config=self.audio_config,
            device=self.device,
            data_dir=data_dir,
        )

        if resume_from and os.path.exists(resume_from):
            trainer.load_checkpoint(resume_from)

        history = trainer.train(num_epochs=num_epochs)
        return history

    def load_pretrained(self, checkpoint_path: str):
        """Load pretrained model weights."""
        state_dict = torch.load(checkpoint_path, map_location=self.device)
        if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
            state_dict = state_dict["model_state_dict"]
        self.model.load_state_dict(state_dict, strict=False)
        logger.info(f"Loaded pretrained model from {checkpoint_path}")

    def process_audio(
        self,
        input_path: Optional[str] = None,
        source_emotion: str = "neutral",
        target_emotions: Optional[List[str]] = None,
        save_outputs: bool = True,
        run_evaluation: bool = True,
    ) -> Dict[str, Any]:
        """
        Full pipeline: load audio -> convert to emotions -> evaluate.

        Args:
            input_path: Path to input WAV file
            source_emotion: Assumed emotion of input audio
            target_emotions: List of emotions to convert to (None = all)
            save_outputs: Whether to save output WAV files
            run_evaluation: Whether to run evaluation metrics

        Returns:
            Dictionary with all results and metrics
        """
        if target_emotions is None:
            target_emotions = list(EMOTION_PROFILES.keys())

        logger.info(f"\n{'='*60}")
        logger.info("EMOTIONAL SPEECH CONVERSION PIPELINE")
        logger.info(f"{'='*60}")

        waveform = self.audio_processor.load_or_generate_audio(input_path, duration_sec=3.0)
        logger.info(f"Input audio: {waveform.shape} @ {self.audio_config.sample_rate}Hz")

        if save_outputs:
            orig_path = os.path.join(self.output_dir, "original.wav")
            self.audio_processor.save_audio(waveform, orig_path)

        features = self.audio_processor.extract_all_features(waveform)
        mel_original = features["mel"].to(self.device)
        f0_original = features["f0"]
        energy_original = features["energy"]
        voiced_original = features["voiced"]

        logger.info(f"Mel shape: {mel_original.shape}")
        logger.info(f"F0 shape: {f0_original.shape}, voiced frames: {voiced_original.sum().int().item()}")
        logger.info(f"Energy mean: {energy_original.mean():.3f}")

        logger.info("\n--- Extracting VAE Latent Representation ---")
        self.model.eval()
        with torch.no_grad():
            mel_batch = mel_original.unsqueeze(0)
            mu, log_var = self.model.get_latent(mel_batch)
            z_source = mu

        logger.info(f"Latent code shape: {z_source.shape}")
        logger.info(f"Latent stats: mean={z_source.mean():.4f}, std={z_source.std():.4f}")

        all_results = {}
        all_metrics = []

        logger.info(f"\n--- Converting to {len(target_emotions)} emotions ---")
        for target_emotion in target_emotions:
            logger.info(f"\n  Processing: {target_emotion.upper()}")
            profile = EMOTION_PROFILES[target_emotion]

            try:
                out = self.model.convert_emotion(
                    mel=mel_original,
                    source_emotion=source_emotion,
                    target_emotion=target_emotion,
                    target_mel_len=mel_original.shape[-1],
                )

                waveform_raw = out["waveform"].squeeze(0)
                mel_converted = out["mel_converted"].squeeze(0)
                f0_converted = out["pitch_pred"].squeeze(0).cpu()
                energy_converted = out["energy_pred"].squeeze(0).cpu()

                waveform_natural = self.post_processor.apply(waveform_raw.cpu(), profile)
                logger.info(f"    Waveform shape: {waveform_natural.shape}")

                if save_outputs:
                    out_path = os.path.join(self.output_dir, f"{target_emotion}.wav")
                    self.audio_processor.save_audio(waveform_natural, out_path)
                    logger.info(f"    Saved: {out_path}")

                if run_evaluation:
                    metrics = self.evaluator.evaluate_conversion(
                        mel_original=mel_original.unsqueeze(0).cpu(),
                        mel_converted=mel_converted.unsqueeze(0).cpu(),
                        waveform_original=waveform,
                        waveform_converted=waveform_natural,
                        f0_original=f0_original,
                        f0_converted=f0_converted,
                        energy_original=energy_original,
                        energy_converted=energy_converted,
                        voiced=voiced_original,
                        emotion_name=target_emotion,
                        emotion_profile=profile,
                        sample_rate=self.audio_config.sample_rate,
                    )
                    all_metrics.append(metrics)

                    mcd = metrics.get("mcd", float('nan'))
                    cos = metrics.get("cosine_similarity", float('nan'))
                    logger.info(f"    MCD={mcd:.3f}  CosSim={cos:.3f}")

                all_results[target_emotion] = {
                    "waveform": waveform_natural,
                    "mel": mel_converted.cpu(),
                    "f0": f0_converted,
                    "energy": energy_converted,
                }

            except Exception as e:
                logger.error(f"  Error converting to {target_emotion}: {e}")
                import traceback
                traceback.print_exc()

        results = {
            "source_waveform": waveform,
            "source_mel": mel_original.cpu(),
            "source_f0": f0_original,
            "source_energy": energy_original,
            "source_voiced": voiced_original,
            "z_source": z_source.cpu(),
            "converted": all_results,
            "metrics": all_metrics,
        }

        if run_evaluation and all_metrics:
            self._print_evaluation_results(all_metrics)

        return results

    def _print_evaluation_results(self, all_metrics: List[Dict[str, Any]]):
        """Print evaluation table and feature analysis."""
        print("\n" + "=" * 100)
        print("EVALUATION RESULTS: EMOTIONAL SPEECH CONVERSION")
        print("=" * 100)
        print("\n[ METRIC LEGEND ]")
        print("  MCD↓         - Mel Cepstral Distortion (dB), lower = more similar spectrum")
        print("  F0-RMSE↓     - F0 Root Mean Square Error (log scale), lower = more accurate pitch")
        print("  F0-Corr↑     - F0 Pearson Correlation, higher = pitch patterns more similar")
        print("  E-RMSE↓      - Energy RMSE, lower = more similar loudness")
        print("  PESQ↑        - Perceptual Eval Speech Quality [1-4.5], higher = better")
        print("  STOI↑        - Short-Time Objective Intelligibility [0-1], higher = better")
        print("  CosSim↑      - Cosine Similarity of mel vectors [0-1], higher = more similar")
        print("  Pitch%Δ      - % change in mean pitch vs. original")
        print("  Energy%Δ     - % change in mean energy vs. original")
        print("  Dur%Δ        - % change in duration vs. original")
        print("  Exp.Pitch    - Expected pitch shift in semitones (from emotion profile)")
        print("  Exp.Energy   - Expected energy scale factor (from emotion profile)")
        print("  Voice Quality- Qualitative voice quality label")

        print("\n[ EVALUATION TABLE ]\n")
        table = self.evaluator.build_evaluation_table(all_metrics)
        print(table)

        self.evaluator.print_feature_analysis(all_metrics)

        if PANDAS_AVAILABLE:
            records = []
            for m in all_metrics:
                records.append({
                    "emotion": m.get("emotion", ""),
                    "mcd": m.get("mcd"),
                    "f0_rmse": m.get("f0_rmse"),
                    "f0_correlation": m.get("f0_correlation"),
                    "energy_rmse": m.get("energy_rmse"),
                    "pesq": m.get("pesq"),
                    "stoi": m.get("stoi"),
                    "cosine_similarity": m.get("cosine_similarity"),
                    "pitch_mean_change_pct": m.get("pitch_mean_change_pct"),
                    "energy_mean_change_pct": m.get("energy_mean_change_pct"),
                    "duration_change_pct": m.get("duration_change_pct"),
                    "expected_pitch_shift_semitones": m.get("expected_pitch_shift_semitones"),
                    "expected_energy_scale": m.get("expected_energy_scale"),
                    "voice_quality": m.get("voice_quality"),
                })
            try:
                df = pd.DataFrame(records)
                csv_path = os.path.join(self.output_dir, "evaluation_results.csv")
                df.to_csv(csv_path, index=False)
                print(f"\n[ Evaluation results saved to: {csv_path} ]")
            except Exception as e:
                logger.warning(f"Could not save CSV: {e}")

    def print_model_architecture(self):
        """Print detailed model architecture summary."""
        print("\n" + "=" * 80)
        print("MODEL ARCHITECTURE: EMOTIONAL SPEECH CONVERSION SYSTEM")
        print("=" * 80)

        print("\n[ COMPONENT 1: VARIATIONAL AUTOENCODER (VAE) ]")
        print("  Purpose: Encode voice audio into latent space (content + speaker)")
        print(f"  Encoder: Conv1D ResBlocks x{self.model_config.vae_num_layers} -> Pooling -> μ,σ ∈ R^{self.model_config.latent_dim}")
        print(f"  Decoder: FC({self.model_config.latent_dim}) -> TransConv1D x3 -> ResBlocks x{self.model_config.vae_num_layers} -> Mel")
        print(f"  Emotion Conditioning: Embedding({self.model_config.num_emotions}) + MLP mapper")
        print(f"  KL Annealing: beta={self.train_config.beta_kl} -> {self.train_config.beta_kl_max} over {self.train_config.beta_kl_anneal_steps} steps")

        print("\n[ COMPONENT 2: GRAPH NEURAL NETWORK (GNN) ]")
        print("  Purpose: Map source emotion latent -> target emotion latent")
        print(f"  Graph: {self.model_config.num_emotions} emotion nodes in Valence-Arousal space")
        print(f"  Edges: weighted by VA-space proximity (similarity > 0.3 threshold)")
        print(f"  Layers: {self.model_config.gnn_num_layers}x GAT ({self.model_config.gnn_num_heads} heads) + FFN + LayerNorm")
        print(f"  Message Passing: Graph Attention with learnable node features")
        print(f"  Output: z_target ∈ R^{self.model_config.latent_dim} + prosody_target ∈ R^6")

        print("\n[ COMPONENT 3: FASTSPEECH 2 VARIANCE ADAPTOR ]")
        print("  Purpose: Generate mel spectrogram with emotion-controlled variance")
        print(f"  Phoneme Encoder: {self.model_config.fastspeech_num_layers}x FFT Blocks (d={self.model_config.fastspeech_hidden})")
        print(f"  Duration Predictor: Conv-based -> Length Regulator (rate control)")
        print(f"  Pitch Predictor: Conv-based -> Quantized embedding ({self.model_config.pitch_bins} bins)")
        print(f"  Energy Predictor: Conv-based -> Quantized embedding ({self.model_config.energy_bins} bins)")
        print(f"  Mel Encoder: {self.model_config.fastspeech_num_layers}x FFT Blocks -> Linear -> Mel")
        print(f"  Emotion Gate: Soft-gate between content hidden and emotion hidden")

        print("\n[ COMPONENT 4: HIFI-GAN VOCODER ]")
        print("  Purpose: Convert mel spectrogram -> natural-sounding waveform")
        print(f"  Architecture: HiFi-GAN V1 (best quality/naturalness tradeoff)")
        print(f"  Upsample Rates: {self.model_config.hifigan_upsample_rates} = {math.prod(self.model_config.hifigan_upsample_rates)}x total upsampling")
        print(f"  ResBlock Kernels: {self.model_config.hifigan_resblock_kernel_sizes}")
        print(f"  Dilations: {self.model_config.hifigan_resblock_dilation_sizes}")
        print(f"  Discriminators: Multi-Scale (3) + Multi-Period (5 periods: 2,3,5,7,11)")
        print(f"  Key: Multi-Receptive Field Fusion (MRF) captures all speech timescales")

        print("\n[ POST-PROCESSING PIPELINE ]")
        print("  1. Micro-jitter & shimmer (humanize voice, remove AI flatness)")
        print("  2. Spectral shaping per voice quality (bright/breathy/tense/creaky/tremulous)")
        print("  3. Breath amplitude modulation (natural breathing rhythm)")
        print("  4. Colored noise addition (air/presence/live feel)")
        print("  5. Dynamic range compression (consistent vocal loudness)")

        param_counts = self.model.count_parameters()
        print("\n[ PARAMETER COUNTS ]")
        for name, count in param_counts.items():
            print(f"  {name:25s}: {count:>10,}")
        print("=" * 80)


# =============================================================================
# SECTION 14: EMOTION GRAPH VISUALIZATION
# =============================================================================

def print_emotion_graph():
    """Print text representation of the emotion graph."""
    print("\n" + "=" * 60)
    print("EMOTION GRAPH (Valence-Arousal Space)")
    print("=" * 60)

    va_positions = {
        "neutral":   (0.5, 0.5),
        "happy":     (0.9, 0.7),
        "sad":       (0.1, 0.2),
        "angry":     (0.1, 0.9),
        "fearful":   (0.2, 0.8),
        "surprised": (0.6, 0.9),
        "disgusted": (0.1, 0.6),
    }

    print("\nValence-Arousal Positions:")
    print(f"  {'Emotion':<12} {'Valence':>8} {'Arousal':>8}")
    print("  " + "-" * 30)
    for emotion, (v, a) in va_positions.items():
        print(f"  {emotion:<12} {v:>8.2f} {a:>8.2f}")

    print("\nEmotion Profiles:")
    print(f"  {'Emotion':<12} {'Pitch Shift':>12} {'Speed':>8} {'Energy':>8} {'Voice Quality':<14}")
    print("  " + "-" * 60)
    for name, p in EMOTION_PROFILES.items():
        print(f"  {name:<12} {p.pitch_shift:>+11.1f}st {p.speed_factor:>7.2f}x {p.energy_scale:>7.2f}x {p.voice_quality:<14}")

    print("\nGraph Edges (high similarity):")
    emotions = list(EMOTION_PROFILES.keys())
    for i, e1 in enumerate(emotions):
        for j, e2 in enumerate(emotions):
            if i < j:
                v1, a1 = va_positions[e1]
                v2, a2 = va_positions[e2]
                dist = math.sqrt((v1 - v2) ** 2 + (a1 - a2) ** 2)
                sim = 1.0 - dist / math.sqrt(2.0)
                if sim > 0.5:
                    print(f"  {e1:<12} <-> {e2:<12}  similarity={sim:.3f}")
    print("=" * 60)


# =============================================================================
# SECTION 15: DEMO AUDIO GENERATOR
# =============================================================================

def generate_emotion_demo_audio(
    output_dir: str = "emotional_speech_system/sample_audio",
    sr: int = 22050,
    duration: float = 2.5
) -> Dict[str, str]:
    """
    Generate demo reference audio for each emotion using synthesis.
    These serve as reference inputs when no real audio is provided.
    """
    os.makedirs(output_dir, exist_ok=True)
    audio_config = AudioConfig(sample_rate=sr)
    processor = AudioProcessor(audio_config)
    saved_paths = {}

    logger.info("Generating demo audio samples for each emotion...")

    for emotion_name, profile in EMOTION_PROFILES.items():
        waveform = processor._generate_demo_signal(duration_sec=duration)

        wav_np = waveform.squeeze().numpy()
        if profile.speed_factor != 1.0:
            indices = np.linspace(0, len(wav_np) - 1, int(len(wav_np) / profile.speed_factor))
            indices = np.clip(indices.astype(int), 0, len(wav_np) - 1)
            wav_np = wav_np[indices]

        f0_base = 120.0 * (2 ** (profile.pitch_shift / 12.0))
        t = np.linspace(0, len(wav_np) / sr, len(wav_np))
        pitch_mod = 1.0 + profile.pitch_variance * np.sin(2 * np.pi * 5 * t)
        phase = np.cumsum(2 * np.pi * f0_base * pitch_mod / sr)
        harmonic = np.zeros_like(wav_np)
        for k in range(1, 8):
            harmonic += (1.0 / k) * np.sin(k * phase)

        harmonic = harmonic / (np.max(np.abs(harmonic)) + 1e-8)
        wav_np = 0.7 * wav_np + 0.3 * harmonic
        wav_np = wav_np * profile.energy_scale
        wav_np = wav_np / (np.max(np.abs(wav_np)) + 1e-8) * 0.85

        out_path = os.path.join(output_dir, f"{emotion_name}_demo.wav")
        wav_int16 = np.clip(wav_np * 32767.0, -32768, 32767).astype(np.int16)
        wav_io.write(out_path, sr, wav_int16)
        saved_paths[emotion_name] = out_path
        logger.info(f"  Generated: {out_path}")

    return saved_paths


# =============================================================================
# SECTION 16: COMMAND LINE INTERFACE (COLAB-FRIENDLY)
# =============================================================================

def parse_args():
    parser = argparse.ArgumentParser(
        description="Emotional Speech Conversion System: VAE + GNN + FastSpeech2 + HiFi-GAN",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # Run full demo with synthetic audio (no real audio needed):
  python emotional_speech_system.py --demo

  # Convert your voice audio to all emotions:
  python emotional_speech_system.py --input your_voice.wav

  # Convert to specific emotion:
  python emotional_speech_system.py --input your_voice.wav --emotion happy

  # Train the model first, then convert:
  python emotional_speech_system.py --input your_voice.wav --train --epochs 10

  # Show model architecture:
  python emotional_speech_system.py --architecture

  # Show emotion graph:
  python emotional_speech_system.py --emotion-graph
        """
    )

    parser.add_argument(
        "--input", "-i", type=str, default=None,
        help="Path to input voice audio file (WAV format)"
    )
    parser.add_argument(
        "--emotion", "-e", type=str, default=None,
        choices=list(EMOTION_PROFILES.keys()),
        help="Target emotion (default: convert to all emotions)"
    )
    parser.add_argument(
        "--all-emotions", action="store_true",
        help="Convert to all supported emotions (default behavior)"
    )
    parser.add_argument(
        "--source-emotion", type=str, default="neutral",
        choices=list(EMOTION_PROFILES.keys()),
        help="Emotion of the source audio (default: neutral)"
    )
    parser.add_argument(
        "--train", action="store_true",
        help="Train the model before converting"
    )
    parser.add_argument(
        "--data-dir", type=str, default=None,
        help="Directory with training data (subdirs per emotion)"
    )
    parser.add_argument(
        "--epochs", type=int, default=5,
        help="Number of training epochs"
    )
    parser.add_argument(
        "--checkpoint", type=str, default=None,
        help="Path to pretrained model checkpoint"
    )
    parser.add_argument(
        "--output-dir", type=str, default="emotional_speech_system/outputs",
        help="Output directory for generated audio files"
    )
    parser.add_argument(
        "--demo", action="store_true",
        help="Run full demo with synthetic audio"
    )
    parser.add_argument(
        "--architecture", action="store_true",
        help="Print model architecture and exit"
    )
    parser.add_argument(
        "--emotion-graph", action="store_true",
        help="Print emotion graph and exit"
    )
    parser.add_argument(
        "--no-eval", action="store_true",
        help="Skip evaluation metrics"
    )
    parser.add_argument(
        "--device", type=str, default=None,
        choices=["cpu", "cuda", "mps"],
        help="Device to run on"
    )
    parser.add_argument(
        "--latent-dim", type=int, default=128,
        help="Latent dimension for VAE"
    )
    parser.add_argument(
        "--learning-rate", type=float, default=1e-4,
        help="Learning rate for training"
    )

    # --- COLAB FIX: use parse_known_args to ignore Colab's kernel connection args ---
    args, unknown = parser.parse_known_args()
    if unknown:
        logger.warning(f"Ignored unknown arguments: {unknown}")
    return args


def main():
    args = parse_args()

    print("\n" + "=" * 80)
    print("EMOTIONAL SPEECH CONVERSION SYSTEM")
    print("VAE + Graph Neural Network + FastSpeech 2 + HiFi-GAN Vocoder")
    print("=" * 80)
    print(f"Python: {sys.version.split()[0]}")
    print(f"PyTorch: {torch.__version__}")
    print(f"torchaudio: {torchaudio.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print("=" * 80 + "\n")

    audio_config = AudioConfig()
    model_config = ModelConfig(latent_dim=args.latent_dim)
    train_config = TrainConfig(
        learning_rate=args.learning_rate,
        num_epochs=args.epochs,
        checkpoint_dir=os.path.join(args.output_dir, "checkpoints"),
    )

    if args.emotion_graph:
        print_emotion_graph()
        return

    pipeline = EmotionalSpeechPipeline(
        audio_config=audio_config,
        model_config=model_config,
        train_config=train_config,
        device=args.device,
        output_dir=args.output_dir,
        checkpoint_dir=os.path.join(args.output_dir, "checkpoints"),
    )

    if args.architecture:
        pipeline.print_model_architecture()
        return

    if args.checkpoint:
        pipeline.load_pretrained(args.checkpoint)

    if args.train or args.demo:
        logger.info(f"\nTraining model for {args.epochs} epochs...")
        history = pipeline.train(
            data_dir=args.data_dir,
            num_epochs=args.epochs,
        )
        if history['gen_loss']:
            logger.info(f"Training complete. Final gen loss: {history['gen_loss'][-1]:.4f}")
        else:
            logger.info("Training skipped (0 epochs)")

    if args.demo and not args.input:
        demo_paths = generate_emotion_demo_audio(
            output_dir=os.path.join(args.output_dir, "demo_reference"),
            sr=audio_config.sample_rate,
        )
        input_path = None
        logger.info("Using synthetic demo audio as input")
    else:
        input_path = args.input

    target_emotions = [args.emotion] if args.emotion else list(EMOTION_PROFILES.keys())

    logger.info(f"\nProcessing audio: {input_path or 'synthetic demo'}")
    logger.info(f"Target emotions: {target_emotions}")

    results = pipeline.process_audio(
        input_path=input_path,
        source_emotion=args.source_emotion,
        target_emotions=target_emotions,
        save_outputs=True,
        run_evaluation=not args.no_eval,
    )

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"\nOutput directory: {os.path.abspath(args.output_dir)}")
    for emotion in target_emotions:
        wav_path = os.path.join(args.output_dir, f"{emotion}.wav")
        if os.path.exists(wav_path):
            size_kb = os.path.getsize(wav_path) / 1024
            print(f"  {emotion:<12}: {wav_path}  ({size_kb:.1f} KB)")

    csv_path = os.path.join(args.output_dir, "evaluation_results.csv")
    if os.path.exists(csv_path):
        print(f"\n  Evaluation CSV: {csv_path}")

    print("\n" + "=" * 80)
    print("PIPELINE COMPLETE")
    print("=" * 80)

    print("\nLATENT SPACE ANALYSIS (VAE Encoding)")
    z = results.get("z_source")
    if z is not None:
        z_np = z.squeeze().numpy()
        print(f"  Latent vector shape : {z_np.shape}")
        print(f"  Mean               : {z_np.mean():.4f}")
        print(f"  Std                : {z_np.std():.4f}")
        print(f"  Min/Max            : {z_np.min():.4f} / {z_np.max():.4f}")
        top_dims = np.argsort(np.abs(z_np))[-5:][::-1]
        print(f"  Top-5 active dims  : {top_dims.tolist()}")
        print(f"    Values: {z_np[top_dims].tolist()}")

    print("\nGRAPH NEURAL NETWORK: Emotion Embedding Space")
    embeddings = pipeline.model.gnn.get_emotion_embeddings()
    print(f"  {'Emotion':<12}  {'Embed-Norm':>12}")
    for ename, emb in embeddings.items():
        norm = emb.norm().item()
        print(f"  {ename:<12}  {norm:>12.4f}")

    print("\n")
    return results


# =============================================================================
# SECTION 17: UNIT TESTS
# =============================================================================

def run_unit_tests():
    """Run basic unit tests to verify all components work."""
    print("\n" + "=" * 60)
    print("RUNNING UNIT TESTS")
    print("=" * 60)

    device = torch.device("cpu")
    audio_config = AudioConfig()
    model_config = ModelConfig(
        latent_dim=64,
        vae_hidden_dim=128,
        vae_num_layers=2,
        gnn_hidden_dim=128,
        gnn_num_layers=2,
        fastspeech_hidden=128,
        fastspeech_num_layers=2,
        fastspeech_filter_size=256,
        hifigan_upsample_initial_channel=128,
    )

    print("\n[1] Testing AudioProcessor...")
    processor = AudioProcessor(audio_config)
    waveform = processor._generate_demo_signal(duration_sec=1.0)
    assert waveform.shape[0] == 1, "Waveform should be mono"
    assert waveform.shape[1] == 22050, f"Expected 22050 samples, got {waveform.shape[1]}"
    mel = processor.get_mel_spectrogram(waveform)
    assert mel.shape[0] == 80, f"Expected 80 mel bins, got {mel.shape[0]}"
    f0, voiced = processor.get_pitch(waveform)
    energy = processor.get_energy(mel)
    print(f"  Mel: {mel.shape}, F0: {f0.shape}, Energy: {energy.shape} -> OK")

    print("\n[2] Testing VAE Encoder/Decoder...")
    vae = VoiceVAE(model_config, n_mels=80)
    mel_batch = mel.unsqueeze(0)
    mu, log_var = vae.encode(mel_batch)
    assert mu.shape == (1, 64), f"Expected (1,64), got {mu.shape}"
    z = vae.reparameterize(mu, log_var)
    mel_recon = vae.decode(z, mel.shape[1])
    assert mel_recon.shape[1] == 80, f"Expected 80 mel bins in output, got {mel_recon.shape[1]}"
    print(f"  mu: {mu.shape}, z: {z.shape}, mel_recon: {mel_recon.shape} -> OK")

    print("\n[3] Testing GNN...")
    gnn = EmotionGNN(model_config)
    z_test = torch.randn(2, 64)
    src_ids = torch.tensor([0, 1])
    tgt_ids = torch.tensor([2, 4])
    gnn_out = gnn(z_test, src_ids, tgt_ids)
    assert gnn_out["z_target"].shape == (2, 64), f"GNN output shape mismatch"
    assert gnn_out["prosody_target"].shape == (2, 6), f"GNN prosody shape mismatch"
    print(f"  z_target: {gnn_out['z_target'].shape}, prosody: {gnn_out['prosody_target'].shape} -> OK")

    print("\n[4] Testing FastSpeech2 Variance Adaptor...")
    va = FastSpeech2VarianceAdaptor(model_config, n_mels=80)
    z_s = torch.randn(1, 64)
    z_e = torch.randn(1, 64)
    profile = EMOTION_PROFILES["happy"]
    va_out = va(z_s, z_e, target_mel_len=100, emotion_profile=profile)
    assert va_out["mel"].shape[-1] == 100, f"Expected mel len 100, got {va_out['mel'].shape[-1]}"
    print(f"  mel: {va_out['mel'].shape}, pitch: {va_out['pitch_pred'].shape} -> OK")

    print("\n[5] Testing HiFi-GAN Generator...")
    hifigan = HiFiGANGenerator(model_config, n_mels=80)
    mel_in = torch.randn(1, 80, 100)
    wav_out = hifigan(mel_in)
    expected_len = 100 * math.prod(model_config.hifigan_upsample_rates)
    assert wav_out.shape[0] == 1 and wav_out.shape[1] == 1, "HiFi-GAN output shape error"
    print(f"  Input mel: {mel_in.shape} -> Waveform: {wav_out.shape} -> OK")

    print("\n[6] Testing Full Forward Pass...")
    full_model = EmotionalSpeechConverter(model_config, audio_config)
    mel_input = mel.unsqueeze(0)
    src_id = torch.tensor([0])
    tgt_id = torch.tensor([1])
    with torch.no_grad():
        output = full_model(
            mel=mel_input,
            source_emotion_id=src_id,
            target_emotion_id=tgt_id,
            target_mel_len=mel.shape[1]
        )
    assert "waveform" in output
    assert "mel_converted" in output
    assert "z_source" in output
    print(f"  waveform: {output['waveform'].shape}, mel: {output['mel_converted'].shape} -> OK")

    print("\n[7] Testing Evaluation Metrics...")
    evaluator = SpeechEvaluator(audio_config)
    mel_ref = torch.randn(1, 80, 100)
    mel_hyp = torch.randn(1, 80, 100)
    mcd = evaluator.mel_cepstral_distortion(mel_ref, mel_hyp)
    cos_sim = evaluator.cosine_similarity(mel_ref, mel_hyp)
    assert 0 <= mcd, "MCD should be non-negative"
    assert -1 <= cos_sim <= 1, "Cosine similarity should be in [-1, 1]"
    print(f"  MCD: {mcd:.3f}, CosSim: {cos_sim:.3f} -> OK")

    print("\n[8] Testing Post-Processor...")
    pp = NaturalAudioPostProcessor(sample_rate=22050)
    wav_raw = torch.randn(1, 22050) * 0.1
    for ename, eprofile in EMOTION_PROFILES.items():
        wav_processed = pp.apply(wav_raw, eprofile)
        assert wav_processed.shape == wav_raw.shape, f"Post-processor shape mismatch for {ename}"
    print(f"  All {len(EMOTION_PROFILES)} emotions processed -> OK")

    print("\n[9] Testing Dataset (synthetic)...")
    dataset = SpeechDataset(None, AudioProcessor(audio_config), use_synthetic=True, synthetic_samples=14)
    assert len(dataset) == 14, f"Dataset length mismatch"
    sample = dataset[0]
    required_keys = ["mel", "f0", "energy", "emotion_id", "prosody"]
    for k in required_keys:
        assert k in sample, f"Missing key {k} in dataset sample"
    print(f"  Dataset length: {len(dataset)}, sample keys: {list(sample.keys())} -> OK")

    print("\n" + "=" * 60)
    print("ALL UNIT TESTS PASSED")
    print("=" * 60 + "\n")


# =============================================================================
# SECTION 18: ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    if "--test" in sys.argv:
        run_unit_tests()
    else:
        main()


EMOTIONAL SPEECH CONVERSION SYSTEM
VAE + Graph Neural Network + FastSpeech 2 + HiFi-GAN Vocoder
Python: 3.12.13
PyTorch: 2.10.0+cpu
torchaudio: 2.10.0+cpu
CUDA available: False


EVALUATION RESULTS: EMOTIONAL SPEECH CONVERSION

[ METRIC LEGEND ]
  MCD↓         - Mel Cepstral Distortion (dB), lower = more similar spectrum
  F0-RMSE↓     - F0 Root Mean Square Error (log scale), lower = more accurate pitch
  F0-Corr↑     - F0 Pearson Correlation, higher = pitch patterns more similar
  E-RMSE↓      - Energy RMSE, lower = more similar loudness
  PESQ↑        - Perceptual Eval Speech Quality [1-4.5], higher = better
  STOI↑        - Short-Time Objective Intelligibility [0-1], higher = better
  CosSim↑      - Cosine Similarity of mel vectors [0-1], higher = more similar
  Pitch%Δ      - % change in mean pitch vs. original
  Energy%Δ     - % change in mean energy vs. original
  Dur%Δ        - % change in duration vs. original
  Exp.Pitch    - Expected pitch shift in semitones (from emotion pr

In [ ]:
# -*- coding: utf-8 -*-


import os, sys, math, time, copy, json, argparse, warnings, logging
from typing import Dict, List, Optional, Tuple, Any
from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import scipy.signal as signal
import scipy.io.wavfile as wav_io
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Normal

import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as TAF

try:
    import librosa
    import librosa.display
    LIBROSA_AVAILABLE = True
except ImportError:
    LIBROSA_AVAILABLE = False
    print("[WARN] librosa not available, using torchaudio fallback")

try:
    import soundfile as sf
    SF_AVAILABLE = True
except ImportError:
    SF_AVAILABLE = False

try:
    from pesq import pesq
    PESQ_AVAILABLE = True
except ImportError:
    PESQ_AVAILABLE = False
    print("[WARN] pesq not available, PESQ metric will be skipped")

try:
    from pystoi import stoi
    STOI_AVAILABLE = True
except ImportError:
    STOI_AVAILABLE = False
    print("[WARN] pystoi not available, STOI metric will be skipped")

try:
    import torch_geometric.nn as gnn_layers
    from torch_geometric.data import Data as GraphData, Batch as GraphBatch
    TORCH_GEOMETRIC_AVAILABLE = True
except ImportError:
    TORCH_GEOMETRIC_AVAILABLE = False
    print("[WARN] torch_geometric not available, using custom GNN implementation")

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

try:
    from tabulate import tabulate
    TABULATE_AVAILABLE = True
except ImportError:
    TABULATE_AVAILABLE = False

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)


# =============================================================================
# SECTION 1: CONFIGURATION
# =============================================================================

@dataclass
class AudioConfig:
    sample_rate: int = 22050
    hop_length: int = 256
    win_length: int = 1024
    n_fft: int = 1024
    n_mels: int = 80
    fmin: float = 80.0
    fmax: float = 7600.0
    min_level_db: float = -100.0
    ref_level_db: float = 20.0
    preemphasis: float = 0.97
    max_wav_value: float = 32768.0
    trim_silence: bool = True
    trim_top_db: float = 40.0

@dataclass
class ModelConfig:
    latent_dim: int = 128
    vae_hidden_dim: int = 256
    vae_num_layers: int = 4
    gnn_hidden_dim: int = 256
    gnn_num_layers: int = 4
    gnn_num_heads: int = 4
    fastspeech_hidden: int = 256
    fastspeech_num_heads: int = 2
    fastspeech_num_layers: int = 4
    fastspeech_kernel_size: int = 9
    fastspeech_filter_size: int = 1024
    hifigan_upsample_rates: List[int] = field(default_factory=lambda: [8, 8, 2, 2])
    hifigan_upsample_kernel_sizes: List[int] = field(default_factory=lambda: [16, 16, 4, 4])
    hifigan_upsample_initial_channel: int = 512
    hifigan_resblock_kernel_sizes: List[int] = field(default_factory=lambda: [3, 7, 11])
    hifigan_resblock_dilation_sizes: List[List[int]] = field(default_factory=lambda: [[1, 3, 5], [1, 3, 5], [1, 3, 5]])
    dropout: float = 0.1
    num_emotions: int = 7
    pitch_bins: int = 256
    energy_bins: int = 256

@dataclass
class TrainConfig:
    batch_size: int = 8
    learning_rate: float = 1e-4
    num_epochs: int = 100
    warmup_steps: int = 4000
    beta_kl: float = 0.0001
    beta_kl_max: float = 0.1
    beta_kl_anneal_steps: int = 20000
    grad_clip: float = 1.0
    checkpoint_dir: str = "checkpoints"
    log_interval: int = 50
    save_interval: int = 10
    lambda_pitch: float = 1.0
    lambda_energy: float = 1.0
    lambda_duration: float = 1.0
    lambda_mel: float = 1.0
    lambda_emotion: float = 0.5

@dataclass
class EmotionProfile:
    name: str
    pitch_shift: float
    speed_factor: float
    energy_scale: float
    pitch_variance: float
    energy_variance: float
    duration_factor: float
    voice_quality: str
    description: str
    intensity_range: Tuple[float, float]


EMOTION_PROFILES: Dict[str, EmotionProfile] = {
    "neutral": EmotionProfile(
        name="neutral",
        pitch_shift=0.0,
        speed_factor=1.0,
        energy_scale=1.0,
        pitch_variance=0.05,
        energy_variance=0.05,
        duration_factor=1.0,
        voice_quality="modal",
        description="Calm, flat affect, balanced pitch and energy",
        intensity_range=(0.3, 0.7)
    ),
    "happy": EmotionProfile(
        name="happy",
        pitch_shift=2.5,
        speed_factor=1.12,
        energy_scale=1.20,
        pitch_variance=0.18,
        energy_variance=0.15,
        duration_factor=0.92,
        voice_quality="bright",
        description="Higher pitch, faster rate, more energy, wider range",
        intensity_range=(0.5, 1.0)
    ),
    "sad": EmotionProfile(
        name="sad",
        pitch_shift=-2.0,
        speed_factor=0.84,
        energy_scale=0.72,
        pitch_variance=0.06,
        energy_variance=0.05,
        duration_factor=1.22,
        voice_quality="breathy",
        description="Lower pitch, slower rate, reduced energy, narrow range",
        intensity_range=(0.3, 0.8)
    ),
    "angry": EmotionProfile(
        name="angry",
        pitch_shift=1.5,
        speed_factor=1.08,
        energy_scale=1.55,
        pitch_variance=0.22,
        energy_variance=0.28,
        duration_factor=0.88,
        voice_quality="tense",
        description="High energy, tense voice, aggressive pitch jumps",
        intensity_range=(0.6, 1.0)
    ),
    "fearful": EmotionProfile(
        name="fearful",
        pitch_shift=3.5,
        speed_factor=1.18,
        energy_scale=0.85,
        pitch_variance=0.25,
        energy_variance=0.20,
        duration_factor=0.90,
        voice_quality="tremulous",
        description="High, unstable pitch, fast rate, irregular energy",
        intensity_range=(0.5, 1.0)
    ),
    "surprised": EmotionProfile(
        name="surprised",
        pitch_shift=4.0,
        speed_factor=1.05,
        energy_scale=1.30,
        pitch_variance=0.30,
        energy_variance=0.22,
        duration_factor=0.85,
        voice_quality="bright",
        description="Very high pitch onset, wide pitch range, strong onset energy",
        intensity_range=(0.5, 1.0)
    ),
    "disgusted": EmotionProfile(
        name="disgusted",
        pitch_shift=-1.0,
        speed_factor=0.92,
        energy_scale=1.10,
        pitch_variance=0.12,
        energy_variance=0.18,
        duration_factor=1.08,
        voice_quality="creaky",
        description="Low, creaky voice, moderate energy, deliberate pacing",
        intensity_range=(0.3, 0.9)
    ),
}

EMOTION_IDX = {name: i for i, name in enumerate(EMOTION_PROFILES.keys())}
IDX_EMOTION = {i: name for name, i in EMOTION_IDX.items()}


# =============================================================================
# SECTION 2: AUDIO PROCESSING UTILITIES
# =============================================================================

class AudioProcessor:
    """
    Handles all audio I/O, feature extraction, and signal processing.
    Features extracted: Mel spectrogram, F0 (pitch), Energy, Duration (phone-level).
    """

    def __init__(self, config: AudioConfig):
        self.config = config
        self.sr = config.sample_rate
        self.hop = config.hop_length
        self.win = config.win_length
        self.n_fft = config.n_fft
        self.n_mels = config.n_mels

        self.mel_transform = T.MelSpectrogram(
            sample_rate=self.sr,
            n_fft=self.n_fft,
            win_length=self.win,
            hop_length=self.hop,
            f_min=config.fmin,
            f_max=config.fmax,
            n_mels=self.n_mels,
            power=1.0,
            normalized=False,
            center=True,
        )
        self.amplitude_to_db = T.AmplitudeToDB(stype='amplitude', top_db=80.0)

    def load_audio(self, path: str) -> Tuple[torch.Tensor, int]:
        """Load audio file and resample if needed."""
        try:
            sample_rate_file, data = wav_io.read(path)
            if data.dtype == np.int16:
                data = data.astype(np.float32) / 32768.0
            elif data.dtype == np.int32:
                data = data.astype(np.float32) / 2147483648.0
            elif data.dtype != np.float32:
                data = data.astype(np.float32)
            if data.ndim == 2:
                data = data.mean(axis=1)
            waveform = torch.from_numpy(data).unsqueeze(0)
            orig_sr = sample_rate_file
        except Exception:
            waveform, orig_sr = torchaudio.load(path)
        if orig_sr != self.sr:
            resampler = T.Resample(orig_sr, self.sr)
            waveform = resampler(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if self.config.trim_silence:
            waveform = self._trim_silence(waveform)
        return waveform, self.sr

    def _trim_silence(self, waveform: torch.Tensor, top_db: float = 40.0) -> torch.Tensor:
        """Trim leading/trailing silence from waveform."""
        rms = torch.sqrt(torch.mean(waveform ** 2, dim=-1))
        thresh = 10 ** (-top_db / 20.0) * waveform.abs().max()
        mask = waveform.abs() > thresh
        if mask.any():
            start = mask.float().argmax().item()
            end = len(mask[0]) - mask[0].flip(0).float().argmax().item()
            waveform = waveform[:, int(start):int(end)]
        return waveform

    def preemphasis(self, waveform: torch.Tensor) -> torch.Tensor:
        """Apply preemphasis filter."""
        coef = self.config.preemphasis
        filtered = torch.cat([waveform[:, :1], waveform[:, 1:] - coef * waveform[:, :-1]], dim=1)
        return filtered

    def get_mel_spectrogram(self, waveform: torch.Tensor) -> torch.Tensor:
        """Compute log-mel spectrogram. Returns [n_mels, T]."""
        waveform = self.preemphasis(waveform)
        mel = self.mel_transform(waveform)
        mel_db = self.amplitude_to_db(mel)
        mel_db = mel_db.squeeze(0)
        return mel_db

    def get_pitch(self, waveform: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Extract F0 (fundamental frequency) using autocorrelation.
        Returns f0_hz [T_frames] and voiced_flag [T_frames].
        """
        wav_np = waveform.squeeze().numpy()
        f0, voiced = self._yin_f0(wav_np, self.sr, self.hop)
        f0_tensor = torch.from_numpy(f0).float()
        voiced_tensor = torch.from_numpy(voiced).float()
        return f0_tensor, voiced_tensor

    def _yin_f0(
        self,
        wav: np.ndarray,
        sr: int,
        hop: int,
        f_min: float = 80.0,
        f_max: float = 400.0,
        threshold: float = 0.12,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        YIN algorithm for fundamental frequency estimation.
        Based on: de Cheveigne & Kawahara (2002).
        """
        frame_size = 1024
        min_lag = int(sr / f_max)
        max_lag = int(sr / f_min)

        num_frames = (len(wav) - frame_size) // hop + 1
        f0 = np.zeros(num_frames)
        voiced = np.zeros(num_frames, dtype=bool)

        for i in range(num_frames):
            start = i * hop
            frame = wav[start: start + frame_size]
            if len(frame) < frame_size:
                break
            df = self._difference_function(frame, frame_size, max_lag)
            cmdf = self._cumulative_mean_normalized(df, max_lag)
            tau = self._absolute_threshold(cmdf, min_lag, max_lag, threshold)
            if tau is not None:
                f0[i] = sr / self._parabolic_interpolation(cmdf, tau)
                voiced[i] = True

        f0 = self._smooth_f0(f0, voiced)
        f0_log = np.where(voiced, np.log(np.maximum(f0, 1e-5)), 0.0)
        return f0_log.astype(np.float32), voiced.astype(np.float32)

    def _difference_function(self, frame: np.ndarray, N: int, max_lag: int) -> np.ndarray:
        df = np.zeros(max_lag)
        for tau in range(1, max_lag):
            diff = frame[:N - tau] - frame[tau:N]
            df[tau] = np.sum(diff ** 2)
        return df

    def _cumulative_mean_normalized(self, df: np.ndarray, max_lag: int) -> np.ndarray:
        cmdf = np.zeros(max_lag)
        cmdf[0] = 1.0
        cumsum = 0.0
        for tau in range(1, max_lag):
            cumsum += df[tau]
            if cumsum > 0:
                cmdf[tau] = df[tau] * tau / cumsum
            else:
                cmdf[tau] = 1.0
        return cmdf

    def _absolute_threshold(
        self,
        cmdf: np.ndarray,
        min_lag: int,
        max_lag: int,
        threshold: float
    ) -> Optional[int]:
        tau = min_lag
        while tau < max_lag:
            if cmdf[tau] < threshold:
                while tau + 1 < max_lag and cmdf[tau + 1] < cmdf[tau]:
                    tau += 1
                return tau
            tau += 1
        return int(np.argmin(cmdf[min_lag:max_lag])) + min_lag if max_lag > min_lag else None

    def _parabolic_interpolation(self, array: np.ndarray, x: int) -> float:
        if x <= 0 or x >= len(array) - 1:
            return float(x)
        denom = 2.0 * (2.0 * array[x] - array[x - 1] - array[x + 1])
        if abs(denom) < 1e-9:
            return float(x)
        delta = (array[x - 1] - array[x + 1]) / denom
        return float(x) + delta

    def _smooth_f0(self, f0: np.ndarray, voiced: np.ndarray, kernel_size: int = 3) -> np.ndarray:
        smoothed = f0.copy()
        for i in range(kernel_size // 2, len(f0) - kernel_size // 2):
            if voiced[i]:
                window = f0[i - kernel_size // 2: i + kernel_size // 2 + 1]
                smoothed[i] = np.mean(window[voiced[i - kernel_size // 2: i + kernel_size // 2 + 1]])
        return smoothed

    def get_energy(self, mel: torch.Tensor) -> torch.Tensor:
        """
        Compute frame-level energy from mel spectrogram.
        Returns [T_frames] energy values.
        """
        energy = torch.norm(mel, dim=0, p=2)
        return energy

    def get_duration(self, waveform: torch.Tensor) -> torch.Tensor:
        """
        Estimate pseudo-phoneme durations using energy-based segmentation.
        Returns list of frame durations per segment.
        """
        mel = self.get_mel_spectrogram(waveform)
        energy = self.get_energy(mel)
        T = energy.shape[0]
        segment_len = max(5, T // 20)
        durations = []
        for i in range(0, T, segment_len):
            end = min(i + segment_len, T)
            durations.append(end - i)
        return torch.tensor(durations, dtype=torch.float32)

    def extract_all_features(self, waveform: torch.Tensor) -> Dict[str, torch.Tensor]:
        """Extract all features needed by the model."""
        mel = self.get_mel_spectrogram(waveform)
        f0, voiced = self.get_pitch(waveform)
        energy = self.get_energy(mel)
        duration = self.get_duration(waveform)

        T_mel = mel.shape[1]
        if f0.shape[0] != T_mel:
            f0 = F.interpolate(f0.unsqueeze(0).unsqueeze(0), size=T_mel, mode='linear', align_corners=False).squeeze()
            voiced = F.interpolate(voiced.unsqueeze(0).unsqueeze(0), size=T_mel, mode='nearest').squeeze()
        if energy.shape[0] != T_mel:
            energy = F.interpolate(energy.unsqueeze(0).unsqueeze(0), size=T_mel, mode='linear', align_corners=False).squeeze()

        return {
            "mel": mel,
            "f0": f0,
            "voiced": voiced,
            "energy": energy,
            "duration": duration,
            "T": T_mel,
        }

    def save_audio(self, waveform: torch.Tensor, path: str, sample_rate: Optional[int] = None):
        """Save waveform to WAV file."""
        sr = sample_rate or self.sr
        wav_np = waveform.squeeze().cpu().numpy()
        wav_int16 = np.clip(wav_np * 32767.0, -32768, 32767).astype(np.int16)
        wav_io.write(path, sr, wav_int16)
        logger.info(f"Saved audio -> {path}")

    def load_or_generate_audio(self, path: Optional[str] = None, duration_sec: float = 3.0) -> torch.Tensor:
        """Load audio or generate demo signal."""
        if path and os.path.exists(path):
            waveform, _ = self.load_audio(path)
            return waveform
        logger.info("No input file found, generating demo audio signal...")
        return self._generate_demo_signal(duration_sec)

    def _generate_demo_signal(self, duration_sec: float = 3.0) -> torch.Tensor:
        """
        Generate a realistic-sounding synthetic voice signal using
        glottal pulse model + formant filtering.
        """
        t = np.linspace(0, duration_sec, int(self.sr * duration_sec), endpoint=False)

        f0_hz = 120.0
        t_pitch = np.linspace(0, duration_sec, len(t))
        f0_contour = f0_hz + 8 * np.sin(2 * np.pi * 0.8 * t_pitch) + \
                     3 * np.sin(2 * np.pi * 3.0 * t_pitch)

        phase = np.cumsum(2 * np.pi * f0_contour / self.sr)
        glottal = np.zeros_like(t)
        for k in range(1, 12):
            amp = 1.0 / (k ** 1.2)
            glottal += amp * np.sin(k * phase)

        formant_freqs = [700, 1200, 2600, 3500]
        formant_bws = [130, 200, 350, 400]
        voiced_signal = np.zeros_like(glottal)
        for ff, bw in zip(formant_freqs, formant_bws):
            b, a = signal.butter(2, [(ff - bw / 2) / (self.sr / 2), (ff + bw / 2) / (self.sr / 2)], btype='band')
            voiced_signal += signal.lfilter(b, a, glottal)

        voiced_signal = voiced_signal / (np.max(np.abs(voiced_signal)) + 1e-8)

        silence_samples = int(0.1 * self.sr)
        speech_on_off = np.ones(len(t))
        speech_on_off[:silence_samples] = 0
        speech_on_off[-silence_samples:] = 0
        mid = len(t) // 2
        speech_on_off[mid: mid + silence_samples // 2] = 0

        amplitude_envelope = np.random.uniform(0.7, 1.0, len(t) // (self.sr // 10) + 1)
        amplitude_envelope = np.repeat(amplitude_envelope, self.sr // 10)[:len(t)]
        amplitude_envelope = np.convolve(amplitude_envelope, np.ones(self.sr // 20) / (self.sr // 20), mode='same')

        signal_out = voiced_signal * speech_on_off * amplitude_envelope
        noise = np.random.randn(len(t)) * 0.005
        signal_out = signal_out + noise
        signal_out = signal_out / (np.max(np.abs(signal_out)) + 1e-8) * 0.9

        return torch.from_numpy(signal_out.astype(np.float32)).unsqueeze(0)


# =============================================================================
# SECTION 3: VARIATIONAL AUTOENCODER (VAE)
# =============================================================================

class ResidualBlock1D(nn.Module):
    """1D Residual block with layer normalization."""

    def __init__(self, channels: int, kernel_size: int = 3, dilation: int = 1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, dilation=dilation, padding=padding)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, dilation=dilation, padding=padding)
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = x.transpose(1, 2)
        x = self.norm1(x)
        x = x.transpose(1, 2)
        x = self.act(self.conv1(x))
        x = x.transpose(1, 2)
        x = self.norm2(x)
        x = x.transpose(1, 2)
        x = self.conv2(x)
        return x + residual


class VAEEncoder(nn.Module):
    """
    Variational Autoencoder Encoder.
    Input: Mel spectrogram [B, n_mels, T]
    Output: mu, log_var [B, latent_dim]
    """

    def __init__(self, n_mels: int, latent_dim: int, hidden_dim: int, num_layers: int):
        super().__init__()
        self.n_mels = n_mels
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim

        self.input_proj = nn.Conv1d(n_mels, hidden_dim, kernel_size=1)

        self.res_blocks = nn.ModuleList([
            ResidualBlock1D(hidden_dim, kernel_size=3, dilation=2 ** i)
            for i in range(num_layers)
        ])

        self.conv_downsample = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.Conv1d(hidden_dim * 2, hidden_dim * 2, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.Conv1d(hidden_dim * 2, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

        self.dropout = nn.Dropout(0.1)

    def forward(self, mel: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.input_proj(mel)
        for block in self.res_blocks:
            x = block(x)
        x = self.dropout(x)
        x = self.conv_downsample(x)
        x = self.pool(x).squeeze(-1)
        mu = self.fc_mu(x)
        log_var = self.fc_log_var(x)
        log_var = torch.clamp(log_var, -10.0, 4.0)
        return mu, log_var


class VAEDecoder(nn.Module):
    """
    Variational Autoencoder Decoder.
    Input: latent z [B, latent_dim], target_len T
    Output: mel [B, n_mels, T]
    """

    def __init__(self, n_mels: int, latent_dim: int, hidden_dim: int, num_layers: int):
        super().__init__()
        self.n_mels = n_mels
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim

        self.fc = nn.Linear(latent_dim, hidden_dim * 8)

        self.upsample = nn.Sequential(
            nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
        )

        self.res_blocks = nn.ModuleList([
            ResidualBlock1D(hidden_dim, kernel_size=3, dilation=2 ** i)
            for i in range(num_layers)
        ])

        self.output_proj = nn.Conv1d(hidden_dim, n_mels, kernel_size=1)

    def forward(self, z: torch.Tensor, target_len: int) -> torch.Tensor:
        x = self.fc(z)
        B = z.shape[0]
        x = x.view(B, self.hidden_dim, 8)
        x = self.upsample(x)
        for block in self.res_blocks:
            x = block(x)
        x = self.output_proj(x)
        if x.shape[-1] != target_len:
            x = F.interpolate(x, size=target_len, mode='linear', align_corners=False)
        return x


class VoiceVAE(nn.Module):
    """
    Full Variational Autoencoder for voice feature encoding.
    Also encodes emotion-conditioning and disentangles speaker identity.
    """

    def __init__(self, config: ModelConfig, n_mels: int = 80):
        super().__init__()
        self.config = config
        self.n_mels = n_mels
        self.latent_dim = config.latent_dim

        self.encoder = VAEEncoder(
            n_mels=n_mels,
            latent_dim=config.latent_dim,
            hidden_dim=config.vae_hidden_dim,
            num_layers=config.vae_num_layers
        )

        self.decoder = VAEDecoder(
            n_mels=n_mels,
            latent_dim=config.latent_dim,
            hidden_dim=config.vae_hidden_dim,
            num_layers=config.vae_num_layers
        )

        self.emotion_embedding = nn.Embedding(config.num_emotions, config.latent_dim)
        nn.init.normal_(self.emotion_embedding.weight, 0.0, 0.1)

        self.emotion_mapper = nn.Sequential(
            nn.Linear(config.latent_dim * 2, config.latent_dim * 2),
            nn.GELU(),
            nn.Linear(config.latent_dim * 2, config.latent_dim),
            nn.Tanh()
        )

        self.pitch_predictor = nn.Sequential(
            nn.Linear(config.latent_dim, config.vae_hidden_dim),
            nn.GELU(),
            nn.Linear(config.vae_hidden_dim, 1)
        )
        self.energy_predictor = nn.Sequential(
            nn.Linear(config.latent_dim, config.vae_hidden_dim),
            nn.GELU(),
            nn.Linear(config.vae_hidden_dim, 1)
        )

    def encode(self, mel: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.encoder(mel)

    def reparameterize(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def decode(self, z: torch.Tensor, target_len: int) -> torch.Tensor:
        return self.decoder(z, target_len)

    def condition_on_emotion(self, z: torch.Tensor, emotion_id: torch.Tensor) -> torch.Tensor:
        """Shift latent code towards target emotion."""
        emotion_emb = self.emotion_embedding(emotion_id)
        combined = torch.cat([z, emotion_emb], dim=-1)
        z_emotion = self.emotion_mapper(combined)
        return z_emotion

    def predict_pitch(self, z: torch.Tensor) -> torch.Tensor:
        return self.pitch_predictor(z).squeeze(-1)

    def predict_energy(self, z: torch.Tensor) -> torch.Tensor:
        return self.energy_predictor(z).squeeze(-1)

    def forward(
        self,
        mel: torch.Tensor,
        emotion_id: Optional[torch.Tensor] = None
    ) -> Dict[str, torch.Tensor]:
        B, n_mels, T = mel.shape
        mu, log_var = self.encode(mel)
        z = self.reparameterize(mu, log_var)

        if emotion_id is not None:
            z_conditioned = self.condition_on_emotion(z, emotion_id)
        else:
            z_conditioned = z

        mel_recon = self.decode(z_conditioned, T)
        pitch_pred = self.predict_pitch(z_conditioned)
        energy_pred = self.predict_energy(z_conditioned)

        return {
            "mel_recon": mel_recon,
            "mu": mu,
            "log_var": log_var,
            "z": z,
            "z_conditioned": z_conditioned,
            "pitch_pred": pitch_pred,
            "energy_pred": energy_pred,
        }

    def get_kl_loss(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        kl = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
        return kl


# =============================================================================
# SECTION 4: GRAPH NEURAL NETWORK (GNN) FOR EMOTION MAPPING
# =============================================================================

class GraphAttentionLayer(nn.Module):
    """
    Multi-head Graph Attention Layer (GAT).
    Computes attention over graph edges to propagate emotion features.
    """

    def __init__(self, in_features: int, out_features: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.num_heads = num_heads
        self.head_dim = out_features // num_heads

        self.W_q = nn.Linear(in_features, out_features)
        self.W_k = nn.Linear(in_features, out_features)
        self.W_v = nn.Linear(in_features, out_features)
        self.W_o = nn.Linear(out_features, out_features)

        self.attn_weight = nn.Parameter(torch.randn(num_heads, self.head_dim * 2))
        nn.init.xavier_uniform_(self.attn_weight.unsqueeze(0))

        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(out_features)
        self.scale = math.sqrt(self.head_dim)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_weight: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        x: [N, in_features]
        edge_index: [2, E]  (source, target)
        edge_weight: [E] optional
        """
        N = x.shape[0]
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(N, self.num_heads, self.head_dim)
        K = K.view(N, self.num_heads, self.head_dim)
        V = V.view(N, self.num_heads, self.head_dim)

        src_idx = edge_index[0]
        tgt_idx = edge_index[1]

        q_src = Q[src_idx]
        k_tgt = K[tgt_idx]
        v_tgt = V[tgt_idx]

        attn_input = torch.cat([q_src, k_tgt], dim=-1)
        e = (attn_input * self.attn_weight.unsqueeze(0)).sum(dim=-1)
        e = self.leaky_relu(e)

        if edge_weight is not None:
            e = e * edge_weight.unsqueeze(-1)

        attn = torch.zeros(N, self.num_heads, device=x.device)
        exp_e = torch.exp(e - e.max())
        attn.index_add_(0, tgt_idx, exp_e)
        attn_norm = attn[tgt_idx] + 1e-8
        alpha = exp_e / attn_norm
        alpha = self.dropout(alpha)

        weighted_v = v_tgt * alpha.unsqueeze(-1)
        out = torch.zeros(N, self.num_heads, self.head_dim, device=x.device)
        out.index_add_(0, tgt_idx, weighted_v)
        out = out.view(N, -1)
        out = self.W_o(out)
        out = self.norm(out + x if out.shape == x.shape else out)
        return out


class EmotionGraphNode:
    """
    Represents a node in the emotion graph.
    Each emotion is a node; voice features are node attributes.
    """

    def __init__(
        self,
        emotion_name: str,
        emotion_id: int,
        feature_vector: np.ndarray
    ):
        self.emotion_name = emotion_name
        self.emotion_id = emotion_id
        self.feature_vector = feature_vector


class EmotionGNN(nn.Module):
    """
    Graph Neural Network for emotion-to-feature mapping.

    Architecture:
    - Nodes: Each emotion is a node with feature attributes
    - Edges: Connections between related emotions (valence/arousal proximity)
    - Message passing learns how to transform voice features across emotions
    - Output: Transformed feature vector for target emotion

    The graph captures emotional relationships:
    - happy <-> surprised (both high arousal, positive valence)
    - sad <-> fearful (both low valence)
    - angry <-> disgusted (both negative valence, different arousal)
    - neutral <-> all (neutral is the anchor)
    """

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.num_emotions = config.num_emotions
        self.hidden_dim = config.gnn_hidden_dim
        self.latent_dim = config.latent_dim

        # Emotion node feature dimension: latent_dim + prosody_features (6)
        self.node_feat_dim = config.latent_dim + 6

        self.node_encoder = nn.Sequential(
            nn.Linear(self.node_feat_dim, self.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(self.hidden_dim, self.hidden_dim),
        )

        self.gat_layers = nn.ModuleList([
            GraphAttentionLayer(
                in_features=self.hidden_dim,
                out_features=self.hidden_dim,
                num_heads=config.gnn_num_heads,
                dropout=config.dropout
            )
            for _ in range(config.gnn_num_layers)
        ])

        self.ffn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.hidden_dim, self.hidden_dim * 2),
                nn.GELU(),
                nn.Dropout(config.dropout),
                nn.Linear(self.hidden_dim * 2, self.hidden_dim),
            )
            for _ in range(config.gnn_num_layers)
        ])

        self.norm_layers = nn.ModuleList([
            nn.LayerNorm(self.hidden_dim)
            for _ in range(config.gnn_num_layers)
        ])

        self.output_proj = nn.Sequential(
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.GELU(),
            nn.Linear(self.hidden_dim, config.latent_dim),
            nn.Tanh()
        )

        self.prosody_decoder = nn.Sequential(
            nn.Linear(config.latent_dim, self.hidden_dim),
            nn.GELU(),
            nn.Linear(self.hidden_dim, 6),
        )

        self.edge_index, self.edge_weight = self._build_emotion_graph()

        self.emotion_base_nodes = nn.Parameter(
            torch.randn(config.num_emotions, self.node_feat_dim) * 0.1
        )

    def _build_emotion_graph(self) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Build emotion graph based on valence-arousal space.
        Emotions closer in VA space have stronger edges.

        Valence-Arousal positions (approximate):
          neutral:   V=0.5, A=0.5
          happy:     V=0.9, A=0.7
          sad:       V=0.1, A=0.2
          angry:     V=0.1, A=0.9
          fearful:   V=0.2, A=0.8
          surprised: V=0.6, A=0.9
          disgusted: V=0.1, A=0.6
        """
        va_positions = {
            "neutral":   (0.5, 0.5),
            "happy":     (0.9, 0.7),
            "sad":       (0.1, 0.2),
            "angry":     (0.1, 0.9),
            "fearful":   (0.2, 0.8),
            "surprised": (0.6, 0.9),
            "disgusted": (0.1, 0.6),
        }

        emotions = list(EMOTION_PROFILES.keys())
        n = len(emotions)
        edges_src = []
        edges_tgt = []
        weights = []

        for i, e1 in enumerate(emotions):
            for j, e2 in enumerate(emotions):
                if i != j:
                    v1, a1 = va_positions[e1]
                    v2, a2 = va_positions[e2]
                    dist = math.sqrt((v1 - v2) ** 2 + (a1 - a2) ** 2)
                    max_dist = math.sqrt(2.0)
                    similarity = 1.0 - dist / max_dist
                    if similarity > 0.3 or e1 == "neutral" or e2 == "neutral":
                        edges_src.append(i)
                        edges_tgt.append(j)
                        weights.append(similarity)

        edge_index = torch.tensor([edges_src, edges_tgt], dtype=torch.long)
        edge_weight = torch.tensor(weights, dtype=torch.float32)
        return edge_index, edge_weight

    def forward(
        self,
        z_source: torch.Tensor,
        source_emotion_id: torch.Tensor,
        target_emotion_id: torch.Tensor,
        source_prosody: Optional[torch.Tensor] = None
    ) -> Dict[str, torch.Tensor]:
        """
        z_source: [B, latent_dim] — source voice latent
        source_emotion_id: [B] — source emotion index
        target_emotion_id: [B] — target emotion index
        source_prosody: [B, 6] — [pitch_mean, pitch_std, energy_mean, energy_std, speed, duration]
        """
        B = z_source.shape[0]
        device = z_source.device

        edge_index = self.edge_index.to(device)
        edge_weight = self.edge_weight.to(device)

        if source_prosody is None:
            source_prosody = torch.zeros(B, 6, device=device)

        results = []
        for b in range(B):
            z_b = z_source[b].unsqueeze(0)
            src_id = source_emotion_id[b].item()
            tgt_id = target_emotion_id[b].item()
            prosody_b = source_prosody[b].unsqueeze(0)

            node_feats = self.emotion_base_nodes.clone()
            source_feat = torch.cat([z_b, prosody_b], dim=-1)
            node_feats[src_id] = node_feats[src_id] + source_feat.squeeze(0)

            x = self.node_encoder(node_feats)

            for layer_idx in range(len(self.gat_layers)):
                x_attn = self.gat_layers[layer_idx](x, edge_index, edge_weight)
                x_ffn = self.ffn_layers[layer_idx](x_attn)
                x = self.norm_layers[layer_idx](x_ffn + x)

            target_node = x[tgt_id].unsqueeze(0)
            z_target = self.output_proj(target_node)
            results.append(z_target)

        z_out = torch.cat(results, dim=0)
        prosody_out = self.prosody_decoder(z_out)

        return {
            "z_target": z_out,
            "prosody_target": prosody_out,
        }

    def get_emotion_embeddings(self) -> Dict[str, torch.Tensor]:
        """Return current learned emotion node representations."""
        embeddings = {}
        for i, name in enumerate(EMOTION_PROFILES.keys()):
            embeddings[name] = self.emotion_base_nodes[i].detach()
        return embeddings


# =============================================================================
# SECTION 5: FASTSPEECH 2 VARIANCE ADAPTOR
# =============================================================================

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)


class FFTBlock(nn.Module):
    """
    Feed-Forward Transformer Block used in FastSpeech2.
    Multi-head self-attention + position-wise FFN with Conv1d.
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        kernel_size: int = 9,
        dropout: float = 0.1
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        padding = (kernel_size - 1) // 2
        self.ffn = nn.Sequential(
            nn.Conv1d(d_model, d_ff, kernel_size=kernel_size, padding=padding),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(d_ff, d_model, kernel_size=kernel_size, padding=padding),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        attn_out, _ = self.self_attn(x, x, x, key_padding_mask=mask)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_in = x.transpose(1, 2)
        ffn_out = self.ffn(ffn_in).transpose(1, 2)
        x = self.norm2(x + self.dropout(ffn_out))
        return x


class VariancePredictor(nn.Module):
    """
    FastSpeech 2 Variance Predictor.
    Predicts frame-level pitch / energy / duration from hidden features.
    """

    def __init__(self, in_channels: int, out_channels: int = 1, kernel_size: int = 3):
        super().__init__()
        padding = (kernel_size - 1) // 2
        self.layers = nn.Sequential(
            nn.Conv1d(in_channels, in_channels, kernel_size=kernel_size, padding=padding),
            nn.GELU(),
            nn.LayerNorm(in_channels),
            nn.Conv1d(in_channels, in_channels, kernel_size=kernel_size, padding=padding),
            nn.GELU(),
            nn.LayerNorm(in_channels),
            nn.Linear(in_channels, out_channels),
        )
        self._init_layers()

    def _init_layers(self):
        for layer in self.layers:
            if isinstance(layer, (nn.Conv1d, nn.Linear)):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, T, C] -> out: [B, T, out_channels]"""
        out = x
        for layer in self.layers:
            if isinstance(layer, nn.Conv1d):
                out = layer(out.transpose(1, 2)).transpose(1, 2)
            elif isinstance(layer, nn.LayerNorm):
                out = layer(out)
            else:
                out = layer(out)
        return out


class LengthRegulator(nn.Module):
    """
    FastSpeech 2 Length Regulator.
    Expands/contracts sequence according to predicted durations.
    """

    def __init__(self):
        super().__init__()

    def forward(
        self,
        x: torch.Tensor,
        duration: torch.Tensor,
        target_len: Optional[int] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        x: [B, T_phone, C]
        duration: [B, T_phone] integer durations
        Returns regulated x: [B, T_mel, C]
        """
        output = []
        out_lens = []
        for b in range(x.shape[0]):
            dur = duration[b].clamp(min=0).long()
            if dur.sum() == 0:
                dur = dur + 1
            regulated = torch.repeat_interleave(x[b], dur, dim=0)
            output.append(regulated)
            out_lens.append(regulated.shape[0])

        max_len = max(out_lens) if target_len is None else target_len
        padded = torch.zeros(x.shape[0], max_len, x.shape[-1], device=x.device)
        for b, (out, length) in enumerate(zip(output, out_lens)):
            actual_len = min(length, max_len)
            padded[b, :actual_len] = out[:actual_len]

        out_lens_tensor = torch.tensor(out_lens, dtype=torch.long, device=x.device)
        return padded, out_lens_tensor


class PitchEmbedding(nn.Module):
    """Quantized pitch embedding with continuous interpolation."""

    def __init__(self, num_bins: int, d_model: int, f0_min: float = -3.0, f0_max: float = 3.0):
        super().__init__()
        self.num_bins = num_bins
        self.f0_min = f0_min
        self.f0_max = f0_max
        self.embedding = nn.Embedding(num_bins + 1, d_model)
        self.register_buffer('bins', torch.linspace(f0_min, f0_max, num_bins))

    def forward(self, f0: torch.Tensor) -> torch.Tensor:
        bins = self.bins.to(f0.device)
        f0_clamp = f0.clamp(self.f0_min, self.f0_max)
        indices = torch.bucketize(f0_clamp, bins).clamp(0, self.num_bins)
        return self.embedding(indices)


class EnergyEmbedding(nn.Module):
    """Quantized energy embedding."""

    def __init__(self, num_bins: int, d_model: int, e_min: float = 0.0, e_max: float = 150.0):
        super().__init__()
        self.num_bins = num_bins
        self.e_min = e_min
        self.e_max = e_max
        self.embedding = nn.Embedding(num_bins + 1, d_model)
        self.register_buffer('bins', torch.linspace(e_min, e_max, num_bins))

    def forward(self, energy: torch.Tensor) -> torch.Tensor:
        bins = self.bins.to(energy.device)
        e_clamp = energy.clamp(self.e_min, self.e_max)
        indices = torch.bucketize(e_clamp, bins).clamp(0, self.num_bins)
        return self.embedding(indices)


class FastSpeech2VarianceAdaptor(nn.Module):
    """
    FastSpeech 2 Variance Adaptor:
    - Duration Predictor -> Length Regulator (controls speech rate)
    - Pitch Predictor + Embedding (controls intonation)
    - Energy Predictor + Embedding (controls loudness/intensity)
    - Emotion conditioning via GNN output
    """

    def __init__(self, config: ModelConfig, n_mels: int = 80):
        super().__init__()
        self.config = config
        d_model = config.fastspeech_hidden
        self.n_mels = n_mels

        self.input_proj = nn.Linear(config.latent_dim, d_model)

        self.phoneme_encoder = nn.ModuleList([
            FFTBlock(
                d_model=d_model,
                num_heads=config.fastspeech_num_heads,
                d_ff=config.fastspeech_filter_size,
                kernel_size=config.fastspeech_kernel_size,
                dropout=config.dropout
            )
            for _ in range(config.fastspeech_num_layers)
        ])

        self.duration_predictor = VariancePredictor(d_model, out_channels=1)
        self.length_regulator = LengthRegulator()
        self.pitch_predictor = VariancePredictor(d_model, out_channels=1)
        self.energy_predictor = VariancePredictor(d_model, out_channels=1)

        self.pitch_embedding = PitchEmbedding(config.pitch_bins, d_model)
        self.energy_embedding = EnergyEmbedding(config.energy_bins, d_model)

        self.mel_encoder = nn.ModuleList([
            FFTBlock(
                d_model=d_model,
                num_heads=config.fastspeech_num_heads,
                d_ff=config.fastspeech_filter_size,
                kernel_size=config.fastspeech_kernel_size,
                dropout=config.dropout
            )
            for _ in range(config.fastspeech_num_layers)
        ])

        self.mel_proj = nn.Linear(d_model, n_mels)

        self.emotion_proj = nn.Linear(config.latent_dim, d_model)
        self.emotion_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )

        self.pos_enc = PositionalEncoding(d_model, dropout=config.dropout)

    def forward(
        self,
        z: torch.Tensor,
        emotion_z: torch.Tensor,
        target_mel_len: int,
        target_f0: Optional[torch.Tensor] = None,
        target_energy: Optional[torch.Tensor] = None,
        target_duration: Optional[torch.Tensor] = None,
        emotion_profile: Optional[EmotionProfile] = None,
    ) -> Dict[str, torch.Tensor]:
        """
        z: [B, latent_dim] — source content latent
        emotion_z: [B, latent_dim] — emotion-conditioned latent from GNN
        target_mel_len: int — desired output mel length
        """
        B = z.shape[0]
        device = z.device

        h = self.input_proj(z).unsqueeze(1)
        h = h.expand(-1, max(target_mel_len // 8, 4), -1)

        h = self.pos_enc(h.transpose(0, 1)).transpose(0, 1)

        for block in self.phoneme_encoder:
            h = block(h)

        emotion_h = self.emotion_proj(emotion_z).unsqueeze(1).expand_as(h)
        gate = self.emotion_gate(torch.cat([h, emotion_h], dim=-1))
        h = h * gate + emotion_h * (1 - gate)

        dur_pred = self.duration_predictor(h).squeeze(-1)
        dur_pred = F.softplus(dur_pred) + 1.0

        if emotion_profile is not None:
            dur_pred = dur_pred * emotion_profile.duration_factor

        if target_duration is not None:
            dur_for_reg = target_duration.unsqueeze(0).expand(B, -1) if target_duration.dim() == 1 else target_duration
            dur_int = dur_for_reg.clamp(min=1)
        else:
            total_dur = dur_pred.sum(dim=-1, keepdim=True)
            scale = target_mel_len / (total_dur + 1e-8)
            dur_int = (dur_pred * scale).round().clamp(min=1).long()

        h_expanded, _ = self.length_regulator(h, dur_int, target_mel_len)

        pitch_pred = self.pitch_predictor(h_expanded).squeeze(-1)
        energy_pred = self.energy_predictor(h_expanded).squeeze(-1)

        if emotion_profile is not None:
            pitch_pred = pitch_pred + emotion_profile.pitch_shift * 0.1
            pitch_std = pitch_pred.std(dim=-1, keepdim=True)
            pitch_mean = pitch_pred.mean(dim=-1, keepdim=True)
            pitch_pred = pitch_mean + (pitch_pred - pitch_mean) * (
                1.0 + emotion_profile.pitch_variance * 2.0
            )
            energy_pred = energy_pred * emotion_profile.energy_scale
            energy_std = energy_pred.std(dim=-1, keepdim=True)
            energy_pred = energy_pred + energy_std * emotion_profile.energy_variance

        if target_f0 is not None:
            f0_for_emb = target_f0
        else:
            f0_for_emb = pitch_pred

        if target_energy is not None:
            e_for_emb = target_energy
        else:
            e_for_emb = energy_pred

        pitch_emb = self.pitch_embedding(f0_for_emb)
        energy_emb = self.energy_embedding(e_for_emb.clamp(0, 150))

        h_expanded = h_expanded + pitch_emb + energy_emb

        h_expanded = self.pos_enc(h_expanded.transpose(0, 1)).transpose(0, 1)
        for block in self.mel_encoder:
            h_expanded = block(h_expanded)

        mel_out = self.mel_proj(h_expanded)
        mel_out = mel_out.transpose(1, 2)

        if mel_out.shape[-1] != target_mel_len:
            mel_out = F.interpolate(mel_out, size=target_mel_len, mode='linear', align_corners=False)

        return {
            "mel": mel_out,
            "pitch_pred": pitch_pred,
            "energy_pred": energy_pred,
            "duration_pred": dur_pred,
        }


# =============================================================================
# SECTION 6: HIFI-GAN VOCODER
# =============================================================================

class ResBlock(nn.Module):
    """HiFi-GAN residual block with dilated convolutions for natural-sounding waveform."""

    def __init__(self, channels: int, kernel_size: int = 3, dilations: List[int] = (1, 3, 5)):
        super().__init__()
        self.convs1 = nn.ModuleList([
            nn.Sequential(
                nn.LeakyReLU(0.1),
                nn.utils.weight_norm(
                    nn.Conv1d(channels, channels, kernel_size, dilation=d,
                              padding=self._get_padding(kernel_size, d))
                )
            )
            for d in dilations
        ])
        self.convs2 = nn.ModuleList([
            nn.Sequential(
                nn.LeakyReLU(0.1),
                nn.utils.weight_norm(
                    nn.Conv1d(channels, channels, kernel_size, dilation=1,
                              padding=self._get_padding(kernel_size, 1))
                )
            )
            for _ in dilations
        ])

    @staticmethod
    def _get_padding(k: int, d: int) -> int:
        return (k * d - d) // 2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for c1, c2 in zip(self.convs1, self.convs2):
            xt = c1(x)
            xt = c2(xt)
            x = xt + x
        return x

    def remove_weight_norm(self):
        for layer in self.convs1:
            for sublayer in layer:
                if hasattr(sublayer, 'weight'):
                    try:
                        nn.utils.remove_weight_norm(sublayer)
                    except ValueError:
                        pass
        for layer in self.convs2:
            for sublayer in layer:
                if hasattr(sublayer, 'weight'):
                    try:
                        nn.utils.remove_weight_norm(sublayer)
                    except ValueError:
                        pass


class HiFiGANGenerator(nn.Module):
    """
    HiFi-GAN Generator (V1).
    Converts mel spectrogram to natural-sounding waveform via
    transposed convolutions and multi-receptive field fusion (MRF).

    Key property: multi-scale dilated residual blocks ensure
    the output captures both fine-grained and coarse speech features,
    resulting in natural-sounding audio (not robotic/AI-sounding).
    """

    def __init__(self, config: ModelConfig, n_mels: int = 80):
        super().__init__()
        self.config = config
        self.n_mels = n_mels

        self.pre_conv = nn.utils.weight_norm(
            nn.Conv1d(n_mels, config.hifigan_upsample_initial_channel, 7, 1, padding=3)
        )

        self.ups = nn.ModuleList()
        self.resblocks = nn.ModuleList()

        in_channels = config.hifigan_upsample_initial_channel
        for i, (u, k) in enumerate(zip(config.hifigan_upsample_rates, config.hifigan_upsample_kernel_sizes)):
            out_channels = in_channels // 2
            self.ups.append(
                nn.utils.weight_norm(
                    nn.ConvTranspose1d(in_channels, out_channels, k, u, padding=(k - u) // 2)
                )
            )
            for j in range(len(config.hifigan_resblock_kernel_sizes)):
                self.resblocks.append(
                    ResBlock(
                        out_channels,
                        config.hifigan_resblock_kernel_sizes[j],
                        config.hifigan_resblock_dilation_sizes[j]
                    )
                )
            in_channels = out_channels

        self.post_conv = nn.utils.weight_norm(
            nn.Conv1d(in_channels, 1, 7, 1, padding=3)
        )

        self.num_kernels = len(config.hifigan_resblock_kernel_sizes)
        self.num_upsamples = len(config.hifigan_upsample_rates)

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        """mel: [B, n_mels, T] -> waveform: [B, 1, T*hop_length]"""
        x = self.pre_conv(mel)
        for i in range(self.num_upsamples):
            x = F.leaky_relu(x, 0.1)
            x = self.ups[i](x)
            xs = None
            for j in range(self.num_kernels):
                if xs is None:
                    xs = self.resblocks[i * self.num_kernels + j](x)
                else:
                    xs += self.resblocks[i * self.num_kernels + j](x)
            x = xs / self.num_kernels
        x = F.leaky_relu(x)
        x = self.post_conv(x)
        x = torch.tanh(x)
        return x

    def remove_weight_norm(self):
        try:
            nn.utils.remove_weight_norm(self.pre_conv)
        except ValueError:
            pass
        for up in self.ups:
            try:
                nn.utils.remove_weight_norm(up)
            except ValueError:
                pass
        for block in self.resblocks:
            block.remove_weight_norm()
        try:
            nn.utils.remove_weight_norm(self.post_conv)
        except ValueError:
            pass


class HiFiGANMultiScaleDiscriminator(nn.Module):
    """Multi-Scale Discriminator for HiFi-GAN training."""

    def __init__(self):
        super().__init__()
        self.discriminators = nn.ModuleList([
            self._make_discriminator(),
            self._make_discriminator(),
            self._make_discriminator(),
        ])
        self.pooling = nn.ModuleList([
            nn.Identity(),
            nn.AvgPool1d(4, 2, padding=2),
            nn.AvgPool1d(4, 2, padding=2),
        ])

    def _make_discriminator(self):
        return nn.Sequential(
            nn.utils.weight_norm(nn.Conv1d(1, 128, 15, 1, padding=7)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(128, 128, 41, 4, groups=4, padding=20)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(128, 256, 41, 4, groups=16, padding=20)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(256, 512, 41, 4, groups=16, padding=20)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(512, 1024, 41, 4, groups=16, padding=20)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(1024, 1024, 5, 1, padding=2)),
            nn.LeakyReLU(0.1),
            nn.utils.weight_norm(nn.Conv1d(1024, 1, 3, 1, padding=1)),
        )

    def forward(self, audio: torch.Tensor) -> List[torch.Tensor]:
        outputs = []
        x = audio
        for disc, pool in zip(self.discriminators, self.pooling):
            x = pool(x)
            y = disc(x)
            outputs.append(y)
        return outputs


class HiFiGANMultiPeriodDiscriminator(nn.Module):
    """Multi-Period Discriminator for HiFi-GAN training."""

    def __init__(self):
        super().__init__()
        self.periods = [2, 3, 5, 7, 11]
        self.discriminators = nn.ModuleList([
            self._make_period_discriminator(p) for p in self.periods
        ])

    def _make_period_discriminator(self, period: int):
        channels = [1, 32, 128, 512, 1024, 1024]
        layers = []
        for i in range(len(channels) - 1):
            k = (5, 1) if i == 0 else (5, 1)
            s = (3, 1) if i < 3 else (1, 1)
            p = (2, 0)
            layers.append(nn.utils.weight_norm(nn.Conv2d(channels[i], channels[i + 1], k, s, padding=p)))
            layers.append(nn.LeakyReLU(0.1))
        layers.append(nn.utils.weight_norm(nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0))))
        return nn.Sequential(*layers)

    def forward(self, audio: torch.Tensor) -> List[torch.Tensor]:
        outputs = []
        for period, disc in zip(self.periods, self.discriminators):
            B, C, T = audio.shape
            pad_size = (period - (T % period)) % period
            x = F.pad(audio, (0, pad_size))
            x = x.view(B, C, -1, period)
            y = disc(x)
            outputs.append(y)
        return outputs


# =============================================================================
# SECTION 7: COMPLETE SPEECH CONVERSION MODEL
# =============================================================================

class EmotionalSpeechConverter(nn.Module):
    """
    Full end-to-end Emotional Speech Conversion System.

    Pipeline:
    1. VAE: audio -> latent z (encodes speaker identity + content)
    2. GNN: (z, source_emotion, target_emotion) -> z_target (maps to emotion)
    3. FastSpeech2: z_target -> mel spectrogram (with pitch/energy/duration control)
    4. HiFi-GAN: mel -> waveform (natural audio)
    """

    def __init__(self, config: ModelConfig, audio_config: AudioConfig):
        super().__init__()
        self.model_config = config
        self.audio_config = audio_config
        n_mels = audio_config.n_mels

        self.vae = VoiceVAE(config, n_mels=n_mels)
        self.gnn = EmotionGNN(config)
        self.variance_adaptor = FastSpeech2VarianceAdaptor(config, n_mels=n_mels)
        self.vocoder = HiFiGANGenerator(config, n_mels=n_mels)

    def forward(
        self,
        mel: torch.Tensor,
        source_emotion_id: torch.Tensor,
        target_emotion_id: torch.Tensor,
        source_prosody: Optional[torch.Tensor] = None,
        target_mel_len: Optional[int] = None,
        emotion_profile: Optional[EmotionProfile] = None,
    ) -> Dict[str, torch.Tensor]:

        B, n_mels, T = mel.shape
        if target_mel_len is None:
            target_mel_len = T

        vae_out = self.vae(mel, emotion_id=source_emotion_id)
        z_source = vae_out["z"]
        mu = vae_out["mu"]
        log_var = vae_out["log_var"]
        mel_recon = vae_out["mel_recon"]

        gnn_out = self.gnn(
            z_source=z_source,
            source_emotion_id=source_emotion_id,
            target_emotion_id=target_emotion_id,
            source_prosody=source_prosody
        )
        z_target = gnn_out["z_target"]
        prosody_target = gnn_out["prosody_target"]

        va_out = self.variance_adaptor(
            z=z_source,
            emotion_z=z_target,
            target_mel_len=target_mel_len,
            emotion_profile=emotion_profile,
        )
        mel_converted = va_out["mel"]

        waveform = self.vocoder(mel_converted)

        return {
            "waveform": waveform,
            "mel_converted": mel_converted,
            "mel_recon": mel_recon,
            "z_source": z_source,
            "z_target": z_target,
            "mu": mu,
            "log_var": log_var,
            "pitch_pred": va_out["pitch_pred"],
            "energy_pred": va_out["energy_pred"],
            "duration_pred": va_out["duration_pred"],
            "prosody_target": prosody_target,
        }

    def convert_emotion(
        self,
        mel: torch.Tensor,
        source_emotion: str,
        target_emotion: str,
        target_mel_len: Optional[int] = None,
    ) -> Dict[str, torch.Tensor]:
        """Convert mel spectrogram from source to target emotion."""
        device = next(self.parameters()).device
        B = mel.shape[0] if mel.dim() == 3 else 1
        mel = mel.unsqueeze(0) if mel.dim() == 2 else mel
        mel = mel.to(device)

        src_id = torch.tensor([EMOTION_IDX[source_emotion]] * B, dtype=torch.long, device=device)
        tgt_id = torch.tensor([EMOTION_IDX[target_emotion]] * B, dtype=torch.long, device=device)
        profile = EMOTION_PROFILES[target_emotion]

        with torch.no_grad():
            out = self.forward(
                mel=mel,
                source_emotion_id=src_id,
                target_emotion_id=tgt_id,
                emotion_profile=profile,
                target_mel_len=target_mel_len or mel.shape[-1]
            )
        return out

    def get_latent(self, mel: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Extract VAE latent code from mel spectrogram."""
        mel = mel.unsqueeze(0) if mel.dim() == 2 else mel
        with torch.no_grad():
            mu, log_var = self.vae.encode(mel)
        return mu, log_var

    def count_parameters(self) -> Dict[str, int]:
        """Count trainable parameters per module."""
        counts = {}
        for name, module in [
            ("vae", self.vae),
            ("gnn", self.gnn),
            ("variance_adaptor", self.variance_adaptor),
            ("vocoder", self.vocoder),
        ]:
            counts[name] = sum(p.numel() for p in module.parameters() if p.requires_grad)
        counts["total"] = sum(counts.values())
        return counts


# =============================================================================
# SECTION 8: LOSS FUNCTIONS
# =============================================================================

class EmotionalSpeechLoss(nn.Module):
    """
    Combined loss function for the emotional speech conversion system.
    - Mel reconstruction loss (L1)
    - KL divergence for VAE
    - Pitch prediction loss
    - Energy prediction loss
    - Duration prediction loss
    - Feature matching loss (for GAN stability)
    - Emotion consistency loss
    """

    def __init__(self, train_config: TrainConfig):
        super().__init__()
        self.config = train_config
        self.l1 = nn.L1Loss()
        self.mse = nn.MSELoss()
        self.bce = nn.BCEWithLogitsLoss()
        self._step = 0

    def compute_kl_weight(self) -> float:
        """Anneal KL weight from 0 to beta_kl_max over warmup steps."""
        if self._step >= self.config.beta_kl_anneal_steps:
            return self.config.beta_kl_max
        frac = self._step / self.config.beta_kl_anneal_steps
        weight = self.config.beta_kl_max * (1 - math.cos(math.pi * frac)) / 2.0
        return max(self.config.beta_kl, weight)

    def generator_loss(
        self,
        disc_outputs: List[torch.Tensor]
    ) -> torch.Tensor:
        loss = torch.tensor(0.0)
        for d_out in disc_outputs:
            target = torch.ones_like(d_out)
            loss = loss + F.mse_loss(d_out, target)
        return loss / len(disc_outputs)

    def discriminator_loss(
        self,
        disc_real: List[torch.Tensor],
        disc_fake: List[torch.Tensor]
    ) -> torch.Tensor:
        loss = torch.tensor(0.0)
        for d_real, d_fake in zip(disc_real, disc_fake):
            real_target = torch.ones_like(d_real)
            fake_target = torch.zeros_like(d_fake)
            loss = loss + F.mse_loss(d_real, real_target) + F.mse_loss(d_fake, fake_target)
        return loss / len(disc_real)

    def forward(
        self,
        outputs: Dict[str, torch.Tensor],
        targets: Dict[str, torch.Tensor],
        beta_kl: Optional[float] = None,
    ) -> Dict[str, torch.Tensor]:
        self._step += 1
        losses = {}

        if "mel_recon" in outputs and "mel" in targets:
            losses["mel_recon"] = self.l1(outputs["mel_recon"], targets["mel"]) * self.config.lambda_mel

        if "mel_converted" in outputs and "mel" in targets:
            tgt_mel = targets["mel"]
            pred_mel = outputs["mel_converted"]
            if pred_mel.shape != tgt_mel.shape:
                pred_mel = F.interpolate(pred_mel, size=tgt_mel.shape[-1], mode='linear', align_corners=False)
            losses["mel_converted"] = self.l1(pred_mel, tgt_mel) * self.config.lambda_mel * 0.5

        if "mu" in outputs and "log_var" in outputs:
            kl_w = beta_kl if beta_kl is not None else self.compute_kl_weight()
            kl = -0.5 * torch.mean(1 + outputs["log_var"] - outputs["mu"].pow(2) - outputs["log_var"].exp())
            losses["kl"] = kl * kl_w

        if "pitch_pred" in outputs and "f0" in targets:
            f0_tgt = targets["f0"]
            f0_pred = outputs["pitch_pred"]
            if f0_pred.shape != f0_tgt.shape:
                f0_pred = F.interpolate(f0_pred.unsqueeze(1), size=f0_tgt.shape[-1], mode='linear', align_corners=False).squeeze(1)
            voiced = targets.get("voiced", torch.ones_like(f0_tgt))
            voiced_mask = voiced > 0.5
            if voiced_mask.sum() > 0:
                losses["pitch"] = self.mse(f0_pred[voiced_mask], f0_tgt[voiced_mask]) * self.config.lambda_pitch

        if "energy_pred" in outputs and "energy" in targets:
            e_tgt = targets["energy"]
            e_pred = outputs["energy_pred"]
            if e_pred.shape != e_tgt.shape:
                e_pred = F.interpolate(e_pred.unsqueeze(1), size=e_tgt.shape[-1], mode='linear', align_corners=False).squeeze(1)
            losses["energy"] = self.mse(e_pred, e_tgt) * self.config.lambda_energy

        if "duration_pred" in outputs and "duration" in targets:
            dur_tgt = targets["duration"].float()
            dur_pred = outputs["duration_pred"]
            min_len = min(dur_pred.shape[-1], dur_tgt.shape[-1])
            losses["duration"] = self.mse(dur_pred[..., :min_len], dur_tgt[..., :min_len]) * self.config.lambda_duration

        losses["total"] = sum(losses.values())
        return losses


# =============================================================================
# SECTION 9: DATASET
# =============================================================================

class SpeechDataset(Dataset):
    """
    Dataset for speech emotion conversion training.
    Supports loading from directory structure:
      data/
        neutral/  *.wav
        happy/    *.wav
        sad/      *.wav
        ...

    Also supports generating synthetic demo data for testing.
    """

    def __init__(
        self,
        data_dir: Optional[str],
        audio_processor: AudioProcessor,
        max_wav_len: int = 16384,
        max_mel_len: int = 512,
        use_synthetic: bool = False,
        synthetic_samples: int = 64,
    ):
        self.processor = audio_processor
        self.max_wav_len = max_wav_len
        self.max_mel_len = max_mel_len
        self.samples = []

        if use_synthetic or data_dir is None or not os.path.exists(data_dir or ""):
            logger.info("Using synthetic dataset for training demo")
            self._create_synthetic_dataset(synthetic_samples)
        else:
            self._load_from_directory(data_dir)

    def _load_from_directory(self, data_dir: str):
        for emotion_name in EMOTION_PROFILES.keys():
            emotion_dir = os.path.join(data_dir, emotion_name)
            if not os.path.exists(emotion_dir):
                continue
            for fname in os.listdir(emotion_dir):
                if fname.lower().endswith(('.wav', '.mp3', '.flac', '.ogg')):
                    self.samples.append({
                        "path": os.path.join(emotion_dir, fname),
                        "emotion": emotion_name,
                        "emotion_id": EMOTION_IDX[emotion_name],
                    })
        logger.info(f"Loaded {len(self.samples)} audio samples from {data_dir}")

    def _create_synthetic_dataset(self, n_samples: int):
        emotions = list(EMOTION_PROFILES.keys())
        for i in range(n_samples):
            emotion = emotions[i % len(emotions)]
            self.samples.append({
                "path": None,
                "emotion": emotion,
                "emotion_id": EMOTION_IDX[emotion],
                "synthetic": True,
                "seed": i,
            })

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        sample = self.samples[idx]
        emotion_id = sample["emotion_id"]
        emotion_name = sample["emotion"]
        profile = EMOTION_PROFILES[emotion_name]

        if sample.get("synthetic"):
            torch.manual_seed(sample["seed"])
            waveform = self.processor._generate_demo_signal(duration_sec=2.0)
            waveform = self._apply_emotion_augmentation(waveform, profile)
        else:
            waveform, _ = self.processor.load_audio(sample["path"])

        waveform = self._pad_or_trim_waveform(waveform, self.max_wav_len)
        features = self.processor.extract_all_features(waveform)

        mel = features["mel"]
        f0 = features["f0"]
        voiced = features["voiced"]
        energy = features["energy"]
        duration = features["duration"]

        mel = self._pad_or_trim_mel(mel, self.max_mel_len)
        f0 = self._pad_or_trim_1d(f0, self.max_mel_len)
        voiced = self._pad_or_trim_1d(voiced, self.max_mel_len)
        energy = self._pad_or_trim_1d(energy, self.max_mel_len)

        pitch_mean = f0[voiced > 0.5].mean() if (voiced > 0.5).any() else torch.tensor(0.0)
        pitch_std = f0[voiced > 0.5].std() if (voiced > 0.5).sum() > 1 else torch.tensor(0.0)
        energy_mean = energy.mean()
        energy_std = energy.std()
        speed = torch.tensor(profile.speed_factor, dtype=torch.float32)
        dur_factor = torch.tensor(profile.duration_factor, dtype=torch.float32)
        prosody = torch.stack([pitch_mean, pitch_std, energy_mean, energy_std, speed, dur_factor])

        return {
            "mel": mel,
            "f0": f0,
            "voiced": voiced,
            "energy": energy,
            "duration": duration[:min(len(duration), 32)],
            "emotion_id": torch.tensor(emotion_id, dtype=torch.long),
            "prosody": prosody,
            "emotion_name": emotion_name,
        }

    def _apply_emotion_augmentation(self, waveform: torch.Tensor, profile: EmotionProfile) -> torch.Tensor:
        """Apply basic emotion-specific augmentation to synthetic waveform."""
        wav_np = waveform.squeeze().numpy()
        if profile.speed_factor != 1.0:
            indices = np.linspace(0, len(wav_np) - 1, int(len(wav_np) / profile.speed_factor))
            indices = np.clip(indices.astype(int), 0, len(wav_np) - 1)
            wav_np = wav_np[indices]
        amp = profile.energy_scale * np.random.uniform(0.8, 1.2)
        wav_np = wav_np * amp
        wav_np = wav_np / (np.max(np.abs(wav_np)) + 1e-8) * 0.9
        return torch.from_numpy(wav_np.astype(np.float32)).unsqueeze(0)

    def _pad_or_trim_waveform(self, waveform: torch.Tensor, target: int) -> torch.Tensor:
        waveform = waveform.squeeze(0)
        if waveform.shape[0] > target:
            return waveform[:target].unsqueeze(0)
        elif waveform.shape[0] < target:
            pad = target - waveform.shape[0]
            return F.pad(waveform, (0, pad)).unsqueeze(0)
        return waveform.unsqueeze(0)

    def _pad_or_trim_mel(self, mel: torch.Tensor, target_len: int) -> torch.Tensor:
        T = mel.shape[1]
        if T > target_len:
            return mel[:, :target_len]
        elif T < target_len:
            return F.pad(mel, (0, target_len - T))
        return mel

    def _pad_or_trim_1d(self, x: torch.Tensor, target: int) -> torch.Tensor:
        if x.shape[0] > target:
            return x[:target]
        elif x.shape[0] < target:
            return F.pad(x, (0, target - x.shape[0]))
        return x


def collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    """Custom collate function for variable-length duration sequences."""
    max_dur_len = max(s["duration"].shape[0] for s in batch)
    padded_dur = []
    for s in batch:
        d = s["duration"]
        if d.shape[0] < max_dur_len:
            d = F.pad(d, (0, max_dur_len - d.shape[0]))
        padded_dur.append(d)

    return {
        "mel": torch.stack([s["mel"] for s in batch]),
        "f0": torch.stack([s["f0"] for s in batch]),
        "voiced": torch.stack([s["voiced"] for s in batch]),
        "energy": torch.stack([s["energy"] for s in batch]),
        "duration": torch.stack(padded_dur),
        "emotion_id": torch.stack([s["emotion_id"] for s in batch]),
        "prosody": torch.stack([s["prosody"] for s in batch]),
    }


# =============================================================================
# SECTION 10: TRAINER
# =============================================================================

class EmotionalSpeechTrainer:
    """
    Full training loop for the Emotional Speech Conversion System.
    Includes:
    - VAE training with KL annealing
    - GNN training with emotion graph
    - FastSpeech2 variance adaptor training
    - HiFi-GAN vocoder training (generator + discriminator)
    """

    def __init__(
        self,
        model: EmotionalSpeechConverter,
        train_config: TrainConfig,
        audio_config: AudioConfig,
        device: torch.device,
        data_dir: Optional[str] = None,
    ):
        self.model = model.to(device)
        self.train_config = train_config
        self.audio_config = audio_config
        self.device = device

        self.msd = HiFiGANMultiScaleDiscriminator().to(device)
        self.mpd = HiFiGANMultiPeriodDiscriminator().to(device)

        self.gen_optimizer = optim.AdamW(
            list(model.vae.parameters()) +
            list(model.gnn.parameters()) +
            list(model.variance_adaptor.parameters()) +
            list(model.vocoder.parameters()),
            lr=train_config.learning_rate,
            betas=(0.9, 0.999),
            weight_decay=1e-4
        )
        self.disc_optimizer = optim.AdamW(
            list(self.msd.parameters()) + list(self.mpd.parameters()),
            lr=train_config.learning_rate,
            betas=(0.9, 0.999),
        )

        self.gen_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.gen_optimizer, T_max=train_config.num_epochs, eta_min=1e-6
        )
        self.disc_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.disc_optimizer, T_max=train_config.num_epochs, eta_min=1e-6
        )

        self.loss_fn = EmotionalSpeechLoss(train_config)

        audio_processor = AudioProcessor(audio_config)
        dataset = SpeechDataset(
            data_dir=data_dir,
            audio_processor=audio_processor,
            use_synthetic=(data_dir is None),
        )
        self.dataloader = DataLoader(
            dataset,
            batch_size=train_config.batch_size,
            shuffle=True,
            num_workers=0,
            collate_fn=collate_fn,
            drop_last=True,
        )

        os.makedirs(train_config.checkpoint_dir, exist_ok=True)
        self.global_step = 0
        self.best_loss = float('inf')

        self.history = {
            "gen_loss": [], "disc_loss": [], "mel_loss": [], "kl_loss": [],
            "pitch_loss": [], "energy_loss": [], "epoch": [],
        }

    def _to_device(self, batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        return {k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

    def _warmup_lr(self) -> float:
        """Noam learning rate warmup."""
        step = max(1, self.global_step)
        d_model = self.model.model_config.fastspeech_hidden
        warmup = self.train_config.warmup_steps
        lr = (d_model ** -0.5) * min(step ** -0.5, step * warmup ** -1.5)
        lr = lr * self.train_config.learning_rate * 100
        return lr

    def train_epoch(self, epoch: int) -> Dict[str, float]:
        """Train one epoch."""
        self.model.train()
        self.msd.train()
        self.mpd.train()

        epoch_losses = {
            "gen_total": 0.0, "disc_total": 0.0,
            "mel_recon": 0.0, "kl": 0.0, "pitch": 0.0, "energy": 0.0
        }
        n_batches = 0

        for batch in self.dataloader:
            batch = self._to_device(batch)
            mel = batch["mel"]
            emotion_id = batch["emotion_id"]
            prosody = batch["prosody"]

            B = mel.shape[0]
            target_ids = torch.randint(0, len(EMOTION_PROFILES), (B,), device=self.device)

            lr = self._warmup_lr()
            for pg in self.gen_optimizer.param_groups:
                pg['lr'] = lr

            outputs = self.model(
                mel=mel,
                source_emotion_id=emotion_id,
                target_emotion_id=target_ids,
                source_prosody=prosody,
                target_mel_len=mel.shape[-1],
            )

            waveform_fake = outputs["waveform"]
            mel_conv = outputs["mel_converted"]

            targets = {
                "mel": mel,
                "f0": batch["f0"],
                "voiced": batch["voiced"],
                "energy": batch["energy"],
                "duration": batch["duration"],
            }

            rec_losses = self.loss_fn(outputs, targets)
            mel_loss = rec_losses.get("mel_recon", torch.tensor(0.0, device=self.device))
            kl_loss = rec_losses.get("kl", torch.tensor(0.0, device=self.device))
            pitch_loss = rec_losses.get("pitch", torch.tensor(0.0, device=self.device))
            energy_loss = rec_losses.get("energy", torch.tensor(0.0, device=self.device))

            waveform_len = waveform_fake.shape[-1]
            dummy_real = torch.randn(B, 1, waveform_len, device=self.device) * 0.1

            msd_fake = self.msd(waveform_fake.detach())
            mpd_fake = self.mpd(waveform_fake.detach())
            msd_real = self.msd(dummy_real)
            mpd_real = self.mpd(dummy_real)

            disc_loss = self.loss_fn.discriminator_loss(
                msd_real + mpd_real, msd_fake + mpd_fake
            )

            self.disc_optimizer.zero_grad()
            disc_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(self.msd.parameters()) + list(self.mpd.parameters()),
                self.train_config.grad_clip
            )
            self.disc_optimizer.step()

            msd_fake_gen = self.msd(waveform_fake)
            mpd_fake_gen = self.mpd(waveform_fake)
            gen_adv_loss = self.loss_fn.generator_loss(msd_fake_gen + mpd_fake_gen)

            gen_loss = mel_loss + kl_loss + pitch_loss + energy_loss + gen_adv_loss * 0.1

            self.gen_optimizer.zero_grad()
            gen_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(self.model.parameters()),
                self.train_config.grad_clip
            )
            self.gen_optimizer.step()

            epoch_losses["gen_total"] += gen_loss.item()
            epoch_losses["disc_total"] += disc_loss.item()
            epoch_losses["mel_recon"] += mel_loss.item()
            epoch_losses["kl"] += kl_loss.item()
            epoch_losses["pitch"] += pitch_loss.item() if isinstance(pitch_loss, torch.Tensor) else pitch_loss
            epoch_losses["energy"] += energy_loss.item() if isinstance(energy_loss, torch.Tensor) else energy_loss

            n_batches += 1
            self.global_step += 1

            if self.global_step % self.train_config.log_interval == 0:
                logger.debug(
                    f"  Step {self.global_step}: "
                    f"G={gen_loss.item():.4f} D={disc_loss.item():.4f} "
                    f"Mel={mel_loss.item():.4f} KL={kl_loss.item():.5f}"
                )

        for k in epoch_losses:
            epoch_losses[k] /= max(n_batches, 1)

        return epoch_losses

    def save_checkpoint(self, epoch: int, loss: float):
        """Save model checkpoint."""
        ckpt_path = os.path.join(
            self.train_config.checkpoint_dir,
            f"checkpoint_epoch_{epoch:04d}.pt"
        )
        torch.save({
            "epoch": epoch,
            "global_step": self.global_step,
            "model_state_dict": self.model.state_dict(),
            "gen_optimizer_state_dict": self.gen_optimizer.state_dict(),
            "loss": loss,
        }, ckpt_path)
        logger.info(f"Saved checkpoint: {ckpt_path}")

        best_path = os.path.join(self.train_config.checkpoint_dir, "best_model.pt")
        if loss < self.best_loss:
            self.best_loss = loss
            torch.save(self.model.state_dict(), best_path)
            logger.info(f"New best model saved (loss={loss:.4f})")

    def load_checkpoint(self, path: str):
        """Load model from checkpoint."""
        ckpt = torch.load(path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.gen_optimizer.load_state_dict(ckpt["gen_optimizer_state_dict"])
        self.global_step = ckpt.get("global_step", 0)
        logger.info(f"Loaded checkpoint from {path} (epoch {ckpt['epoch']})")
        return ckpt["epoch"]

    def train(self, num_epochs: Optional[int] = None) -> Dict[str, List[float]]:
        """Full training loop."""
        n_epochs = num_epochs or self.train_config.num_epochs
        logger.info(f"Starting training for {n_epochs} epochs on {self.device}")
        param_counts = self.model.count_parameters()
        logger.info(f"Model parameters: {param_counts}")

        for epoch in range(1, n_epochs + 1):
            t0 = time.time()
            epoch_losses = self.train_epoch(epoch)
            elapsed = time.time() - t0

            self.gen_scheduler.step()
            self.disc_scheduler.step()

            self.history["gen_loss"].append(epoch_losses["gen_total"])
            self.history["disc_loss"].append(epoch_losses["disc_total"])
            self.history["mel_loss"].append(epoch_losses["mel_recon"])
            self.history["kl_loss"].append(epoch_losses["kl"])
            self.history["pitch_loss"].append(epoch_losses["pitch"])
            self.history["energy_loss"].append(epoch_losses["energy"])
            self.history["epoch"].append(epoch)

            logger.info(
                f"Epoch {epoch:3d}/{n_epochs} | "
                f"G={epoch_losses['gen_total']:.4f} | "
                f"D={epoch_losses['disc_total']:.4f} | "
                f"Mel={epoch_losses['mel_recon']:.4f} | "
                f"KL={epoch_losses['kl']:.5f} | "
                f"t={elapsed:.1f}s"
            )

            if epoch % self.train_config.save_interval == 0:
                self.save_checkpoint(epoch, epoch_losses["gen_total"])

        if self.history["gen_loss"]:
            self.save_checkpoint(n_epochs, self.history["gen_loss"][-1])
        return self.history


# =============================================================================
# SECTION 11: EVALUATION METRICS
# =============================================================================

class SpeechEvaluator:
    """
    Comprehensive evaluation suite for emotional speech conversion.

    Metrics:
    - MCD (Mel Cepstral Distortion): spectral distance, lower is better
    - F0-RMSE: pitch accuracy, lower is better
    - F0 Correlation: pitch pattern correlation, higher is better
    - Energy-RMSE: loudness accuracy, lower is better
    - PESQ: perceptual evaluation of speech quality [1-4.5], higher is better
    - STOI: short-time objective intelligibility [0-1], higher is better
    - Cosine Similarity: overall feature similarity [0-1], higher is better
    - Emotion Feature Changes: quantified changes vs neutral reference
    """

    def __init__(self, audio_config: AudioConfig):
        self.audio_config = audio_config
        self.processor = AudioProcessor(audio_config)

    def mel_cepstral_distortion(
        self,
        mel_ref: torch.Tensor,
        mel_hyp: torch.Tensor,
        n_mfcc: int = 13
    ) -> float:
        """
        Compute Mel Cepstral Distortion (MCD) between two mel spectrograms.
        MCD = (10/ln(10)) * sqrt(2 * sum((mc1-mc2)^2))
        """
        m1 = mel_ref.detach().cpu().numpy()
        m2 = mel_hyp.detach().cpu().numpy()

        if m1.shape != m2.shape:
            min_T = min(m1.shape[-1], m2.shape[-1])
            m1 = m1[..., :min_T]
            m2 = m2[..., :min_T]

        if m1.ndim == 3:
            m1 = m1[0]
            m2 = m2[0]

        m1 = m1[:n_mfcc, :]
        m2 = m2[:n_mfcc, :]

        diff = m1 - m2
        mcd = (10.0 / np.log(10.0)) * np.sqrt(2.0 * np.mean(np.sum(diff ** 2, axis=0)))
        return float(mcd)

    def f0_rmse(
        self,
        f0_ref: torch.Tensor,
        f0_hyp: torch.Tensor,
        voiced_ref: Optional[torch.Tensor] = None
    ) -> float:
        """Compute F0 RMSE in semitones on voiced frames."""
        r = f0_ref.detach().cpu().numpy().flatten()
        h = f0_hyp.detach().cpu().numpy().flatten()
        min_len = min(len(r), len(h))
        r = r[:min_len]
        h = h[:min_len]

        if voiced_ref is not None:
            v = voiced_ref.detach().cpu().numpy().flatten()[:min_len]
            mask = v > 0.5
        else:
            mask = (r > 0) & (h > 0)

        if mask.sum() == 0:
            return float('inf')

        r_voiced = r[mask]
        h_voiced = h[mask]
        rmse = np.sqrt(np.mean((r_voiced - h_voiced) ** 2))
        return float(rmse)

    def f0_correlation(
        self,
        f0_ref: torch.Tensor,
        f0_hyp: torch.Tensor,
        voiced_ref: Optional[torch.Tensor] = None
    ) -> float:
        """Compute Pearson correlation coefficient of F0 trajectories."""
        r = f0_ref.detach().cpu().numpy().flatten()
        h = f0_hyp.detach().cpu().numpy().flatten()
        min_len = min(len(r), len(h))
        r = r[:min_len]
        h = h[:min_len]

        if voiced_ref is not None:
            v = voiced_ref.detach().cpu().numpy().flatten()[:min_len]
            mask = v > 0.5
        else:
            mask = (r > 0) & (h > 0)

        if mask.sum() < 2:
            return 0.0

        corr = np.corrcoef(r[mask], h[mask])[0, 1]
        return float(corr) if not np.isnan(corr) else 0.0

    def energy_rmse(self, energy_ref: torch.Tensor, energy_hyp: torch.Tensor) -> float:
        """Compute Energy RMSE."""
        r = energy_ref.detach().cpu().numpy().flatten()
        h = energy_hyp.detach().cpu().numpy().flatten()
        min_len = min(len(r), len(h))
        return float(np.sqrt(np.mean((r[:min_len] - h[:min_len]) ** 2)))

    def cosine_similarity(self, mel_ref: torch.Tensor, mel_hyp: torch.Tensor) -> float:
        """Compute cosine similarity between average mel frame vectors."""
        r = mel_ref.detach().cpu()
        h = mel_hyp.detach().cpu()
        if r.dim() == 3:
            r = r[0]
        if h.dim() == 3:
            h = h[0]
        min_T = min(r.shape[1], h.shape[1])
        r_mean = r[:, :min_T].mean(dim=1)
        h_mean = h[:, :min_T].mean(dim=1)
        cos = F.cosine_similarity(r_mean.unsqueeze(0), h_mean.unsqueeze(0)).item()
        return float(cos)

    def compute_pesq(self, wav_ref: np.ndarray, wav_hyp: np.ndarray, sr: int = 16000) -> float:
        """Compute PESQ score (requires pesq package)."""
        if not PESQ_AVAILABLE:
            return float('nan')
        try:
            min_len = min(len(wav_ref), len(wav_hyp))
            score = pesq(sr, wav_ref[:min_len], wav_hyp[:min_len], 'wb')
            return float(score)
        except Exception:
            return float('nan')

    def compute_stoi(self, wav_ref: np.ndarray, wav_hyp: np.ndarray, sr: int = 16000) -> float:
        """Compute STOI (Short-Time Objective Intelligibility) score."""
        if not STOI_AVAILABLE:
            return float('nan')
        try:
            min_len = min(len(wav_ref), len(wav_hyp))
            score = stoi(wav_ref[:min_len], wav_hyp[:min_len], sr, extended=False)
            return float(score)
        except Exception:
            return float('nan')

    def compute_emotion_feature_changes(
        self,
        mel_ref: torch.Tensor,
        mel_hyp: torch.Tensor,
        f0_ref: Optional[torch.Tensor] = None,
        f0_hyp: Optional[torch.Tensor] = None,
        energy_ref: Optional[torch.Tensor] = None,
        energy_hyp: Optional[torch.Tensor] = None,
        emotion_profile: Optional[EmotionProfile] = None,
    ) -> Dict[str, float]:
        """
        Compute quantified feature changes between reference and converted speech.
        Returns percentage changes and expected changes from emotion profile.
        """
        changes = {}

        if f0_ref is not None and f0_hyp is not None:
            r = f0_ref.detach().cpu().numpy().flatten()
            h = f0_hyp.detach().cpu().numpy().flatten()
            min_len = min(len(r), len(h))
            voiced_r = r[:min_len] != 0
            if voiced_r.sum() > 0:
                mean_ref = np.mean(r[:min_len][voiced_r])
                mean_hyp = np.mean(h[:min_len][voiced_r])
                changes["pitch_mean_change_pct"] = (mean_hyp - mean_ref) / (abs(mean_ref) + 1e-8) * 100
                std_ref = np.std(r[:min_len][voiced_r])
                std_hyp = np.std(h[:min_len][voiced_r])
                changes["pitch_variance_change_pct"] = (std_hyp - std_ref) / (abs(std_ref) + 1e-8) * 100
            else:
                changes["pitch_mean_change_pct"] = 0.0
                changes["pitch_variance_change_pct"] = 0.0

        if energy_ref is not None and energy_hyp is not None:
            r = energy_ref.detach().cpu().numpy().flatten()
            h = energy_hyp.detach().cpu().numpy().flatten()
            min_len = min(len(r), len(h))
            mean_ref = np.mean(r[:min_len])
            mean_hyp = np.mean(h[:min_len])
            changes["energy_mean_change_pct"] = (mean_hyp - mean_ref) / (abs(mean_ref) + 1e-8) * 100
            std_ref = np.std(r[:min_len])
            std_hyp = np.std(h[:min_len])
            changes["energy_variance_change_pct"] = (std_hyp - std_ref) / (abs(std_ref) + 1e-8) * 100

        if mel_ref is not None and mel_hyp is not None:
            ref_dur = mel_ref.shape[-1]
            hyp_dur = mel_hyp.shape[-1]
            changes["duration_change_pct"] = (hyp_dur - ref_dur) / (ref_dur + 1e-8) * 100

        if emotion_profile is not None:
            changes["expected_pitch_shift_semitones"] = emotion_profile.pitch_shift
            changes["expected_speed_factor"] = emotion_profile.speed_factor
            changes["expected_energy_scale"] = emotion_profile.energy_scale
            changes["expected_pitch_variance"] = emotion_profile.pitch_variance
            changes["expected_energy_variance"] = emotion_profile.energy_variance
            changes["expected_duration_factor"] = emotion_profile.duration_factor
            changes["voice_quality"] = emotion_profile.voice_quality

        return changes

    def evaluate_conversion(
        self,
        mel_original: torch.Tensor,
        mel_converted: torch.Tensor,
        waveform_original: Optional[torch.Tensor] = None,
        waveform_converted: Optional[torch.Tensor] = None,
        f0_original: Optional[torch.Tensor] = None,
        f0_converted: Optional[torch.Tensor] = None,
        energy_original: Optional[torch.Tensor] = None,
        energy_converted: Optional[torch.Tensor] = None,
        voiced: Optional[torch.Tensor] = None,
        emotion_name: str = "unknown",
        emotion_profile: Optional[EmotionProfile] = None,
        sample_rate: int = 22050,
    ) -> Dict[str, Any]:
        """Run full evaluation suite and return metrics dict."""
        metrics = {"emotion": emotion_name}

        metrics["mcd"] = self.mel_cepstral_distortion(mel_original, mel_converted)
        metrics["cosine_similarity"] = self.cosine_similarity(mel_original, mel_converted)

        if f0_original is not None and f0_converted is not None:
            metrics["f0_rmse"] = self.f0_rmse(f0_original, f0_converted, voiced)
            metrics["f0_correlation"] = self.f0_correlation(f0_original, f0_converted, voiced)
        else:
            metrics["f0_rmse"] = float('nan')
            metrics["f0_correlation"] = float('nan')

        if energy_original is not None and energy_converted is not None:
            metrics["energy_rmse"] = self.energy_rmse(energy_original, energy_converted)
        else:
            metrics["energy_rmse"] = float('nan')

        if waveform_original is not None and waveform_converted is not None:
            wav_ref = waveform_original.squeeze().cpu().numpy()
            wav_hyp = waveform_converted.squeeze().cpu().numpy()

            resample_sr = 16000
            if sample_rate != resample_sr:
                resampler = T.Resample(sample_rate, resample_sr)
                wav_ref_t = resampler(torch.from_numpy(wav_ref).unsqueeze(0)).squeeze().numpy()
                wav_hyp_t = resampler(torch.from_numpy(wav_hyp).unsqueeze(0)).squeeze().numpy()
            else:
                wav_ref_t = wav_ref
                wav_hyp_t = wav_hyp

            metrics["pesq"] = self.compute_pesq(wav_ref_t, wav_hyp_t, resample_sr)
            metrics["stoi"] = self.compute_stoi(wav_ref_t, wav_hyp_t, resample_sr)
        else:
            metrics["pesq"] = float('nan')
            metrics["stoi"] = float('nan')

        feature_changes = self.compute_emotion_feature_changes(
            mel_original, mel_converted,
            f0_original, f0_converted,
            energy_original, energy_converted,
            emotion_profile
        )
        metrics.update(feature_changes)

        return metrics

    def build_evaluation_table(
        self,
        all_metrics: List[Dict[str, Any]],
        reference_emotion: str = "neutral"
    ) -> str:
        """
        Build a comprehensive evaluation table as a formatted string.

        Table columns:
          Emotion | MCD↓ | F0-RMSE↓ | F0-Corr↑ | E-RMSE↓ | PESQ↑ | STOI↑ | CosSim↑ |
          Pitch%Δ | Energy%Δ | Dur%Δ | Expected Shift | Voice Quality
        """
        headers = [
            "Emotion", "MCD↓", "F0-RMSE↓", "F0-Corr↑", "E-RMSE↓",
            "PESQ↑", "STOI↑", "CosSim↑",
            "Pitch%Δ", "Energy%Δ", "Dur%Δ",
            "Exp.Pitch(st)", "Exp.Energy×", "Voice Quality"
        ]

        rows = []
        for m in all_metrics:
            emotion = m.get("emotion", "unknown")
            profile = EMOTION_PROFILES.get(emotion)

            def fmt(v, decimals=3):
                if v is None or (isinstance(v, float) and math.isnan(v)):
                    return "N/A"
                return f"{v:.{decimals}f}"

            row = [
                emotion.upper(),
                fmt(m.get("mcd"), 2),
                fmt(m.get("f0_rmse"), 3),
                fmt(m.get("f0_correlation"), 3),
                fmt(m.get("energy_rmse"), 3),
                fmt(m.get("pesq"), 2),
                fmt(m.get("stoi"), 3),
                fmt(m.get("cosine_similarity"), 3),
                f"{m.get('pitch_mean_change_pct', 0.0):+.1f}%",
                f"{m.get('energy_mean_change_pct', 0.0):+.1f}%",
                f"{m.get('duration_change_pct', 0.0):+.1f}%",
                f"{m.get('expected_pitch_shift_semitones', 0.0):+.1f}st" if profile else "N/A",
                f"{m.get('expected_energy_scale', 1.0):.2f}×" if profile else "N/A",
                m.get("voice_quality", profile.voice_quality if profile else "N/A"),
            ]
            rows.append(row)

        if TABULATE_AVAILABLE:
            table = tabulate(rows, headers=headers, tablefmt="grid", floatfmt=".3f")
        else:
            col_widths = [max(len(h), max((len(str(r[i])) for r in rows), default=0))
                         for i, h in enumerate(headers)]
            sep = "+" + "+".join("-" * (w + 2) for w in col_widths) + "+"
            header_row = "|" + "|".join(f" {h:<{w}} " for h, w in zip(headers, col_widths)) + "|"
            lines = [sep, header_row, sep]
            for row in rows:
                data_row = "|" + "|".join(f" {str(c):<{w}} " for c, w in zip(row, col_widths)) + "|"
                lines.append(data_row)
            lines.append(sep)
            table = "\n".join(lines)

        return table

    def print_feature_analysis(self, all_metrics: List[Dict[str, Any]]):
        """Print detailed feature change analysis per emotion."""
        print("\n" + "=" * 80)
        print("EMOTION-WISE FEATURE CHANGE ANALYSIS")
        print("=" * 80)

        for m in all_metrics:
            emotion = m.get("emotion", "unknown")
            profile = EMOTION_PROFILES.get(emotion)
            if profile is None:
                continue

            print(f"\n[ {emotion.upper()} ]")
            print(f"  Description  : {profile.description}")
            print(f"  Voice Quality: {profile.voice_quality}")
            print(f"  Intensity    : {profile.intensity_range[0]:.1f} - {profile.intensity_range[1]:.1f}")
            print(f"\n  ACOUSTIC CHANGES vs NEUTRAL:")
            print(f"    Pitch Shift   : {profile.pitch_shift:+.1f} semitones (expected)")
            print(f"    Speed Factor  : {profile.speed_factor:.2f}× (expected)")
            print(f"    Energy Scale  : {profile.energy_scale:.2f}× (expected)")
            print(f"    Pitch Variance: {profile.pitch_variance:.3f} (expected)")
            print(f"    Energy Variance: {profile.energy_variance:.3f} (expected)")
            print(f"    Duration Factor: {profile.duration_factor:.2f}× (expected)")

            print(f"\n  MEASURED CHANGES:")
            pitch_chg = m.get("pitch_mean_change_pct", None)
            energy_chg = m.get("energy_mean_change_pct", None)
            dur_chg = m.get("duration_change_pct", None)
            if pitch_chg is not None:
                print(f"    Pitch Mean   : {pitch_chg:+.1f}%")
            if energy_chg is not None:
                print(f"    Energy Mean  : {energy_chg:+.1f}%")
            if dur_chg is not None:
                print(f"    Duration     : {dur_chg:+.1f}%")

            print(f"\n  QUALITY METRICS:")
            print(f"    MCD          : {m.get('mcd', float('nan')):.3f} dB (lower=better)")
            print(f"    F0 Corr.     : {m.get('f0_correlation', float('nan')):.3f} (higher=better)")
            print(f"    Cosine Sim.  : {m.get('cosine_similarity', float('nan')):.3f} (higher=better)")
            if not math.isnan(m.get("pesq", float('nan'))):
                print(f"    PESQ         : {m.get('pesq'):.2f} (higher=better)")
            if not math.isnan(m.get("stoi", float('nan'))):
                print(f"    STOI         : {m.get('stoi'):.3f} (higher=better)")

        print("\n" + "=" * 80)


# =============================================================================
# SECTION 12: POST-PROCESSING FOR NATURAL AUDIO
# =============================================================================

class NaturalAudioPostProcessor:
    """
    Post-processing pipeline to make generated audio sound more natural (human-like).

    Techniques:
    1. Spectral smoothing (reduces harshness)
    2. Micro-pitch jitter (human voice naturalness)
    3. Amplitude modulation (breath-like variations)
    4. Phase randomization in high frequencies (de-robotize)
    5. Formant enhancement
    6. Low-level noise addition (presence/air)
    7. Dynamic range compression (consistent loudness)
    """

    def __init__(self, sample_rate: int = 22050):
        self.sr = sample_rate

    def apply(self, waveform: torch.Tensor, emotion_profile: Optional[EmotionProfile] = None) -> torch.Tensor:
        """Apply full post-processing pipeline."""
        wav = waveform.squeeze().numpy()
        wav = self._normalize(wav)
        wav = self._add_jitter(wav, emotion_profile)
        wav = self._apply_spectral_shaping(wav, emotion_profile)
        wav = self._add_breath_modulation(wav, emotion_profile)
        wav = self._apply_natural_noise(wav, emotion_profile)
        wav = self._dynamic_range_compression(wav)
        wav = self._normalize(wav)
        return torch.from_numpy(wav.astype(np.float32)).unsqueeze(0)

    def _normalize(self, wav: np.ndarray, target_rms: float = 0.1) -> np.ndarray:
        rms = np.sqrt(np.mean(wav ** 2)) + 1e-8
        return wav * (target_rms / rms)

    def _add_jitter(self, wav: np.ndarray, profile: Optional[EmotionProfile] = None) -> np.ndarray:
        """
        Add micro-pitch jitter and shimmer to humanize the voice.
        The amount scales with emotion intensity and voice quality.
        """
        jitter_amount = 0.002
        shimmer_amount = 0.03

        if profile is not None:
            if profile.voice_quality in ("tremulous",):
                jitter_amount *= 3.0
                shimmer_amount *= 2.5
            elif profile.voice_quality in ("creaky",):
                jitter_amount *= 0.5
                shimmer_amount *= 0.8
            elif profile.voice_quality in ("tense",):
                jitter_amount *= 1.5

        np.random.seed(42)
        jitter_signal = 1.0 + np.random.normal(0, jitter_amount, len(wav))
        shimmer_signal = 1.0 + np.random.normal(0, shimmer_amount, len(wav))

        jitter_signal = np.convolve(jitter_signal, np.ones(50) / 50, mode='same')
        shimmer_signal = np.convolve(shimmer_signal, np.ones(100) / 100, mode='same')

        return wav * jitter_signal * shimmer_signal

    def _apply_spectral_shaping(self, wav: np.ndarray, profile: Optional[EmotionProfile] = None) -> np.ndarray:
        """Shape frequency content based on emotion's voice quality."""
        voice_quality = profile.voice_quality if profile else "modal"

        if voice_quality == "bright":
            b, a = signal.butter(2, 3000 / (self.sr / 2), btype='high')
            high_freq = signal.lfilter(b, a, wav)
            wav = wav + high_freq * 0.15
        elif voice_quality == "breathy":
            b, a = signal.butter(2, 500 / (self.sr / 2), btype='low')
            low_freq = signal.lfilter(b, a, wav)
            noise = np.random.randn(len(wav)) * 0.01
            b_noise, a_noise = signal.butter(2, [2000 / (self.sr / 2), 6000 / (self.sr / 2)], btype='band')
            colored_noise = signal.lfilter(b_noise, a_noise, noise)
            wav = wav + colored_noise
        elif voice_quality == "tense":
            b, a = signal.butter(2, [1500 / (self.sr / 2), 4000 / (self.sr / 2)], btype='band')
            mid = signal.lfilter(b, a, wav)
            wav = wav + mid * 0.2
        elif voice_quality == "creaky":
            period = int(self.sr / 100)
            creak = np.zeros(len(wav))
            for i in range(0, len(wav) - period, period):
                creak[i] = np.random.uniform(0.02, 0.08)
            b, a = signal.butter(2, 300 / (self.sr / 2), btype='low')
            creak_filtered = signal.lfilter(b, a, creak)
            wav = wav + creak_filtered
        elif voice_quality == "tremulous":
            freq = np.random.uniform(4.5, 6.5)
            t = np.linspace(0, len(wav) / self.sr, len(wav))
            tremolo = 1.0 + 0.04 * np.sin(2 * np.pi * freq * t)
            wav = wav * tremolo

        return wav

    def _add_breath_modulation(self, wav: np.ndarray, profile: Optional[EmotionProfile] = None) -> np.ndarray:
        """Simulate natural amplitude envelope variations (breathing rhythm)."""
        breath_rate = 0.3 if profile and profile.speed_factor < 0.9 else 0.45
        t = np.linspace(0, len(wav) / self.sr, len(wav))
        breath_env = 1.0 + 0.02 * np.sin(2 * np.pi * breath_rate * t)
        phrase_env = 1.0 + 0.04 * np.sin(2 * np.pi * 0.08 * t + np.pi / 4)
        return wav * breath_env * phrase_env

    def _apply_natural_noise(self, wav: np.ndarray, profile: Optional[EmotionProfile] = None) -> np.ndarray:
        """Add subtle colored noise for air/presence (makes audio feel 'live')."""
        noise_level = 0.003
        if profile and profile.voice_quality == "breathy":
            noise_level = 0.008
        elif profile and profile.voice_quality == "tense":
            noise_level = 0.001

        noise = np.random.randn(len(wav)) * noise_level
        b, a = signal.butter(1, 4000 / (self.sr / 2), btype='low')
        colored_noise = signal.lfilter(b, a, noise)
        return wav + colored_noise

    def _dynamic_range_compression(self, wav: np.ndarray, threshold: float = 0.3, ratio: float = 4.0) -> np.ndarray:
        """Apply dynamic range compression for consistent loudness (like a human vocal mic)."""
        compressed = wav.copy()
        above_threshold = np.abs(wav) > threshold
        compressed[above_threshold] = np.sign(wav[above_threshold]) * (
            threshold + (np.abs(wav[above_threshold]) - threshold) / ratio
        )
        return compressed


# =============================================================================
# SECTION 13: MAIN PIPELINE
# =============================================================================

class EmotionalSpeechPipeline:
    """
    High-level pipeline that ties everything together:
    1. Load audio
    2. Extract features
    3. Train model (or load pretrained)
    4. Convert to all emotions
    5. Post-process for naturalness
    6. Save outputs
    7. Run evaluation
    8. Print evaluation table
    """

    def __init__(
        self,
        audio_config: Optional[AudioConfig] = None,
        model_config: Optional[ModelConfig] = None,
        train_config: Optional[TrainConfig] = None,
        device: Optional[str] = None,
        output_dir: str = "emotional_speech_system/outputs",
        checkpoint_dir: str = "emotional_speech_system/checkpoints",
    ):
        self.audio_config = audio_config or AudioConfig()
        self.model_config = model_config or ModelConfig()
        self.train_config = train_config or TrainConfig(checkpoint_dir=checkpoint_dir)
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(checkpoint_dir, exist_ok=True)

        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(device)
        logger.info(f"Using device: {self.device}")

        self.audio_processor = AudioProcessor(self.audio_config)
        self.post_processor = NaturalAudioPostProcessor(self.audio_config.sample_rate)
        self.evaluator = SpeechEvaluator(self.audio_config)

        self.model = EmotionalSpeechConverter(self.model_config, self.audio_config)
        self.model = self.model.to(self.device)

        param_counts = self.model.count_parameters()
        logger.info(f"Model initialized. Parameters: {param_counts}")

    def train(
        self,
        data_dir: Optional[str] = None,
        num_epochs: int = 5,
        resume_from: Optional[str] = None,
    ):
        """Train the model."""
        trainer = EmotionalSpeechTrainer(
            model=self.model,
            train_config=self.train_config,
            audio_config=self.audio_config,
            device=self.device,
            data_dir=data_dir,
        )

        if resume_from and os.path.exists(resume_from):
            trainer.load_checkpoint(resume_from)

        history = trainer.train(num_epochs=num_epochs)
        return history

    def load_pretrained(self, checkpoint_path: str):
        """Load pretrained model weights."""
        state_dict = torch.load(checkpoint_path, map_location=self.device)
        if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
            state_dict = state_dict["model_state_dict"]
        self.model.load_state_dict(state_dict, strict=False)
        logger.info(f"Loaded pretrained model from {checkpoint_path}")

    def process_audio(
        self,
        input_path: Optional[str] = None,
        source_emotion: str = "neutral",
        target_emotions: Optional[List[str]] = None,
        save_outputs: bool = True,
        run_evaluation: bool = True,
    ) -> Dict[str, Any]:
        """
        Full pipeline: load audio -> convert to emotions -> evaluate.

        Args:
            input_path: Path to input WAV file
            source_emotion: Assumed emotion of input audio
            target_emotions: List of emotions to convert to (None = all)
            save_outputs: Whether to save output WAV files
            run_evaluation: Whether to run evaluation metrics

        Returns:
            Dictionary with all results and metrics
        """
        if target_emotions is None:
            target_emotions = list(EMOTION_PROFILES.keys())

        logger.info(f"\n{'='*60}")
        logger.info("EMOTIONAL SPEECH CONVERSION PIPELINE")
        logger.info(f"{'='*60}")

        waveform = self.audio_processor.load_or_generate_audio(input_path, duration_sec=3.0)
        logger.info(f"Input audio: {waveform.shape} @ {self.audio_config.sample_rate}Hz")

        if save_outputs:
            orig_path = os.path.join(self.output_dir, "original.wav")
            self.audio_processor.save_audio(waveform, orig_path)

        features = self.audio_processor.extract_all_features(waveform)
        mel_original = features["mel"].to(self.device)
        f0_original = features["f0"]
        energy_original = features["energy"]
        voiced_original = features["voiced"]

        logger.info(f"Mel shape: {mel_original.shape}")
        logger.info(f"F0 shape: {f0_original.shape}, voiced frames: {voiced_original.sum().int().item()}")
        logger.info(f"Energy mean: {energy_original.mean():.3f}")

        logger.info("\n--- Extracting VAE Latent Representation ---")
        self.model.eval()
        with torch.no_grad():
            mel_batch = mel_original.unsqueeze(0)
            mu, log_var = self.model.get_latent(mel_batch)
            z_source = mu

        logger.info(f"Latent code shape: {z_source.shape}")
        logger.info(f"Latent stats: mean={z_source.mean():.4f}, std={z_source.std():.4f}")

        all_results = {}
        all_metrics = []

        logger.info(f"\n--- Converting to {len(target_emotions)} emotions ---")
        for target_emotion in target_emotions:
            logger.info(f"\n  Processing: {target_emotion.upper()}")
            profile = EMOTION_PROFILES[target_emotion]

            try:
                out = self.model.convert_emotion(
                    mel=mel_original,
                    source_emotion=source_emotion,
                    target_emotion=target_emotion,
                    target_mel_len=mel_original.shape[-1],
                )

                waveform_raw = out["waveform"].squeeze(0)
                mel_converted = out["mel_converted"].squeeze(0)
                f0_converted = out["pitch_pred"].squeeze(0).cpu()
                energy_converted = out["energy_pred"].squeeze(0).cpu()

                waveform_natural = self.post_processor.apply(waveform_raw.cpu(), profile)
                logger.info(f"    Waveform shape: {waveform_natural.shape}")

                if save_outputs:
                    out_path = os.path.join(self.output_dir, f"{target_emotion}.wav")
                    self.audio_processor.save_audio(waveform_natural, out_path)
                    logger.info(f"    Saved: {out_path}")

                if run_evaluation:
                    metrics = self.evaluator.evaluate_conversion(
                        mel_original=mel_original.unsqueeze(0).cpu(),
                        mel_converted=mel_converted.unsqueeze(0).cpu(),
                        waveform_original=waveform,
                        waveform_converted=waveform_natural,
                        f0_original=f0_original,
                        f0_converted=f0_converted,
                        energy_original=energy_original,
                        energy_converted=energy_converted,
                        voiced=voiced_original,
                        emotion_name=target_emotion,
                        emotion_profile=profile,
                        sample_rate=self.audio_config.sample_rate,
                    )
                    all_metrics.append(metrics)

                    mcd = metrics.get("mcd", float('nan'))
                    cos = metrics.get("cosine_similarity", float('nan'))
                    logger.info(f"    MCD={mcd:.3f}  CosSim={cos:.3f}")

                all_results[target_emotion] = {
                    "waveform": waveform_natural,
                    "mel": mel_converted.cpu(),
                    "f0": f0_converted,
                    "energy": energy_converted,
                }

            except Exception as e:
                logger.error(f"  Error converting to {target_emotion}: {e}")
                import traceback
                traceback.print_exc()

        results = {
            "source_waveform": waveform,
            "source_mel": mel_original.cpu(),
            "source_f0": f0_original,
            "source_energy": energy_original,
            "source_voiced": voiced_original,
            "z_source": z_source.cpu(),
            "converted": all_results,
            "metrics": all_metrics,
        }

        if run_evaluation and all_metrics:
            self._print_evaluation_results(all_metrics)

        return results

    def _print_evaluation_results(self, all_metrics: List[Dict[str, Any]]):
        """Print evaluation table and feature analysis."""
        print("\n" + "=" * 100)
        print("EVALUATION RESULTS: EMOTIONAL SPEECH CONVERSION")
        print("=" * 100)
        print("\n[ METRIC LEGEND ]")
        print("  MCD↓         - Mel Cepstral Distortion (dB), lower = more similar spectrum")
        print("  F0-RMSE↓     - F0 Root Mean Square Error (log scale), lower = more accurate pitch")
        print("  F0-Corr↑     - F0 Pearson Correlation, higher = pitch patterns more similar")
        print("  E-RMSE↓      - Energy RMSE, lower = more similar loudness")
        print("  PESQ↑        - Perceptual Eval Speech Quality [1-4.5], higher = better")
        print("  STOI↑        - Short-Time Objective Intelligibility [0-1], higher = better")
        print("  CosSim↑      - Cosine Similarity of mel vectors [0-1], higher = more similar")
        print("  Pitch%Δ      - % change in mean pitch vs. original")
        print("  Energy%Δ     - % change in mean energy vs. original")
        print("  Dur%Δ        - % change in duration vs. original")
        print("  Exp.Pitch    - Expected pitch shift in semitones (from emotion profile)")
        print("  Exp.Energy   - Expected energy scale factor (from emotion profile)")
        print("  Voice Quality- Qualitative voice quality label")

        print("\n[ EVALUATION TABLE ]\n")
        table = self.evaluator.build_evaluation_table(all_metrics)
        print(table)

        self.evaluator.print_feature_analysis(all_metrics)

        if PANDAS_AVAILABLE:
            records = []
            for m in all_metrics:
                records.append({
                    "emotion": m.get("emotion", ""),
                    "mcd": m.get("mcd"),
                    "f0_rmse": m.get("f0_rmse"),
                    "f0_correlation": m.get("f0_correlation"),
                    "energy_rmse": m.get("energy_rmse"),
                    "pesq": m.get("pesq"),
                    "stoi": m.get("stoi"),
                    "cosine_similarity": m.get("cosine_similarity"),
                    "pitch_mean_change_pct": m.get("pitch_mean_change_pct"),
                    "energy_mean_change_pct": m.get("energy_mean_change_pct"),
                    "duration_change_pct": m.get("duration_change_pct"),
                    "expected_pitch_shift_semitones": m.get("expected_pitch_shift_semitones"),
                    "expected_energy_scale": m.get("expected_energy_scale"),
                    "voice_quality": m.get("voice_quality"),
                })
            try:
                df = pd.DataFrame(records)
                csv_path = os.path.join(self.output_dir, "evaluation_results.csv")
                df.to_csv(csv_path, index=False)
                print(f"\n[ Evaluation results saved to: {csv_path} ]")
            except Exception as e:
                logger.warning(f"Could not save CSV: {e}")

    def print_model_architecture(self):
        """Print detailed model architecture summary."""
        print("\n" + "=" * 80)
        print("MODEL ARCHITECTURE: EMOTIONAL SPEECH CONVERSION SYSTEM")
        print("=" * 80)

        print("\n[ COMPONENT 1: VARIATIONAL AUTOENCODER (VAE) ]")
        print("  Purpose: Encode voice audio into latent space (content + speaker)")
        print(f"  Encoder: Conv1D ResBlocks x{self.model_config.vae_num_layers} -> Pooling -> μ,σ ∈ R^{self.model_config.latent_dim}")
        print(f"  Decoder: FC({self.model_config.latent_dim}) -> TransConv1D x3 -> ResBlocks x{self.model_config.vae_num_layers} -> Mel")
        print(f"  Emotion Conditioning: Embedding({self.model_config.num_emotions}) + MLP mapper")
        print(f"  KL Annealing: beta={self.train_config.beta_kl} -> {self.train_config.beta_kl_max} over {self.train_config.beta_kl_anneal_steps} steps")

        print("\n[ COMPONENT 2: GRAPH NEURAL NETWORK (GNN) ]")
        print("  Purpose: Map source emotion latent -> target emotion latent")
        print(f"  Graph: {self.model_config.num_emotions} emotion nodes in Valence-Arousal space")
        print(f"  Edges: weighted by VA-space proximity (similarity > 0.3 threshold)")
        print(f"  Layers: {self.model_config.gnn_num_layers}x GAT ({self.model_config.gnn_num_heads} heads) + FFN + LayerNorm")
        print(f"  Message Passing: Graph Attention with learnable node features")
        print(f"  Output: z_target ∈ R^{self.model_config.latent_dim} + prosody_target ∈ R^6")

        print("\n[ COMPONENT 3: FASTSPEECH 2 VARIANCE ADAPTOR ]")
        print("  Purpose: Generate mel spectrogram with emotion-controlled variance")
        print(f"  Phoneme Encoder: {self.model_config.fastspeech_num_layers}x FFT Blocks (d={self.model_config.fastspeech_hidden})")
        print(f"  Duration Predictor: Conv-based -> Length Regulator (rate control)")
        print(f"  Pitch Predictor: Conv-based -> Quantized embedding ({self.model_config.pitch_bins} bins)")
        print(f"  Energy Predictor: Conv-based -> Quantized embedding ({self.model_config.energy_bins} bins)")
        print(f"  Mel Encoder: {self.model_config.fastspeech_num_layers}x FFT Blocks -> Linear -> Mel")
        print(f"  Emotion Gate: Soft-gate between content hidden and emotion hidden")

        print("\n[ COMPONENT 4: HIFI-GAN VOCODER ]")
        print("  Purpose: Convert mel spectrogram -> natural-sounding waveform")
        print(f"  Architecture: HiFi-GAN V1 (best quality/naturalness tradeoff)")
        print(f"  Upsample Rates: {self.model_config.hifigan_upsample_rates} = {math.prod(self.model_config.hifigan_upsample_rates)}x total upsampling")
        print(f"  ResBlock Kernels: {self.model_config.hifigan_resblock_kernel_sizes}")
        print(f"  Dilations: {self.model_config.hifigan_resblock_dilation_sizes}")
        print(f"  Discriminators: Multi-Scale (3) + Multi-Period (5 periods: 2,3,5,7,11)")
        print(f"  Key: Multi-Receptive Field Fusion (MRF) captures all speech timescales")

        print("\n[ POST-PROCESSING PIPELINE ]")
        print("  1. Micro-jitter & shimmer (humanize voice, remove AI flatness)")
        print("  2. Spectral shaping per voice quality (bright/breathy/tense/creaky/tremulous)")
        print("  3. Breath amplitude modulation (natural breathing rhythm)")
        print("  4. Colored noise addition (air/presence/live feel)")
        print("  5. Dynamic range compression (consistent vocal loudness)")

        param_counts = self.model.count_parameters()
        print("\n[ PARAMETER COUNTS ]")
        for name, count in param_counts.items():
            print(f"  {name:25s}: {count:>10,}")
        print("=" * 80)


# =============================================================================
# SECTION 14: EMOTION GRAPH VISUALIZATION
# =============================================================================

def print_emotion_graph():
    """Print text representation of the emotion graph."""
    print("\n" + "=" * 60)
    print("EMOTION GRAPH (Valence-Arousal Space)")
    print("=" * 60)

    va_positions = {
        "neutral":   (0.5, 0.5),
        "happy":     (0.9, 0.7),
        "sad":       (0.1, 0.2),
        "angry":     (0.1, 0.9),
        "fearful":   (0.2, 0.8),
        "surprised": (0.6, 0.9),
        "disgusted": (0.1, 0.6),
    }

    print("\nValence-Arousal Positions:")
    print(f"  {'Emotion':<12} {'Valence':>8} {'Arousal':>8}")
    print("  " + "-" * 30)
    for emotion, (v, a) in va_positions.items():
        print(f"  {emotion:<12} {v:>8.2f} {a:>8.2f}")

    print("\nEmotion Profiles:")
    print(f"  {'Emotion':<12} {'Pitch Shift':>12} {'Speed':>8} {'Energy':>8} {'Voice Quality':<14}")
    print("  " + "-" * 60)
    for name, p in EMOTION_PROFILES.items():
        print(f"  {name:<12} {p.pitch_shift:>+11.1f}st {p.speed_factor:>7.2f}x {p.energy_scale:>7.2f}x {p.voice_quality:<14}")

    print("\nGraph Edges (high similarity):")
    emotions = list(EMOTION_PROFILES.keys())
    for i, e1 in enumerate(emotions):
        for j, e2 in enumerate(emotions):
            if i < j:
                v1, a1 = va_positions[e1]
                v2, a2 = va_positions[e2]
                dist = math.sqrt((v1 - v2) ** 2 + (a1 - a2) ** 2)
                sim = 1.0 - dist / math.sqrt(2.0)
                if sim > 0.5:
                    print(f"  {e1:<12} <-> {e2:<12}  similarity={sim:.3f}")
    print("=" * 60)


# =============================================================================
# SECTION 15: DEMO AUDIO GENERATOR
# =============================================================================

def generate_emotion_demo_audio(
    output_dir: str = "emotional_speech_system/sample_audio",
    sr: int = 22050,
    duration: float = 2.5
) -> Dict[str, str]:
    """
    Generate demo reference audio for each emotion using synthesis.
    These serve as reference inputs when no real audio is provided.
    """
    os.makedirs(output_dir, exist_ok=True)
    audio_config = AudioConfig(sample_rate=sr)
    processor = AudioProcessor(audio_config)
    saved_paths = {}

    logger.info("Generating demo audio samples for each emotion...")

    for emotion_name, profile in EMOTION_PROFILES.items():
        waveform = processor._generate_demo_signal(duration_sec=duration)

        wav_np = waveform.squeeze().numpy()
        if profile.speed_factor != 1.0:
            indices = np.linspace(0, len(wav_np) - 1, int(len(wav_np) / profile.speed_factor))
            indices = np.clip(indices.astype(int), 0, len(wav_np) - 1)
            wav_np = wav_np[indices]

        f0_base = 120.0 * (2 ** (profile.pitch_shift / 12.0))
        t = np.linspace(0, len(wav_np) / sr, len(wav_np))
        pitch_mod = 1.0 + profile.pitch_variance * np.sin(2 * np.pi * 5 * t)
        phase = np.cumsum(2 * np.pi * f0_base * pitch_mod / sr)
        harmonic = np.zeros_like(wav_np)
        for k in range(1, 8):
            harmonic += (1.0 / k) * np.sin(k * phase)

        harmonic = harmonic / (np.max(np.abs(harmonic)) + 1e-8)
        wav_np = 0.7 * wav_np + 0.3 * harmonic
        wav_np = wav_np * profile.energy_scale
        wav_np = wav_np / (np.max(np.abs(wav_np)) + 1e-8) * 0.85

        out_path = os.path.join(output_dir, f"{emotion_name}_demo.wav")
        wav_int16 = np.clip(wav_np * 32767.0, -32768, 32767).astype(np.int16)
        wav_io.write(out_path, sr, wav_int16)
        saved_paths[emotion_name] = out_path
        logger.info(f"  Generated: {out_path}")

    return saved_paths


# =============================================================================
# SECTION 16: USER-FRIENDLY AUDIO PROCESSING FUNCTION (COLAB-READY)
# =============================================================================

def process_user_audio_file(
    input_audio_path: str,
    output_dir: str = "emotional_speech_outputs",
    checkpoint_path: Optional[str] = None,
    source_emotion: str = "neutral",
    target_emotions: Optional[List[str]] = None,
    device: str = "auto",
    run_evaluation: bool = True,
) -> Dict[str, Any]:
    """
    Simple function to convert a user's audio file to all emotions.

    This function is designed for interactive use in Colab/Jupyter notebooks.
    It loads a pretrained model (or trains a quick one if no checkpoint provided),
    processes the input audio, and saves output WAV files.

    Args:
        input_audio_path: Path to the user's WAV file.
        output_dir: Directory where output files will be saved.
        checkpoint_path: Optional path to a pretrained model checkpoint.
                         If None, a minimal model will be used (not fully trained).
        source_emotion: The emotion label of the input audio (default 'neutral').
        target_emotions: List of target emotions (default: all).
        device: Device to use ('auto', 'cpu', 'cuda').
        run_evaluation: Whether to compute and print evaluation metrics.

    Returns:
        Dictionary containing results and metrics.

    Example usage in Colab:
        from google.colab import files
        uploaded = files.upload()
        for filename in uploaded.keys():
            results = process_user_audio_file(filename)
    """
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # Create pipeline with default configs
    pipeline = EmotionalSpeechPipeline(
        output_dir=output_dir,
        checkpoint_dir=os.path.join(output_dir, "checkpoints"),
        device=device,
    )

    # Load checkpoint if provided, otherwise warn that model is untrained
    if checkpoint_path and os.path.exists(checkpoint_path):
        pipeline.load_pretrained(checkpoint_path)
        logger.info("Loaded pretrained model.")
    else:
        logger.warning("No pretrained model provided. Using randomly initialized weights. "
                       "For best results, provide a trained checkpoint or train first.")

    # Process the audio
    results = pipeline.process_audio(
        input_path=input_audio_path,
        source_emotion=source_emotion,
        target_emotions=target_emotions,
        save_outputs=True,
        run_evaluation=run_evaluation,
    )

    return results


# =============================================================================
# SECTION 17: COMMAND LINE INTERFACE (COLAB-FRIENDLY)
# =============================================================================

def parse_args():
    parser = argparse.ArgumentParser(
        description="Emotional Speech Conversion System: VAE + GNN + FastSpeech2 + HiFi-GAN",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # Run full demo with synthetic audio (no real audio needed):
  python emotional_speech_system.py --demo

  # Convert your voice audio to all emotions:
  python emotional_speech_system.py --input your_voice.wav

  # Convert to specific emotion:
  python emotional_speech_system.py --input your_voice.wav --emotion happy

  # Train the model first, then convert:
  python emotional_speech_system.py --input your_voice.wav --train --epochs 10

  # Show model architecture:
  python emotional_speech_system.py --architecture

  # Show emotion graph:
  python emotional_speech_system.py --emotion-graph
        """
    )

    parser.add_argument(
        "--input", "-i", type=str, default=None,
        help="Path to input voice audio file (WAV format)"
    )
    parser.add_argument(
        "--emotion", "-e", type=str, default=None,
        choices=list(EMOTION_PROFILES.keys()),
        help="Target emotion (default: convert to all emotions)"
    )
    parser.add_argument(
        "--all-emotions", action="store_true",
        help="Convert to all supported emotions (default behavior)"
    )
    parser.add_argument(
        "--source-emotion", type=str, default="neutral",
        choices=list(EMOTION_PROFILES.keys()),
        help="Emotion of the source audio (default: neutral)"
    )
    parser.add_argument(
        "--train", action="store_true",
        help="Train the model before converting"
    )
    parser.add_argument(
        "--data-dir", type=str, default=None,
        help="Directory with training data (subdirs per emotion)"
    )
    parser.add_argument(
        "--epochs", type=int, default=5,
        help="Number of training epochs"
    )
    parser.add_argument(
        "--checkpoint", type=str, default=None,
        help="Path to pretrained model checkpoint"
    )
    parser.add_argument(
        "--output-dir", type=str, default="emotional_speech_system/outputs",
        help="Output directory for generated audio files"
    )
    parser.add_argument(
        "--demo", action="store_true",
        help="Run full demo with synthetic audio"
    )
    parser.add_argument(
        "--architecture", action="store_true",
        help="Print model architecture and exit"
    )
    parser.add_argument(
        "--emotion-graph", action="store_true",
        help="Print emotion graph and exit"
    )
    parser.add_argument(
        "--no-eval", action="store_true",
        help="Skip evaluation metrics"
    )
    parser.add_argument(
        "--device", type=str, default=None,
        choices=["cpu", "cuda", "mps"],
        help="Device to run on"
    )
    parser.add_argument(
        "--latent-dim", type=int, default=128,
        help="Latent dimension for VAE"
    )
    parser.add_argument(
        "--learning-rate", type=float, default=1e-4,
        help="Learning rate for training"
    )

    # --- COLAB FIX: use parse_known_args to ignore Colab's kernel connection args ---
    args, unknown = parser.parse_known_args()
    if unknown:
        logger.warning(f"Ignored unknown arguments: {unknown}")
    return args


def main():
    args = parse_args()

    print("\n" + "=" * 80)
    print("EMOTIONAL SPEECH CONVERSION SYSTEM")
    print("VAE + Graph Neural Network + FastSpeech 2 + HiFi-GAN Vocoder")
    print("=" * 80)
    print(f"Python: {sys.version.split()[0]}")
    print(f"PyTorch: {torch.__version__}")
    print(f"torchaudio: {torchaudio.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print("=" * 80 + "\n")

    audio_config = AudioConfig()
    model_config = ModelConfig(latent_dim=args.latent_dim)
    train_config = TrainConfig(
        learning_rate=args.learning_rate,
        num_epochs=args.epochs,
        checkpoint_dir=os.path.join(args.output_dir, "checkpoints"),
    )

    if args.emotion_graph:
        print_emotion_graph()
        return

    pipeline = EmotionalSpeechPipeline(
        audio_config=audio_config,
        model_config=model_config,
        train_config=train_config,
        device=args.device,
        output_dir=args.output_dir,
        checkpoint_dir=os.path.join(args.output_dir, "checkpoints"),
    )

    if args.architecture:
        pipeline.print_model_architecture()
        return

    if args.checkpoint:
        pipeline.load_pretrained(args.checkpoint)

    if args.train or args.demo:
        logger.info(f"\nTraining model for {args.epochs} epochs...")
        history = pipeline.train(
            data_dir=args.data_dir,
            num_epochs=args.epochs,
        )
        if history['gen_loss']:
            logger.info(f"Training complete. Final gen loss: {history['gen_loss'][-1]:.4f}")
        else:
            logger.info("Training skipped (0 epochs)")

    if args.demo and not args.input:
        demo_paths = generate_emotion_demo_audio(
            output_dir=os.path.join(args.output_dir, "demo_reference"),
            sr=audio_config.sample_rate,
        )
        input_path = None
        logger.info("Using synthetic demo audio as input")
    else:
        input_path = args.input

    target_emotions = [args.emotion] if args.emotion else list(EMOTION_PROFILES.keys())

    logger.info(f"\nProcessing audio: {input_path or 'synthetic demo'}")
    logger.info(f"Target emotions: {target_emotions}")

    results = pipeline.process_audio(
        input_path=input_path,
        source_emotion=args.source_emotion,
        target_emotions=target_emotions,
        save_outputs=True,
        run_evaluation=not args.no_eval,
    )

    print("\n" + "=" * 80)
    print("OUTPUT FILES")
    print("=" * 80)
    print(f"\nOutput directory: {os.path.abspath(args.output_dir)}")
    for emotion in target_emotions:
        wav_path = os.path.join(args.output_dir, f"{emotion}.wav")
        if os.path.exists(wav_path):
            size_kb = os.path.getsize(wav_path) / 1024
            print(f"  {emotion:<12}: {wav_path}  ({size_kb:.1f} KB)")

    csv_path = os.path.join(args.output_dir, "evaluation_results.csv")
    if os.path.exists(csv_path):
        print(f"\n  Evaluation CSV: {csv_path}")

    print("\n" + "=" * 80)
    print("PIPELINE COMPLETE")
    print("=" * 80)

    print("\nLATENT SPACE ANALYSIS (VAE Encoding)")
    z = results.get("z_source")
    if z is not None:
        z_np = z.squeeze().numpy()
        print(f"  Latent vector shape : {z_np.shape}")
        print(f"  Mean               : {z_np.mean():.4f}")
        print(f"  Std                : {z_np.std():.4f}")
        print(f"  Min/Max            : {z_np.min():.4f} / {z_np.max():.4f}")
        top_dims = np.argsort(np.abs(z_np))[-5:][::-1]
        print(f"  Top-5 active dims  : {top_dims.tolist()}")
        print(f"    Values: {z_np[top_dims].tolist()}")

    print("\nGRAPH NEURAL NETWORK: Emotion Embedding Space")
    embeddings = pipeline.model.gnn.get_emotion_embeddings()
    print(f"  {'Emotion':<12}  {'Embed-Norm':>12}")
    for ename, emb in embeddings.items():
        norm = emb.norm().item()
        print(f"  {ename:<12}  {norm:>12.4f}")

    print("\n")
    return results


# =============================================================================
# SECTION 18: UNIT TESTS
# =============================================================================

def run_unit_tests():
    """Run basic unit tests to verify all components work."""
    print("\n" + "=" * 60)
    print("RUNNING UNIT TESTS")
    print("=" * 60)

    device = torch.device("cpu")
    audio_config = AudioConfig()
    model_config = ModelConfig(
        latent_dim=64,
        vae_hidden_dim=128,
        vae_num_layers=2,
        gnn_hidden_dim=128,
        gnn_num_layers=2,
        fastspeech_hidden=128,
        fastspeech_num_layers=2,
        fastspeech_filter_size=256,
        hifigan_upsample_initial_channel=128,
    )

    print("\n[1] Testing AudioProcessor...")
    processor = AudioProcessor(audio_config)
    waveform = processor._generate_demo_signal(duration_sec=1.0)
    assert waveform.shape[0] == 1, "Waveform should be mono"
    assert waveform.shape[1] == 22050, f"Expected 22050 samples, got {waveform.shape[1]}"
    mel = processor.get_mel_spectrogram(waveform)
    assert mel.shape[0] == 80, f"Expected 80 mel bins, got {mel.shape[0]}"
    f0, voiced = processor.get_pitch(waveform)
    energy = processor.get_energy(mel)
    print(f"  Mel: {mel.shape}, F0: {f0.shape}, Energy: {energy.shape} -> OK")

    print("\n[2] Testing VAE Encoder/Decoder...")
    vae = VoiceVAE(model_config, n_mels=80)
    mel_batch = mel.unsqueeze(0)
    mu, log_var = vae.encode(mel_batch)
    assert mu.shape == (1, 64), f"Expected (1,64), got {mu.shape}"
    z = vae.reparameterize(mu, log_var)
    mel_recon = vae.decode(z, mel.shape[1])
    assert mel_recon.shape[1] == 80, f"Expected 80 mel bins in output, got {mel_recon.shape[1]}"
    print(f"  mu: {mu.shape}, z: {z.shape}, mel_recon: {mel_recon.shape} -> OK")

    print("\n[3] Testing GNN...")
    gnn = EmotionGNN(model_config)
    z_test = torch.randn(2, 64)
    src_ids = torch.tensor([0, 1])
    tgt_ids = torch.tensor([2, 4])
    gnn_out = gnn(z_test, src_ids, tgt_ids)
    assert gnn_out["z_target"].shape == (2, 64), f"GNN output shape mismatch"
    assert gnn_out["prosody_target"].shape == (2, 6), f"GNN prosody shape mismatch"
    print(f"  z_target: {gnn_out['z_target'].shape}, prosody: {gnn_out['prosody_target'].shape} -> OK")

    print("\n[4] Testing FastSpeech2 Variance Adaptor...")
    va = FastSpeech2VarianceAdaptor(model_config, n_mels=80)
    z_s = torch.randn(1, 64)
    z_e = torch.randn(1, 64)
    profile = EMOTION_PROFILES["happy"]
    va_out = va(z_s, z_e, target_mel_len=100, emotion_profile=profile)
    assert va_out["mel"].shape[-1] == 100, f"Expected mel len 100, got {va_out['mel'].shape[-1]}"
    print(f"  mel: {va_out['mel'].shape}, pitch: {va_out['pitch_pred'].shape} -> OK")

    print("\n[5] Testing HiFi-GAN Generator...")
    hifigan = HiFiGANGenerator(model_config, n_mels=80)
    mel_in = torch.randn(1, 80, 100)
    wav_out = hifigan(mel_in)
    expected_len = 100 * math.prod(model_config.hifigan_upsample_rates)
    assert wav_out.shape[0] == 1 and wav_out.shape[1] == 1, "HiFi-GAN output shape error"
    print(f"  Input mel: {mel_in.shape} -> Waveform: {wav_out.shape} -> OK")

    print("\n[6] Testing Full Forward Pass...")
    full_model = EmotionalSpeechConverter(model_config, audio_config)
    mel_input = mel.unsqueeze(0)
    src_id = torch.tensor([0])
    tgt_id = torch.tensor([1])
    with torch.no_grad():
        output = full_model(
            mel=mel_input,
            source_emotion_id=src_id,
            target_emotion_id=tgt_id,
            target_mel_len=mel.shape[1]
        )
    assert "waveform" in output
    assert "mel_converted" in output
    assert "z_source" in output
    print(f"  waveform: {output['waveform'].shape}, mel: {output['mel_converted'].shape} -> OK")

    print("\n[7] Testing Evaluation Metrics...")
    evaluator = SpeechEvaluator(audio_config)
    mel_ref = torch.randn(1, 80, 100)
    mel_hyp = torch.randn(1, 80, 100)
    mcd = evaluator.mel_cepstral_distortion(mel_ref, mel_hyp)
    cos_sim = evaluator.cosine_similarity(mel_ref, mel_hyp)
    assert 0 <= mcd, "MCD should be non-negative"
    assert -1 <= cos_sim <= 1, "Cosine similarity should be in [-1, 1]"
    print(f"  MCD: {mcd:.3f}, CosSim: {cos_sim:.3f} -> OK")

    print("\n[8] Testing Post-Processor...")
    pp = NaturalAudioPostProcessor(sample_rate=22050)
    wav_raw = torch.randn(1, 22050) * 0.1
    for ename, eprofile in EMOTION_PROFILES.items():
        wav_processed = pp.apply(wav_raw, eprofile)
        assert wav_processed.shape == wav_raw.shape, f"Post-processor shape mismatch for {ename}"
    print(f"  All {len(EMOTION_PROFILES)} emotions processed -> OK")

    print("\n[9] Testing Dataset (synthetic)...")
    dataset = SpeechDataset(None, AudioProcessor(audio_config), use_synthetic=True, synthetic_samples=14)
    assert len(dataset) == 14, f"Dataset length mismatch"
    sample = dataset[0]
    required_keys = ["mel", "f0", "energy", "emotion_id", "prosody"]
    for k in required_keys:
        assert k in sample, f"Missing key {k} in dataset sample"
    print(f"  Dataset length: {len(dataset)}, sample keys: {list(sample.keys())} -> OK")

    print("\n" + "=" * 60)
    print("ALL UNIT TESTS PASSED")
    print("=" * 60 + "\n")


# =============================================================================
# SECTION 19: ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    if "--test" in sys.argv:
        run_unit_tests()
    else:
        main()


EMOTIONAL SPEECH CONVERSION SYSTEM
VAE + Graph Neural Network + FastSpeech 2 + HiFi-GAN Vocoder
Python: 3.12.13
PyTorch: 2.10.0+cpu
torchaudio: 2.10.0+cpu
CUDA available: False


EVALUATION RESULTS: EMOTIONAL SPEECH CONVERSION

[ METRIC LEGEND ]
  MCD↓         - Mel Cepstral Distortion (dB), lower = more similar spectrum
  F0-RMSE↓     - F0 Root Mean Square Error (log scale), lower = more accurate pitch
  F0-Corr↑     - F0 Pearson Correlation, higher = pitch patterns more similar
  E-RMSE↓      - Energy RMSE, lower = more similar loudness
  PESQ↑        - Perceptual Eval Speech Quality [1-4.5], higher = better
  STOI↑        - Short-Time Objective Intelligibility [0-1], higher = better
  CosSim↑      - Cosine Similarity of mel vectors [0-1], higher = more similar
  Pitch%Δ      - % change in mean pitch vs. original
  Energy%Δ     - % change in mean energy vs. original
  Dur%Δ        - % change in duration vs. original
  Exp.Pitch    - Expected pitch shift in semitones (from emotion pr

In [ ]:
# ============================================================
# COLAB CELL: Upload Your Audio and Convert to All Emotions
# ============================================================
# This cell will:
#   1. Ask you to choose an audio file from your computer.
#   2. Convert it to all 7 emotions.
#   3. Save the output files in 'emotional_speech_outputs/'.
#   4. Print a detailed evaluation table.

# First, make sure you've run the entire system code in a previous cell.
# Then run this cell.

try:
    from google.colab import files
    print("📁 Click 'Choose File' to upload your audio (WAV, MP3, FLAC, etc.)")
    uploaded = files.upload()
except ImportError:
    print("⚠️ This cell is designed for Google Colab. If you're running locally, use the command line instead.")
    uploaded = {}

if uploaded:
    # Get the first uploaded filename
    input_filename = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded: {input_filename}")

    # -----------------------------------------------------------------
    # CONVERT THE AUDIO TO ALL EMOTIONS
    # -----------------------------------------------------------------
    print("\n🔄 Processing audio... This may take a minute.\n")
    results = process_user_audio_file(
        input_audio_path=input_filename,
        output_dir="emotional_speech_outputs",
        checkpoint_path=None,           # Replace with your own checkpoint if you have one
        source_emotion="neutral",       # Assume input is neutral; change if needed
        target_emotions=None,           # None = convert to all 7 emotions
        device="auto",
        run_evaluation=True
    )

    # -----------------------------------------------------------------
    # SHOW OUTPUT FILES
    # -----------------------------------------------------------------
    print("\n" + "="*60)
    print("✅ Conversion complete! Your output files:")
    print("="*60)
    import os
    out_dir = "emotional_speech_outputs"
    for file in sorted(os.listdir(out_dir)):
        if file.endswith(".wav"):
            print(f"   🎵 {file}")

    # -----------------------------------------------------------------
    # (Optional) Download all output files as a ZIP
    # -----------------------------------------------------------------
    download = input("\n⬇️ Download all results as ZIP? (y/n): ").strip().lower()
    if download == 'y':
        import zipfile
        zip_path = "emotional_speech_outputs.zip"
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            for root, dirs, files_in_dir in os.walk(out_dir):
                for file in files_in_dir:
                    zipf.write(os.path.join(root, file), arcname=file)
        files.download(zip_path)
        print("ZIP downloaded.")
else:
    print("❌ No file uploaded. Exiting.")

📁 Click 'Choose File' to upload your audio (WAV, MP3, FLAC, etc.)


Saving 04-10-2026 13.56(2).wav to 04-10-2026 13.56(2).wav

✅ Uploaded: 04-10-2026 13.56(2).wav

🔄 Processing audio... This may take a minute.




EVALUATION RESULTS: EMOTIONAL SPEECH CONVERSION

[ METRIC LEGEND ]
  MCD↓         - Mel Cepstral Distortion (dB), lower = more similar spectrum
  F0-RMSE↓     - F0 Root Mean Square Error (log scale), lower = more accurate pitch
  F0-Corr↑     - F0 Pearson Correlation, higher = pitch patterns more similar
  E-RMSE↓      - Energy RMSE, lower = more similar loudness
  PESQ↑        - Perceptual Eval Speech Quality [1-4.5], higher = better
  STOI↑        - Short-Time Objective Intelligibility [0-1], higher = better
  CosSim↑      - Cosine Similarity of mel vectors [0-1], higher = more similar
  Pitch%Δ      - % change in mean pitch vs. original
  Energy%Δ     - % change in mean energy vs. original
  Dur%Δ        - % change in duration vs. original
  Exp.Pitch    - Expected pitch shift in semitones (from emotion profile)
  Exp.Energy   - Expected energy scale factor (from emotion profile)
  Voice Quality- Qualitative voice quality label

[ EVALUATION TABLE ]

+-----------+---------+--------